# Analysis pipeline (all experiments in the article)

This is the Colab notebook in which every experiment of the article was run, with its outputs kept. It reads
the raw files from `MyDrive/Colab Notebooks/` and writes the cleaned splits, emotion scores, result tables and
figures back there. Several steps were run more than once while the analysis was revised; where a step appears
more than once, the **last** run is the one reported in the article.

| Cells | Step |
|---|---|
| 0-5 | Load the raw Stack Overflow, GitHub and Bugzilla files |
| 6 | Bugzilla cleaning: real severity labels, de-duplication, 70/15/15 split, class weights |
| 7-17, 20, 24-27, 31-34 | Bugzilla and GitHub baselines, fine-tuned transformers, distress features on TF-IDF + XGBoost, final tables and paired bootstrap |
| 18-19 | GitHub resolution-time classes and distress features |
| 21 | Emotion/distress analysis on all three platforms |
| 22 | SHAP explanation and error analysis (unmasked text) |
| 23 | Masking sensitivity check |
| 28 | Ablation (Bugzilla masked, GitHub) |
| 29 | ROC-AUC, per-class metrics, confusion matrices, McNemar tests |
| 35 | Ten repeated splits with Wilcoxon tests |
| 37 | Rerun with emotion scores recomputed on masked Bugzilla text (`scripts/rerun_masked_emotion.py`) |
| 38 | GitHub resplits with per-partition thresholds, per-class metrics, saved predictions (`scripts/github_resplits_thresholds.py`) |


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
print(os.listdir('/content/drive/MyDrive/Colab Notebooks')[:80])

['GitHub_bugs.csv', 'github_emotion_dataset.csv', 'github_balanced_emotion_dataset.csv', 'github_tfidf_vectorizer.pkl', 'github_logistic_regression_model.pkl', 'github_random_forest_model.pkl', 'github_svm_model.pkl', 'github_naive_bayes_model.pkl', 'github_xgboost_model.pkl', 'github_tokenizer.pkl', 'github_lstm_model.keras', 'github_bilstm_model.keras', 'github_cnn_bilstm_model.keras', 'github_bert_model', 'github_roberta_model', 'github_codebert_model', 'github_model_comparison.csv', 'github_bar_chart.png', 'github_line_chart.png', 'Emotion_GitHub.ipynb', 'so_emotion_balanced.csv', 'so_severity_balanced.csv', 'github_emotion_balanced.csv', 'github_severity_balanced.csv', 'bugzilla_severity_balanced.csv', 'bugzilla_emotion_balanced.csv', 'api_related_dataset.csv', 'Bugzilla_fix.csv', 'Bugzilla_sev.csv', 'Untitled0.ipynb', 'dataset_version.csv', 'data_dictionary.csv', 'validation.ipynb', 'bugzilla_complete_ablation_results.csv', 'wilcoxon_results.csv', 'so_roc_auc_results.csv', 'githu

In [ ]:
import pandas as pd
base = '/content/drive/MyDrive/Colab Notebooks/'
so = pd.read_csv(base + 'so_severity_balanced.csv')
gh = pd.read_csv(base + 'github_severity_balanced.csv')

print("=== SO: emotion vs severity ===")
print(pd.crosstab(so['emotion'], so['severity']))
print("SO rows:", len(so), "| unique texts:", so['text'].nunique())

print("\n=== GitHub ===")
print("rows:", len(gh), "| unique texts:", gh['text'].nunique())
print("columns:", gh.columns.tolist())
print(gh['severity'].value_counts())
print(gh.head(3).to_string(max_colwidth=60))

=== SO: emotion vs severity ===
severity      Critical  High   Low
emotion                           
anger              468     0     0
anxiety              0   124     0
confusion            0     0   148
frustration          0  4518     0
helplessness      2707     0     0
satisfaction         0     0  4494
urgency           1467     0     0
SO rows: 13926 | unique texts: 7017

=== GitHub ===
rows: 32829 | unique texts: 20530
columns: ['Issue id', 'Summary', 'Created', 'Resolved', 'Description', 'Resolution', 'text', 'severity']
severity
Low         10943
High        10943
Critical    10943
Name: count, dtype: int64
   Issue id                                                      Summary                    Created                   Resolved                                                  Description Resolution                                                         text severity
0     61269                     Explorer rename text field screen cheese  2018-10-18 21:00:02+00:00  201

In [ ]:
import pandas as pd
base='/content/drive/MyDrive/Colab Notebooks/'
for f in ['api_related_dataset.csv','GitHub_bugs.csv','Bugzilla_fix.csv']:
    d=pd.read_csv(base+f)
    print(f, d.shape); print(d.columns.tolist())
    for c in d.columns:
        if d[c].nunique()<30: print(' ',c, d[c].value_counts().head(8).to_dict())

api_related_dataset.csv (13927, 5)
['QuestionId', 'Title', 'Body', 'Tags', 'CollectedTag']
  CollectedTag {'google-api': 2498, 'github-api': 2497, 'android-api': 2494, 'facebook-graph-api': 2492, 'java-api': 2256, 'twitter-api': 998, 'rest-api': 600, 'youtube-api': 92}
GitHub_bugs.csv (32829, 6)
['Issue id', 'Summary', 'Created', 'Resolved', 'Description', 'Resolution']
  Resolution {'completed': 29387, 'not_planned': 659, 'reopened': 191}
Bugzilla_fix.csv (32428, 5)
['Unnamed: 0.1', 'Unnamed: 0', 'Description', 'Fixing_time', 'Label']
  Label {0: 24797, 1: 7631}


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, pandas as pd
base='/content/drive/MyDrive/'
keys=('sever','priorit','label','emotion','fix','resol')
for root,_,files in os.walk(base):
    if '/.' in root or 'model' in root.lower(): continue
    for f in files:
        if f.lower().endswith('.csv'):
            p=os.path.join(root,f)
            try:
                d=pd.read_csv(p,nrows=200)
                cols=[c for c in d.columns if any(k in c.lower() for k in keys)]
                n=sum(1 for _ in open(p,'rb'))-1
                print(p.replace(base,''),'| ~rows',n,'|',list(d.columns)[:12],'| key:',cols)
            except Exception as e:
                print(p,'ERR',e)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Research Project/data set/stackoverflow_bug_dataset.csv | ~rows 8796 | ['post_id', 'post_type', 'title', 'body', 'tags', 'creation_date', 'user_id', 'score', 'comments', 'comment_count'] | key: []
Colab Notebooks/GitHub_bugs.csv | ~rows 795850 | ['Issue id', 'Summary', 'Created', 'Resolved', 'Description', 'Resolution'] | key: ['Resolved', 'Resolution']
Colab Notebooks/github_emotion_dataset.csv | ~rows 1558871 | ['Issue id', 'Summary', 'Created', 'Resolved', 'Description', 'Resolution', 'text', 'clean_text', 'emotion_label'] | key: ['Resolved', 'Resolution', 'emotion_label']
Colab Notebooks/github_balanced_emotion_dataset.csv | ~rows 1526339 | ['Issue id', 'Summary', 'Created', 'Resolved', 'Description', 'Resolution', 'text', 'clean_text', 'emotion_label'] | key: ['Resolved', 'Resolution', 'emotion_label']
Colab Notebooks/github_model_comparison.csv | ~rows 

In [ ]:
import pandas as pd
b='/content/drive/MyDrive/'
s=pd.read_csv(b+'Colab Notebooks/Bugzilla_sev.csv')
print(s.shape); print(s['Severity'].value_counts()); print(pd.crosstab(s['Severity'],s['Label']))
print('unique desc:',s['Description'].nunique())
f=pd.read_csv(b+'Colab Notebooks/Bugzilla_fix.csv')
print('same rows/order as fix file:', len(s)==len(f), (s['Description'].fillna('')==f['Description'].fillna('')).mean() if len(s)==len(f) else 'n/a')

o=pd.read_csv(b+'Research Project/data set/stackoverflow_bug_dataset.csv')
print(o.shape); print(o.columns.tolist()); print(o['post_type'].value_counts())
print(o.head(3).T)

pd.set_option('display.max_colwidth',150, 'display.width',250)
print(pd.read_csv(b+'Colab Notebooks/data_dictionary.csv').to_string())
print(pd.read_csv(b+'Colab Notebooks/dataset_version.csv').to_string())

(32428, 5)
Severity
normal      22828
major        3760
critical     2726
minor        1462
blocker      1082
trivial       570
Name: count, dtype: int64
Label         0     1
Severity             
blocker       0  1082
critical      0  2726
major         0  3760
minor      1462     0
normal    22828     0
trivial     570     0
unique desc: 32427
same rows/order as fix file: True 1.0
(500, 10)
['post_id', 'post_type', 'title', 'body', 'tags', 'creation_date', 'user_id', 'score', 'comments', 'comment_count']
post_type
question    500
Name: count, dtype: int64
                                                               0  \
post_id                                                      108   
post_type                                               question   
title          How can I best retest a bug in a newer or deve...   
body           <p>I have reported a bug that occurs in Karmic...   
tags                                   |launchpad|bug-reporting|   
creation_date              

In [ ]:
# Clean Bugzilla baseline: real Severity labels, dedupe, 70:15:15, class weights, no Fixing_time
from google.colab import drive
drive.mount('/content/drive')
import pandas as pd, numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report

P='/content/drive/MyDrive/Colab Notebooks/'
df=pd.read_csv(P+'Bugzilla_sev.csv')[['Description','Severity']].dropna()
df['text']=df['Description'].astype(str).str.lower().str.replace(r'\s+',' ',regex=True).str.strip()
n0=len(df); df=df.drop_duplicates('text').reset_index(drop=True)
print('rows',n0,'-> after dedupe',len(df))
m={'blocker':'Critical','critical':'Critical','major':'High','normal':'Low','minor':'Low','trivial':'Low'}
df['y']=df['Severity'].map(m); print(df['y'].value_counts())

tr,tmp=train_test_split(df,test_size=0.30,stratify=df['y'],random_state=42)
va,te=train_test_split(tmp,test_size=0.50,stratify=tmp['y'],random_state=42)
assert not (set(tr.text)&set(te.text)) and not (set(tr.text)&set(va.text))
print('train/val/test',len(tr),len(va),len(te))
for n,d in [('train',tr),('val',va),('test',te)]: d.to_csv(P+f'clean_bugzilla_{n}.csv',index=False)

v=TfidfVectorizer(max_features=20000,ngram_range=(1,2),min_df=2,sublinear_tf=True)
Xtr=v.fit_transform(tr.text)
clf=LogisticRegression(max_iter=1000,class_weight='balanced').fit(Xtr,tr.y)
for n,d in [('val',va),('test',te)]:
    p=clf.predict(v.transform(d.text))
    print(n,'acc %.4f  macroF1 %.4f'%(accuracy_score(d.y,p),f1_score(d.y,p,average='macro')))
print(classification_report(te.y,clf.predict(v.transform(te.text))))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
rows 32428 -> after dedupe 32415
y
Low         24850
Critical     3807
High         3758
Name: count, dtype: int64
train/val/test 22690 4862 4863
val acc 0.7038  macroF1 0.5369
test acc 0.6990  macroF1 0.5352
              precision    recall  f1-score   support

    Critical       0.44      0.62      0.51       571
        High       0.23      0.33      0.27       564
         Low       0.89      0.77      0.82      3728

    accuracy                           0.70      4863
   macro avg       0.52      0.57      0.54      4863
weighted avg       0.76      0.70      0.72      4863



In [ ]:
# Bugzilla clean Hybrid rerun (Colab, GPU). Uses clean_bugzilla_{train,val,test}.csv
# No Fixing_time, no oversampling, class-weighted loss, model selection on VAL only, test touched once.
from google.colab import drive
drive.mount('/content/drive')
import os, numpy as np, pandas as pd, torch, torch.nn as nn
from transformers import AutoTokenizer, AutoModel
from sklearn.metrics import accuracy_score, f1_score, classification_report
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.utils.class_weight import compute_class_weight

P='/content/drive/MyDrive/Colab Notebooks/'
dev='cuda' if torch.cuda.is_available() else 'cpu'
tr,va,te=[pd.read_csv(P+f'clean_bugzilla_{n}.csv') for n in ('train','val','test')]
classes=['Critical','High','Low']; cid={c:i for i,c in enumerate(classes)}
for d in (tr,va,te): d['text']=d['text'].fillna('').astype(str)
y={n:d['y'].map(cid).values for n,d in (('train',tr),('val',va),('test',te))}

@torch.no_grad()
def embed(name,texts,bs=64,maxlen=256):
    tok=AutoTokenizer.from_pretrained(name); m=AutoModel.from_pretrained(name).to(dev).eval().half()
    out=[]
    for i in range(0,len(texts),bs):
        b=tok(list(texts[i:i+bs]),padding=True,truncation=True,max_length=maxlen,return_tensors='pt').to(dev)
        out.append(m(**b).last_hidden_state[:,0].float().cpu().numpy())
    return np.vstack(out)

feats={}
for tag,name in (('roberta','roberta-base'),('bert','bert-base-uncased')):
    for n,d in (('train',tr),('val',va),('test',te)):
        f=P+f'emb_bz_{tag}_{n}.npy'
        if not os.path.exists(f): np.save(f,embed(name,d['text'].astype(str).values))
        feats[(tag,n)]=np.load(f)
tf=TfidfVectorizer(max_features=20000,ngram_range=(1,2),min_df=2,sublinear_tf=True).fit(tr.text)
svd=TruncatedSVD(256,random_state=42).fit(tf.transform(tr.text))
for n,d in (('train',tr),('val',va),('test',te)): feats[('tfidf',n)]=svd.transform(tf.transform(d.text))

def X(tags,n): return np.hstack([feats[(t,n)] for t in tags])

def run(tags,seed=42,epochs=60):
    torch.manual_seed(seed); np.random.seed(seed)
    Xtr,Xva,Xte=[X(tags,n) for n in ('train','val','test')]
    mu,sd=Xtr.mean(0),Xtr.std(0)+1e-6
    T=lambda a:torch.tensor((a-mu)/sd,dtype=torch.float32,device=dev)
    Xtr_,Xva_,Xte_=T(Xtr),T(Xva),T(Xte); ytr=torch.tensor(y['train'],device=dev)
    w=torch.tensor(compute_class_weight('balanced',classes=np.arange(3),y=y['train']),dtype=torch.float32,device=dev)
    net=nn.Sequential(nn.Dropout(0.2),nn.Linear(Xtr.shape[1],512),nn.ReLU(),nn.Dropout(0.3),nn.Linear(512,3)).to(dev)
    opt=torch.optim.AdamW(net.parameters(),1e-3,weight_decay=1e-2); lossf=nn.CrossEntropyLoss(weight=w)
    best,state=-1,None
    for ep in range(epochs):
        net.train(); perm=torch.randperm(len(ytr),device=dev)
        for i in range(0,len(perm),256):
            idx=perm[i:i+256]; opt.zero_grad(); lossf(net(Xtr_[idx]),ytr[idx]).backward(); opt.step()
        net.eval()
        with torch.no_grad(): pv=net(Xva_).argmax(1).cpu().numpy()
        f=f1_score(y['val'],pv,average='macro')
        if f>best: best,state=f,{k:v.clone() for k,v in net.state_dict().items()}
    net.load_state_dict(state); net.eval()
    with torch.no_grad(): pt=net(Xte_).argmax(1).cpu().numpy()
    return dict(acc=accuracy_score(y['test'],pt),macroF1=f1_score(y['test'],pt,average='macro'),
                valF1=best,pred=pt)

configs={'TF-IDF only':['tfidf'],'RoBERTa only':['roberta'],'BERT(emotion enc.) only':['bert'],
         'RoBERTa+BERT (Hybrid)':['roberta','bert'],'Hybrid + TF-IDF':['roberta','bert','tfidf']}
rows=[]
for name,tags in configs.items():
    rs=[run(tags,seed=s) for s in (42,43,44,45,46)]
    rows.append(dict(Config=name,Acc=np.mean([r['acc'] for r in rs]),Acc_sd=np.std([r['acc'] for r in rs]),
                     MacroF1=np.mean([r['macroF1'] for r in rs]),MacroF1_sd=np.std([r['macroF1'] for r in rs])))
    print(rows[-1])
    if name=='RoBERTa+BERT (Hybrid)':
        print(classification_report(y['test'],rs[0]['pred'],target_names=classes))
res=pd.DataFrame(rows); print(res.round(4).to_string(index=False))
res.to_csv(P+'clean_bugzilla_results.csv',index=False)

Mounted at /content/drive


KeyboardInterrupt: 

In [ ]:
# Step A: fine-tune RoBERTa end-to-end on clean Bugzilla split (real Severity, class-weighted loss)
# Colab T4. Model chosen on VAL macro-F1; test scored once per seed. Saves logits for later stacking.
from google.colab import drive
drive.mount('/content/drive')
import numpy as np, pandas as pd, torch, torch.nn as nn, time
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from sklearn.metrics import accuracy_score, f1_score, classification_report
from sklearn.utils.class_weight import compute_class_weight

P='/content/drive/MyDrive/Colab Notebooks/'
dev='cuda'
tr,va,te=[pd.read_csv(P+f'clean_bugzilla_{n}.csv') for n in ('train','val','test')]
for d in (tr,va,te): d['text']=d['text'].fillna('').astype(str)
classes=['Critical','High','Low']; cid={c:i for i,c in enumerate(classes)}
Y={n:d['y'].map(cid).values for n,d in (('train',tr),('val',va),('test',te))}
tok=AutoTokenizer.from_pretrained('roberta-base'); MAXLEN=192; BS=32

def enc(d):
    e=tok(list(d.text),truncation=True,max_length=MAXLEN,padding='max_length',return_tensors='pt')
    return e['input_ids'],e['attention_mask']
E={n:enc(d) for n,d in (('train',tr),('val',va),('test',te))}

@torch.no_grad()
def logits_of(model,n):
    model.eval(); ids,am=E[n]; out=[]
    for i in range(0,len(ids),128):
        with torch.autocast('cuda',dtype=torch.float16):
            out.append(model(input_ids=ids[i:i+128].to(dev),attention_mask=am[i:i+128].to(dev)).logits.float().cpu())
    return torch.cat(out).numpy()

def run(seed,epochs=4,lr=2e-5):
    torch.manual_seed(seed); np.random.seed(seed)
    model=AutoModelForSequenceClassification.from_pretrained('roberta-base',num_labels=3).to(dev)
    w=torch.tensor(compute_class_weight('balanced',classes=np.arange(3),y=Y['train']),dtype=torch.float32,device=dev)
    lossf=nn.CrossEntropyLoss(weight=w)
    opt=torch.optim.AdamW(model.parameters(),lr=lr,weight_decay=0.01)
    ids,am=E['train']; ytr=torch.tensor(Y['train'])
    steps=epochs*((len(ids)+BS-1)//BS); sch=get_linear_schedule_with_warmup(opt,int(0.06*steps),steps)
    scaler=torch.cuda.amp.GradScaler(); best,bstate=-1,None
    for ep in range(epochs):
        model.train(); perm=torch.randperm(len(ids)); t0=time.time()
        for i in range(0,len(perm),BS):
            ix=perm[i:i+BS]
            with torch.autocast('cuda',dtype=torch.float16):
                lg=model(input_ids=ids[ix].to(dev),attention_mask=am[ix].to(dev)).logits
            loss=lossf(lg.float(),ytr[ix].to(dev))
            opt.zero_grad(); scaler.scale(loss).backward(); scaler.unscale_(opt)
            nn.utils.clip_grad_norm_(model.parameters(),1.0); scaler.step(opt); scaler.update(); sch.step()
        lv=logits_of(model,'val'); f=f1_score(Y['val'],lv.argmax(1),average='macro')
        print(f'seed {seed} ep {ep+1} valMacroF1 {f:.4f} valAcc {accuracy_score(Y["val"],lv.argmax(1)):.4f} ({time.time()-t0:.0f}s)')
        if f>best: best=f; bstate={k:v.detach().clone() for k,v in model.state_dict().items()}
    model.load_state_dict(bstate)
    lv,lt=logits_of(model,'val'),logits_of(model,'test')
    np.save(P+f'ft_roberta_bz_val_logits_s{seed}.npy',lv); np.save(P+f'ft_roberta_bz_test_logits_s{seed}.npy',lt)
    pt=lt.argmax(1)
    r=dict(seed=seed,valF1=best,acc=accuracy_score(Y['test'],pt),macroF1=f1_score(Y['test'],pt,average='macro'))
    print(r); print(classification_report(Y['test'],pt,target_names=classes))
    del model,opt; torch.cuda.empty_cache(); return r

rows=[run(s) for s in (42,43,44)]
res=pd.DataFrame(rows); print(res.round(4)); print('mean',res[['acc','macroF1']].mean().round(4).to_dict(),'sd',res[['acc','macroF1']].std().round(4).to_dict())
res.to_csv(P+'clean_bugzilla_finetune_results.csv',index=False)

Mounted at /content/drive


config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

... (7 outputs left out)

seed 43 ep 1 valMacroF1 0.4079 valAcc 0.4539 (223s)
seed 43 ep 2 valMacroF1 0.4871 valAcc 0.5954 (223s)
seed 43 ep 3 valMacroF1 0.5144 valAcc 0.6808 (223s)
seed 43 ep 4 valMacroF1 0.4867 valAcc 0.5849 (223s)
{'seed': 43, 'valF1': 0.5143864468899529, 'acc': 0.6882582767838783, 'macroF1': 0.5188512936341897}
              precision    recall  f1-score   support

    Critical       0.41      0.67      0.51       571
        High       0.20      0.27      0.23       564
         Low       0.89      0.75      0.82      3728

    accuracy                           0.69      4863
   macro avg       0.50      0.57      0.52      4863
weighted avg       0.75      0.69      0.71      4863



Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
classifier.dense.weight    | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.weight | MISSING    | 
classifier.out_proj.bias   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
/tmp/ipykernel_646/1791065073.py:39: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler=torch.cuda.amp.GradScaler()

seed 44 ep 1 valMacroF1 0.5114 valAcc 0.6520 (223s)
seed 44 ep 2 valMacroF1 0.5189 valAcc 0.6705 (223s)
seed 44 ep 3 valMacroF1 0.5100 valAcc 0.6158 (223s)
seed 44 ep 4 valMacroF1 0.5005 valAcc 0.6041 (223s)
{'seed': 44, 'valF1': 0.5188669213568922, 'acc': 0.6847624922887107, 'macroF1': 0.5285167638421154}
              precision    recall  f1-score   support

    Critical       0.49      0.55      0.52       571
        High       0.20      0.37      0.25       564
         Low       0.89      0.75      0.82      3728

    accuracy                           0.68      4863
   macro avg       0.52      0.56      0.53      4863
weighted avg       0.76      0.68      0.72      4863

   seed   valF1     acc  macroF1
0    42  0.5119  0.6638   0.5110
1    43  0.5144  0.6883   0.5189
2    44  0.5189  0.6848   0.5285
mean {'acc': 0.6789, 'macroF1': 0.5195} sd {'acc': 0.0132, 'macroF1': 0.0088}


In [ ]:
# Step B: does distress/emotion information add signal beyond fine-tuned RoBERTa? (Bugzilla, real Severity)
# Emotion probs come from an independent pretrained emotion model (not trained on severity).
# Stacker is fit on VAL (CV for checking) and scored once on TEST. Uses saved logits from Step A (seeds 42-44).
from google.colab import drive
drive.mount('/content/drive')
import os, re, numpy as np, pandas as pd, torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_predict, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score, f1_score
from scipy.stats import wilcoxon

P='/content/drive/MyDrive/Colab Notebooks/'; dev='cuda'
va,te=[pd.read_csv(P+f'clean_bugzilla_{n}.csv') for n in ('val','test')]
for d in (va,te): d['text']=d['text'].fillna('').astype(str)
cid={'Critical':0,'High':1,'Low':2}; yv=va['y'].map(cid).values; yt=te['y'].map(cid).values

# 1) independent emotion probabilities (7 classes)
name='j-hartmann/emotion-english-distilroberta-base'
tok=AutoTokenizer.from_pretrained(name); em=AutoModelForSequenceClassification.from_pretrained(name).to(dev).eval().half()
@torch.no_grad()
def emo(texts,bs=128):
    out=[]
    for i in range(0,len(texts),bs):
        b=tok(list(texts[i:i+bs]),padding=True,truncation=True,max_length=256,return_tensors='pt').to(dev)
        out.append(torch.softmax(em(**b).logits.float(),-1).cpu().numpy())
    return np.vstack(out)
def cached(tag,d):
    f=P+f'emo_probs_bz_{tag}.npy'
    if not os.path.exists(f): np.save(f,emo(d.text.values))
    return np.load(f)
Ev,Et=cached('val',va),cached('test',te)
print('emotion labels:',em.config.id2label)

# 2) keyword DDS counts (same lexicon family as the conference paper, input-side only)
LEX={'frustration':['frustrating','annoying','broken','failed','crash','error'],
 'confusion':['confused','unclear','dont understand','unknown','complicated'],
 'anxiety':['worried','anxious','fear','unsafe','security'],
 'urgency':['urgent','immediately','asap','critical','important'],
 'helplessness':['stuck','unable','cannot','helpless','nothing works'],
 'anger':['hate','terrible','worst','angry','useless']}
def dds(d): return np.array([[sum(t.lower().count(w) for w in ws) for ws in LEX.values()] for t in d.text]),
Kv,Kt=dds(va)[0],dds(te)[0]
def ddsi(E,K): return np.c_[1-E[:,[3,4]].sum(1), K.sum(1)]   # crude: non-neutral/non-joy mass + keyword count

def fit_eval(Fv,Ft,seed=0):
    clf=make_pipeline(StandardScaler(),LogisticRegression(max_iter=2000,class_weight='balanced',C=0.5))
    cvp=cross_val_predict(clf,Fv,yv,cv=StratifiedKFold(5,shuffle=True,random_state=seed))
    clf.fit(Fv,yv); p=clf.predict(Ft)
    return f1_score(yv,cvp,average='macro'),accuracy_score(yt,p),f1_score(yt,p,average='macro')

rows=[]
r=fit_eval(np.c_[Ev,Kv,ddsi(Ev,Kv)],np.c_[Et,Kt,ddsi(Et,Kt)]); rows.append(('Distress features only',*r))
per_seed={}
for s in (42,43,44):
    Lv=np.load(P+f'ft_roberta_bz_val_logits_s{s}.npy'); Lt=np.load(P+f'ft_roberta_bz_test_logits_s{s}.npy')
    a=fit_eval(Lv,Lt); b=fit_eval(np.c_[Lv,Ev],np.c_[Lt,Et]); c=fit_eval(np.c_[Lv,Ev,Kv,ddsi(Ev,Kv)],np.c_[Lt,Et,Kt,ddsi(Et,Kt)])
    per_seed[s]=(a,b,c)
names=['RoBERTa logits only','RoBERTa + emotion probs','RoBERTa + emotion + DDS/DDSI']
for i,n in enumerate(names):
    arr=np.array([per_seed[s][i] for s in per_seed]); rows.append((n,*arr.mean(0)))
res=pd.DataFrame(rows,columns=['Config','val_CV_macroF1','test_Acc','test_MacroF1']); print(res.round(4).to_string(index=False))
d=[per_seed[s][2][2]-per_seed[s][0][2] for s in per_seed]; print('test MacroF1 gain (full - logits only) per seed:',np.round(d,4))
res.to_csv(P+'clean_bugzilla_distress_results.csv',index=False)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


config.json:   0%|          | 0.00/1.00k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/294 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/798k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

... (3 outputs left out)

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  329MB            

model.safetensors: downloading bytes:           |  0.00B            

emotion labels: {0: 'anger', 1: 'disgust', 2: 'fear', 3: 'joy', 4: 'neutral', 5: 'sadness', 6: 'surprise'}
                      Config  val_CV_macroF1  test_Acc  test_MacroF1
      Distress features only          0.4379    0.6167        0.4570
         RoBERTa logits only          0.4893    0.5972        0.4887
     RoBERTa + emotion probs          0.4860    0.5974        0.4879
RoBERTa + emotion + DDS/DDSI          0.4868    0.6019        0.4941
test MacroF1 gain (full - logits only) per seed: [0.008  0.0052 0.0029]


In [ ]:
# Baselines on the SAME clean Bugzilla split: LR, SVM, RF, NB, XGBoost, LSTM, BiLSTM, CNN+BiLSTM
# Class-balanced where supported; model selection on VAL only; test scored once. Saves test predictions.
from google.colab import drive
drive.mount('/content/drive')
import numpy as np, pandas as pd, tensorflow as tf
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import ComplementNB
from sklearn.utils.class_weight import compute_class_weight, compute_sample_weight
from sklearn.metrics import accuracy_score, f1_score
from xgboost import XGBClassifier

P='/content/drive/MyDrive/Colab Notebooks/'
tr,va,te=[pd.read_csv(P+f'clean_bugzilla_{n}.csv') for n in ('train','val','test')]
for d in (tr,va,te): d['text']=d['text'].fillna('').astype(str)
cid={'Critical':0,'High':1,'Low':2}
ytr,yva,yte=[d['y'].map(cid).values for d in (tr,va,te)]
tf_=TfidfVectorizer(max_features=20000,ngram_range=(1,2),min_df=2,sublinear_tf=True).fit(tr.text)
Xtr,Xva,Xte=[tf_.transform(d.text) for d in (tr,va,te)]
rows=[]; preds={}
def rec(name,seed,p):
    preds[f'{name}_s{seed}']=p
    rows.append(dict(Model=name,seed=seed,Acc=accuracy_score(yte,p),MacroF1=f1_score(yte,p,average='macro')))
    print(rows[-1])

for s in (42,43,44):
    rec('LogReg',s,LogisticRegression(max_iter=1000,class_weight='balanced',random_state=s).fit(Xtr,ytr).predict(Xte))
    rec('SVM',s,LinearSVC(class_weight='balanced',random_state=s).fit(Xtr,ytr).predict(Xte))
    rec('RandomForest',s,RandomForestClassifier(300,class_weight='balanced_subsample',n_jobs=-1,random_state=s).fit(Xtr,ytr).predict(Xte))
    rec('NaiveBayes',s,ComplementNB().fit(Xtr,ytr).predict(Xte))
    sw=compute_sample_weight('balanced',ytr)
    rec('XGBoost',s,XGBClassifier(n_estimators=300,max_depth=6,learning_rate=0.1,tree_method='hist',random_state=s,n_jobs=-1).fit(Xtr,ytr,sample_weight=sw).predict(Xte))

# --- Keras DL ---
MAXLEN,VOC=200,30000
tk=tf.keras.preprocessing.text.Tokenizer(num_words=VOC,oov_token='<unk>').fit_on_texts(list(tr.text)) or None
tk=tf.keras.preprocessing.text.Tokenizer(num_words=VOC,oov_token='<unk>'); tk.fit_on_texts(list(tr.text))
seq=lambda d: tf.keras.preprocessing.sequence.pad_sequences(tk.texts_to_sequences(list(d.text)),maxlen=MAXLEN)
Str,Sva,Ste=seq(tr),seq(va),seq(te)
cw=dict(enumerate(compute_class_weight('balanced',classes=np.arange(3),y=ytr)))
L=tf.keras.layers
def build(kind):
    m=tf.keras.Sequential([L.Embedding(VOC,128)])
    if kind=='LSTM': m.add(L.LSTM(64))
    elif kind=='BiLSTM': m.add(L.Bidirectional(L.LSTM(64)))
    else:
        m.add(L.Conv1D(64,5,activation='relu')); m.add(L.MaxPooling1D(2)); m.add(L.Bidirectional(L.LSTM(64)))
    m.add(L.Dropout(0.4)); m.add(L.Dense(3,activation='softmax'))
    m.compile('adam','sparse_categorical_crossentropy'); return m
class ValF1(tf.keras.callbacks.Callback):
    def on_train_begin(self,logs=None): self.best=-1; self.w=None
    def on_epoch_end(self,ep,logs=None):
        f=f1_score(yva,self.model.predict(Sva,verbose=0).argmax(1),average='macro')
        if f>self.best: self.best,self.w=f,self.model.get_weights()
for kind in ('LSTM','BiLSTM','CNN+BiLSTM'):
    for s in (42,43,44):
        tf.keras.utils.set_random_seed(s); m=build(kind); cb=ValF1()
        m.fit(Str,ytr,validation_data=(Sva,yva),epochs=8,batch_size=64,class_weight=cw,callbacks=[cb],verbose=0)
        m.set_weights(cb.w); rec(kind,s,m.predict(Ste,verbose=0).argmax(1))

res=pd.DataFrame(rows); res.to_csv(P+'clean_bugzilla_baselines_seedwise.csv',index=False)
np.savez(P+'clean_bugzilla_baseline_preds.npz',y_test=yte,**preds)
print(res.groupby('Model')[['Acc','MacroF1']].agg(['mean','std']).round(4))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
{'Model': 'LogReg', 'seed': 42, 'Acc': 0.6985399958873123, 'MacroF1': 0.5357635002031954}
{'Model': 'SVM', 'seed': 42, 'Acc': 0.7542669134279252, 'MacroF1': 0.5330497666406299}
{'Model': 'RandomForest', 'seed': 42, 'Acc': 0.7884022208513264, 'MacroF1': 0.4365753097288147}
{'Model': 'NaiveBayes', 'seed': 42, 'Acc': 0.7571457947768867, 'MacroF1': 0.48593900365308046}
{'Model': 'XGBoost', 'seed': 42, 'Acc': 0.7452190006169032, 'MacroF1': 0.5475995327600262}
{'Model': 'LogReg', 'seed': 43, 'Acc': 0.6985399958873123, 'MacroF1': 0.5357635002031954}
{'Model': 'SVM', 'seed': 43, 'Acc': 0.7542669134279252, 'MacroF1': 0.5330497666406299}
{'Model': 'RandomForest', 'seed': 43, 'Acc': 0.7865515114127082, 'MacroF1': 0.42995647026174844}
{'Model': 'NaiveBayes', 'seed': 43, 'Acc': 0.7571457947768867, 'MacroF1': 0.48593900365308046}
{'Model': 'XGBoost', 'seed': 43, 'Acc': 0.7

In [ ]:
# Step A: fine-tune RoBERTa end-to-end on clean Bugzilla split (real Severity, class-weighted loss)
# Colab T4. Model chosen on VAL macro-F1; test scored once per seed. Saves logits for later stacking.
from google.colab import drive
drive.mount('/content/drive')
import numpy as np, pandas as pd, torch, torch.nn as nn, time
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from sklearn.metrics import accuracy_score, f1_score, classification_report
from sklearn.utils.class_weight import compute_class_weight

# Set these two, run once for BERT and once for CodeBERT:
MODEL='bert-base-uncased'   # or 'microsoft/codebert-base'
TAG='bert'                  # or 'codebert'
P='/content/drive/MyDrive/Colab Notebooks/'
dev='cuda'
tr,va,te=[pd.read_csv(P+f'clean_bugzilla_{n}.csv') for n in ('train','val','test')]
for d in (tr,va,te): d['text']=d['text'].fillna('').astype(str)
classes=['Critical','High','Low']; cid={c:i for i,c in enumerate(classes)}
Y={n:d['y'].map(cid).values for n,d in (('train',tr),('val',va),('test',te))}
tok=AutoTokenizer.from_pretrained(MODEL); MAXLEN=192; BS=32

def enc(d):
    e=tok(list(d.text),truncation=True,max_length=MAXLEN,padding='max_length',return_tensors='pt')
    return e['input_ids'],e['attention_mask']
E={n:enc(d) for n,d in (('train',tr),('val',va),('test',te))}

@torch.no_grad()
def logits_of(model,n):
    model.eval(); ids,am=E[n]; out=[]
    for i in range(0,len(ids),128):
        with torch.autocast('cuda',dtype=torch.float16):
            out.append(model(input_ids=ids[i:i+128].to(dev),attention_mask=am[i:i+128].to(dev)).logits.float().cpu())
    return torch.cat(out).numpy()

def run(seed,epochs=4,lr=2e-5):
    torch.manual_seed(seed); np.random.seed(seed)
    model=AutoModelForSequenceClassification.from_pretrained(MODEL,num_labels=3).to(dev)
    w=torch.tensor(compute_class_weight('balanced',classes=np.arange(3),y=Y['train']),dtype=torch.float32,device=dev)
    lossf=nn.CrossEntropyLoss(weight=w)
    opt=torch.optim.AdamW(model.parameters(),lr=lr,weight_decay=0.01)
    ids,am=E['train']; ytr=torch.tensor(Y['train'])
    steps=epochs*((len(ids)+BS-1)//BS); sch=get_linear_schedule_with_warmup(opt,int(0.06*steps),steps)
    scaler=torch.cuda.amp.GradScaler(); best,bstate=-1,None
    for ep in range(epochs):
        model.train(); perm=torch.randperm(len(ids)); t0=time.time()
        for i in range(0,len(perm),BS):
            ix=perm[i:i+BS]
            with torch.autocast('cuda',dtype=torch.float16):
                lg=model(input_ids=ids[ix].to(dev),attention_mask=am[ix].to(dev)).logits
            loss=lossf(lg.float(),ytr[ix].to(dev))
            opt.zero_grad(); scaler.scale(loss).backward(); scaler.unscale_(opt)
            nn.utils.clip_grad_norm_(model.parameters(),1.0); scaler.step(opt); scaler.update(); sch.step()
        lv=logits_of(model,'val'); f=f1_score(Y['val'],lv.argmax(1),average='macro')
        print(f'seed {seed} ep {ep+1} valMacroF1 {f:.4f} valAcc {accuracy_score(Y["val"],lv.argmax(1)):.4f} ({time.time()-t0:.0f}s)')
        if f>best: best=f; bstate={k:v.detach().clone() for k,v in model.state_dict().items()}
    model.load_state_dict(bstate)
    lv,lt=logits_of(model,'val'),logits_of(model,'test')
    np.save(P+f'ft_{TAG}_bz_val_logits_s{seed}.npy',lv); np.save(P+f'ft_{TAG}_bz_test_logits_s{seed}.npy',lt)
    pt=lt.argmax(1)
    r=dict(seed=seed,valF1=best,acc=accuracy_score(Y['test'],pt),macroF1=f1_score(Y['test'],pt,average='macro'))
    print(r); print(classification_report(Y['test'],pt,target_names=classes))
    del model,opt; torch.cuda.empty_cache(); return r

rows=[run(s) for s in (42,43,44)]
res=pd.DataFrame(rows); print(res.round(4)); print('mean',res[['acc','macroF1']].mean().round(4).to_dict(),'sd',res[['acc','macroF1']].std().round(4).to_dict())
res.to_csv(P+f'clean_bugzilla_finetune_{TAG}_results.csv',index=False)

Mounted at /content/drive


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

... (6 outputs left out)

seed 43 ep 1 valMacroF1 0.4814 valAcc 0.6440 (215s)
seed 43 ep 2 valMacroF1 0.4814 valAcc 0.6604 (215s)
seed 43 ep 3 valMacroF1 0.4635 valAcc 0.5263 (214s)
seed 43 ep 4 valMacroF1 0.5052 valAcc 0.6456 (215s)
{'seed': 43, 'valF1': 0.505215378029921, 'acc': 0.644458153403249, 'macroF1': 0.5045313769418556}
              precision    recall  f1-score   support

    Critical       0.41      0.57      0.47       571
        High       0.19      0.40      0.26       564
         Low       0.89      0.69      0.78      3728

    accuracy                           0.64      4863
   macro avg       0.50      0.55      0.50      4863
weighted avg       0.75      0.64      0.68      4863



Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
/tmp/ipykernel_633/354

seed 44 ep 1 valMacroF1 0.4492 valAcc 0.5522 (214s)
seed 44 ep 2 valMacroF1 0.4999 valAcc 0.6242 (215s)
seed 44 ep 3 valMacroF1 0.4910 valAcc 0.5903 (214s)
seed 44 ep 4 valMacroF1 0.5011 valAcc 0.6246 (215s)
{'seed': 44, 'valF1': 0.5011148558258385, 'acc': 0.6282130372198231, 'macroF1': 0.4976391346710074}
              precision    recall  f1-score   support

    Critical       0.39      0.57      0.46       571
        High       0.19      0.43      0.27       564
         Low       0.90      0.67      0.77      3728

    accuracy                           0.63      4863
   macro avg       0.49      0.56      0.50      4863
weighted avg       0.76      0.63      0.67      4863

   seed   valF1     acc  macroF1
0    42  0.5031  0.6245   0.5010
1    43  0.5052  0.6445   0.5045
2    44  0.5011  0.6282   0.4976
mean {'acc': 0.6324, 'macroF1': 0.5011} sd {'acc': 0.0106, 'macroF1': 0.0034}


In [ ]:
# Step A: fine-tune RoBERTa end-to-end on clean Bugzilla split (real Severity, class-weighted loss)
# Colab T4. Model chosen on VAL macro-F1; test scored once per seed. Saves logits for later stacking.
from google.colab import drive
drive.mount('/content/drive')
import numpy as np, pandas as pd, torch, torch.nn as nn, time
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from sklearn.metrics import accuracy_score, f1_score, classification_report
from sklearn.utils.class_weight import compute_class_weight

# Set these two, run once for BERT and once for CodeBERT:
MODEL='microsoft/codebert-base'
TAG='codebert'
P='/content/drive/MyDrive/Colab Notebooks/'
dev='cuda'
tr,va,te=[pd.read_csv(P+f'clean_bugzilla_{n}.csv') for n in ('train','val','test')]
for d in (tr,va,te): d['text']=d['text'].fillna('').astype(str)
classes=['Critical','High','Low']; cid={c:i for i,c in enumerate(classes)}
Y={n:d['y'].map(cid).values for n,d in (('train',tr),('val',va),('test',te))}
tok=AutoTokenizer.from_pretrained(MODEL); MAXLEN=192; BS=32

def enc(d):
    e=tok(list(d.text),truncation=True,max_length=MAXLEN,padding='max_length',return_tensors='pt')
    return e['input_ids'],e['attention_mask']
E={n:enc(d) for n,d in (('train',tr),('val',va),('test',te))}

@torch.no_grad()
def logits_of(model,n):
    model.eval(); ids,am=E[n]; out=[]
    for i in range(0,len(ids),128):
        with torch.autocast('cuda',dtype=torch.float16):
            out.append(model(input_ids=ids[i:i+128].to(dev),attention_mask=am[i:i+128].to(dev)).logits.float().cpu())
    return torch.cat(out).numpy()

def run(seed,epochs=4,lr=2e-5):
    torch.manual_seed(seed); np.random.seed(seed)
    model=AutoModelForSequenceClassification.from_pretrained(MODEL,num_labels=3).to(dev)
    w=torch.tensor(compute_class_weight('balanced',classes=np.arange(3),y=Y['train']),dtype=torch.float32,device=dev)
    lossf=nn.CrossEntropyLoss(weight=w)
    opt=torch.optim.AdamW(model.parameters(),lr=lr,weight_decay=0.01)
    ids,am=E['train']; ytr=torch.tensor(Y['train'])
    steps=epochs*((len(ids)+BS-1)//BS); sch=get_linear_schedule_with_warmup(opt,int(0.06*steps),steps)
    scaler=torch.cuda.amp.GradScaler(); best,bstate=-1,None
    for ep in range(epochs):
        model.train(); perm=torch.randperm(len(ids)); t0=time.time()
        for i in range(0,len(perm),BS):
            ix=perm[i:i+BS]
            with torch.autocast('cuda',dtype=torch.float16):
                lg=model(input_ids=ids[ix].to(dev),attention_mask=am[ix].to(dev)).logits
            loss=lossf(lg.float(),ytr[ix].to(dev))
            opt.zero_grad(); scaler.scale(loss).backward(); scaler.unscale_(opt)
            nn.utils.clip_grad_norm_(model.parameters(),1.0); scaler.step(opt); scaler.update(); sch.step()
        lv=logits_of(model,'val'); f=f1_score(Y['val'],lv.argmax(1),average='macro')
        print(f'seed {seed} ep {ep+1} valMacroF1 {f:.4f} valAcc {accuracy_score(Y["val"],lv.argmax(1)):.4f} ({time.time()-t0:.0f}s)')
        if f>best: best=f; bstate={k:v.detach().clone() for k,v in model.state_dict().items()}
    model.load_state_dict(bstate)
    lv,lt=logits_of(model,'val'),logits_of(model,'test')
    np.save(P+f'ft_{TAG}_bz_val_logits_s{seed}.npy',lv); np.save(P+f'ft_{TAG}_bz_test_logits_s{seed}.npy',lt)
    pt=lt.argmax(1)
    r=dict(seed=seed,valF1=best,acc=accuracy_score(Y['test'],pt),macroF1=f1_score(Y['test'],pt,average='macro'))
    print(r); print(classification_report(Y['test'],pt,target_names=classes))
    del model,opt; torch.cuda.empty_cache(); return r

rows=[run(s) for s in (42,43,44)]
res=pd.DataFrame(rows); print(res.round(4)); print('mean',res[['acc','macroF1']].mean().round(4).to_dict(),'sd',res[['acc','macroF1']].std().round(4).to_dict())
res.to_csv(P+f'clean_bugzilla_finetune_{TAG}_results.csv',index=False)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


config.json:   0%|          | 0.00/498 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/150 [00:00<?, ?B/s]

... (8 outputs left out)

[transformers] RobertaForSequenceClassification LOAD REPORT from: microsoft/codebert-base
Key                        | Status     | 
---------------------------+------------+-
pooler.dense.weight        | UNEXPECTED | 
pooler.dense.bias          | UNEXPECTED | 
classifier.out_proj.bias   | MISSING    | 
classifier.out_proj.weight | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.dense.weight    | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
/tmp/ipykernel_633/373693269.py:42: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler=torch.cuda.amp.GradScaler(); best,bstate=-1,None


seed 43 ep 1 valMacroF1 0.4334 valAcc 0.4774 (219s)
seed 43 ep 2 valMacroF1 0.5011 valAcc 0.5829 (219s)
seed 43 ep 3 valMacroF1 0.5328 valAcc 0.7020 (219s)
seed 43 ep 4 valMacroF1 0.5332 valAcc 0.6538 (219s)
{'seed': 43, 'valF1': 0.5331519428881846, 'acc': 0.6514497223935842, 'macroF1': 0.5213370974093322}
              precision    recall  f1-score   support

    Critical       0.48      0.55      0.51       571
        High       0.19      0.44      0.27       564
         Low       0.89      0.70      0.78      3728

    accuracy                           0.65      4863
   macro avg       0.52      0.56      0.52      4863
weighted avg       0.76      0.65      0.69      4863



Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: microsoft/codebert-base
Key                        | Status     | 
---------------------------+------------+-
pooler.dense.weight        | UNEXPECTED | 
pooler.dense.bias          | UNEXPECTED | 
classifier.out_proj.bias   | MISSING    | 
classifier.out_proj.weight | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.dense.weight    | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
/tmp/ipykernel_633/373693269.py:42: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler=torch.cuda.amp.GradScaler(); best,bstate=-1,None


In [ ]:
# Baselines on the SAME clean Bugzilla split: LR, SVM, RF, NB, XGBoost, LSTM, BiLSTM, CNN+BiLSTM
# Class-balanced where supported; model selection on VAL only; test scored once. Saves test predictions.
from google.colab import drive
drive.mount('/content/drive')
import numpy as np, pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import ComplementNB
from sklearn.utils.class_weight import compute_class_weight, compute_sample_weight
from sklearn.metrics import accuracy_score, f1_score
from xgboost import XGBClassifier

P='/content/drive/MyDrive/Colab Notebooks/'
tr,va,te=[pd.read_csv(P+f'clean_bugzilla_{n}.csv') for n in ('train','val','test')]
for d in (tr,va,te): d['text']=d['text'].fillna('').astype(str)
cid={'Critical':0,'High':1,'Low':2}
ytr,yva,yte=[d['y'].map(cid).values for d in (tr,va,te)]
tf_=TfidfVectorizer(max_features=20000,ngram_range=(1,2),min_df=2,sublinear_tf=True).fit(tr.text)
Xtr,Xva,Xte=[tf_.transform(d.text) for d in (tr,va,te)]
rows=[]; preds={}
def rec(name,seed,p):
    preds[f'{name}_s{seed}']=p
    rows.append(dict(Model=name,seed=seed,Acc=accuracy_score(yte,p),MacroF1=f1_score(yte,p,average='macro')))
    print(rows[-1])

for s in (42,43,44):
    rec('LogReg',s,LogisticRegression(max_iter=1000,class_weight='balanced',random_state=s).fit(Xtr,ytr).predict(Xte))
    rec('SVM',s,LinearSVC(class_weight='balanced',random_state=s).fit(Xtr,ytr).predict(Xte))
    rec('RandomForest',s,RandomForestClassifier(300,class_weight='balanced_subsample',n_jobs=-1,random_state=s).fit(Xtr,ytr).predict(Xte))
    rec('NaiveBayes',s,ComplementNB().fit(Xtr,ytr).predict(Xte))
    sw=compute_sample_weight('balanced',ytr)
    rec('XGBoost',s,XGBClassifier(n_estimators=300,max_depth=6,learning_rate=0.1,tree_method='hist',random_state=s,n_jobs=-1).fit(Xtr,ytr,sample_weight=sw).predict(Xte))


res=pd.DataFrame(rows); res.to_csv(P+'clean_bugzilla_classic_seedwise.csv',index=False)
np.savez(P+'clean_bugzilla_classic_preds.npz',y_test=yte,**preds)
print(res.groupby('Model')[['Acc','MacroF1']].agg(['mean','std']).round(4))

Mounted at /content/drive
{'Model': 'LogReg', 'seed': 42, 'Acc': 0.6985399958873123, 'MacroF1': 0.5357635002031954}
{'Model': 'SVM', 'seed': 42, 'Acc': 0.7542669134279252, 'MacroF1': 0.5330497666406299}
{'Model': 'RandomForest', 'seed': 42, 'Acc': 0.7884022208513264, 'MacroF1': 0.4365753097288147}
{'Model': 'NaiveBayes', 'seed': 42, 'Acc': 0.7571457947768867, 'MacroF1': 0.48593900365308046}
{'Model': 'XGBoost', 'seed': 42, 'Acc': 0.7452190006169032, 'MacroF1': 0.5475995327600262}
{'Model': 'LogReg', 'seed': 43, 'Acc': 0.6985399958873123, 'MacroF1': 0.5357635002031954}
{'Model': 'SVM', 'seed': 43, 'Acc': 0.7542669134279252, 'MacroF1': 0.5330497666406299}
{'Model': 'RandomForest', 'seed': 43, 'Acc': 0.7865515114127082, 'MacroF1': 0.42995647026174844}
{'Model': 'NaiveBayes', 'seed': 43, 'Acc': 0.7571457947768867, 'MacroF1': 0.48593900365308046}
{'Model': 'XGBoost', 'seed': 43, 'Acc': 0.7452190006169032, 'MacroF1': 0.5475995327600262}
{'Model': 'LogReg', 'seed': 44, 'Acc': 0.69853999588731

In [ ]:
# Baselines on the SAME clean Bugzilla split: LR, SVM, RF, NB, XGBoost, LSTM, BiLSTM, CNN+BiLSTM
# Class-balanced where supported; model selection on VAL only; test scored once. Saves test predictions.
from google.colab import drive
drive.mount('/content/drive')
import numpy as np, pandas as pd, tensorflow as tf
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import ComplementNB
from sklearn.utils.class_weight import compute_class_weight, compute_sample_weight
from sklearn.metrics import accuracy_score, f1_score


P='/content/drive/MyDrive/Colab Notebooks/'
tr,va,te=[pd.read_csv(P+f'clean_bugzilla_{n}.csv') for n in ('train','val','test')]
for d in (tr,va,te): d['text']=d['text'].fillna('').astype(str)
cid={'Critical':0,'High':1,'Low':2}
ytr,yva,yte=[d['y'].map(cid).values for d in (tr,va,te)]
rows=[]; preds={}
def rec(name,seed,p):
    preds[f'{name}_s{seed}']=p
    rows.append(dict(Model=name,seed=seed,Acc=accuracy_score(yte,p),MacroF1=f1_score(yte,p,average='macro')))
    print(rows[-1])
# --- Keras DL ---
MAXLEN,VOC=200,30000
tk=tf.keras.preprocessing.text.Tokenizer(num_words=VOC,oov_token='<unk>'); tk.fit_on_texts(list(tr.text))
seq=lambda d: tf.keras.preprocessing.sequence.pad_sequences(tk.texts_to_sequences(list(d.text)),maxlen=MAXLEN)
Str,Sva,Ste=seq(tr),seq(va),seq(te)
cw=dict(enumerate(compute_class_weight('balanced',classes=np.arange(3),y=ytr)))
L=tf.keras.layers
def build(kind):
    m=tf.keras.Sequential([L.Embedding(VOC,128)])
    if kind=='LSTM': m.add(L.LSTM(64))
    elif kind=='BiLSTM': m.add(L.Bidirectional(L.LSTM(64)))
    else:
        m.add(L.Conv1D(64,5,activation='relu')); m.add(L.MaxPooling1D(2)); m.add(L.Bidirectional(L.LSTM(64)))
    m.add(L.Dropout(0.4)); m.add(L.Dense(3,activation='softmax'))
    m.compile('adam','sparse_categorical_crossentropy'); return m
class ValF1(tf.keras.callbacks.Callback):
    def on_train_begin(self,logs=None): self.best=-1; self.w=None
    def on_epoch_end(self,ep,logs=None):
        f=f1_score(yva,self.model.predict(Sva,verbose=0).argmax(1),average='macro')
        if f>self.best: self.best,self.w=f,self.model.get_weights()
for kind in ('LSTM','BiLSTM','CNN+BiLSTM'):
    for s in (42,43,44):
        tf.keras.utils.set_random_seed(s); m=build(kind); cb=ValF1()
        m.fit(Str,ytr,validation_data=(Sva,yva),epochs=8,batch_size=64,class_weight=cw,callbacks=[cb],verbose=0)
        m.set_weights(cb.w); rec(kind,s,m.predict(Ste,verbose=0).argmax(1))

res=pd.DataFrame(rows); res.to_csv(P+'clean_bugzilla_dl_seedwise.csv',index=False)
np.savez(P+'clean_bugzilla_dl_preds.npz',y_test=yte,**preds)
print(res.groupby('Model')[['Acc','MacroF1']].agg(['mean','std']).round(4))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
{'Model': 'LSTM', 'seed': 42, 'Acc': 0.6454863253135924, 'MacroF1': 0.4571940876722511}
{'Model': 'LSTM', 'seed': 43, 'Acc': 0.6214271026115566, 'MacroF1': 0.44656536452229617}
{'Model': 'LSTM', 'seed': 44, 'Acc': 0.7365823565700185, 'MacroF1': 0.4718722456705336}
{'Model': 'BiLSTM', 'seed': 42, 'Acc': 0.6551511412708205, 'MacroF1': 0.4821266379536538}
{'Model': 'BiLSTM', 'seed': 43, 'Acc': 0.6129960929467407, 'MacroF1': 0.44391633786828805}
{'Model': 'BiLSTM', 'seed': 44, 'Acc': 0.6253341558708616, 'MacroF1': 0.45020418209448837}
{'Model': 'CNN+BiLSTM', 'seed': 42, 'Acc': 0.7217766810610734, 'MacroF1': 0.4940024305782083}
{'Model': 'CNN+BiLSTM', 'seed': 43, 'Acc': 0.6977174583590376, 'MacroF1': 0.4939938125825951}
{'Model': 'CNN+BiLSTM', 'seed': 44, 'Acc': 0.6372609500308452, 'MacroF1': 0.46486199952246204}
               Acc         MacroF1        
        

In [ ]:
# Step A: fine-tune RoBERTa end-to-end on clean Bugzilla split (real Severity, class-weighted loss)
# Colab T4. Model chosen on VAL macro-F1; test scored once per seed. Saves logits for later stacking.
from google.colab import drive
drive.mount('/content/drive')
import numpy as np, pandas as pd, torch, torch.nn as nn, time
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from sklearn.metrics import accuracy_score, f1_score, classification_report
from sklearn.utils.class_weight import compute_class_weight

# Set these two, run once for BERT and once for CodeBERT:
MODEL='microsoft/codebert-base'
TAG='codebert'
P='/content/drive/MyDrive/Colab Notebooks/'
dev='cuda'
tr,va,te=[pd.read_csv(P+f'clean_bugzilla_{n}.csv') for n in ('train','val','test')]
for d in (tr,va,te): d['text']=d['text'].fillna('').astype(str)
classes=['Critical','High','Low']; cid={c:i for i,c in enumerate(classes)}
Y={n:d['y'].map(cid).values for n,d in (('train',tr),('val',va),('test',te))}
tok=AutoTokenizer.from_pretrained(MODEL); MAXLEN=192; BS=32

def enc(d):
    e=tok(list(d.text),truncation=True,max_length=MAXLEN,padding='max_length',return_tensors='pt')
    return e['input_ids'],e['attention_mask']
E={n:enc(d) for n,d in (('train',tr),('val',va),('test',te))}

@torch.no_grad()
def logits_of(model,n):
    model.eval(); ids,am=E[n]; out=[]
    for i in range(0,len(ids),128):
        with torch.autocast('cuda',dtype=torch.float16):
            out.append(model(input_ids=ids[i:i+128].to(dev),attention_mask=am[i:i+128].to(dev)).logits.float().cpu())
    return torch.cat(out).numpy()

def run(seed,epochs=4,lr=2e-5):
    torch.manual_seed(seed); np.random.seed(seed)
    model=AutoModelForSequenceClassification.from_pretrained(MODEL,num_labels=3).to(dev)
    w=torch.tensor(compute_class_weight('balanced',classes=np.arange(3),y=Y['train']),dtype=torch.float32,device=dev)
    lossf=nn.CrossEntropyLoss(weight=w)
    opt=torch.optim.AdamW(model.parameters(),lr=lr,weight_decay=0.01)
    ids,am=E['train']; ytr=torch.tensor(Y['train'])
    steps=epochs*((len(ids)+BS-1)//BS); sch=get_linear_schedule_with_warmup(opt,int(0.06*steps),steps)
    scaler=torch.cuda.amp.GradScaler(); best,bstate=-1,None
    for ep in range(epochs):
        model.train(); perm=torch.randperm(len(ids)); t0=time.time()
        for i in range(0,len(perm),BS):
            ix=perm[i:i+BS]
            with torch.autocast('cuda',dtype=torch.float16):
                lg=model(input_ids=ids[ix].to(dev),attention_mask=am[ix].to(dev)).logits
            loss=lossf(lg.float(),ytr[ix].to(dev))
            opt.zero_grad(); scaler.scale(loss).backward(); scaler.unscale_(opt)
            nn.utils.clip_grad_norm_(model.parameters(),1.0); scaler.step(opt); scaler.update(); sch.step()
        lv=logits_of(model,'val'); f=f1_score(Y['val'],lv.argmax(1),average='macro')
        print(f'seed {seed} ep {ep+1} valMacroF1 {f:.4f} valAcc {accuracy_score(Y["val"],lv.argmax(1)):.4f} ({time.time()-t0:.0f}s)')
        if f>best: best=f; bstate={k:v.detach().clone() for k,v in model.state_dict().items()}
    model.load_state_dict(bstate)
    lv,lt=logits_of(model,'val'),logits_of(model,'test')
    np.save(P+f'ft_{TAG}_bz_val_logits_s{seed}.npy',lv); np.save(P+f'ft_{TAG}_bz_test_logits_s{seed}.npy',lt)
    pt=lt.argmax(1)
    r=dict(seed=seed,valF1=best,acc=accuracy_score(Y['test'],pt),macroF1=f1_score(Y['test'],pt,average='macro'))
    print(r); print(classification_report(Y['test'],pt,target_names=classes))
    del model,opt; torch.cuda.empty_cache(); return r

rows=[run(s) for s in (42,43,44)]
res=pd.DataFrame(rows); print(res.round(4)); print('mean',res[['acc','macroF1']].mean().round(4).to_dict(),'sd',res[['acc','macroF1']].std().round(4).to_dict())
res.to_csv(P+f'clean_bugzilla_finetune_{TAG}_results.csv',index=False)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: microsoft/codebert-base
Key                        | Status     | 
---------------------------+------------+-
pooler.dense.bias          | UNEXPECTED | 
pooler.dense.weight        | UNEXPECTED | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.weight | MISSING    | 
classifier.dense.bias      | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


model.safetensors: reconstructing file:   0%|          |  0.00B /  499MB            

model.safetensors: downloading bytes:           |  0.00B            

/tmp/ipykernel_32631/373693269.py:42: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler=torch.cuda.amp.GradScaler(); best,bstate=-1,None


... (3 outputs left out)

seed 43 ep 1 valMacroF1 0.4339 valAcc 0.4778 (231s)
seed 43 ep 2 valMacroF1 0.5003 valAcc 0.5802 (231s)
seed 43 ep 3 valMacroF1 0.5317 valAcc 0.7007 (231s)
seed 43 ep 4 valMacroF1 0.5345 valAcc 0.6510 (231s)
{'seed': 43, 'valF1': 0.53449540839566, 'acc': 0.6481595722804853, 'macroF1': 0.5198795763727114}
              precision    recall  f1-score   support

    Critical       0.48      0.55      0.51       571
        High       0.19      0.45      0.27       564
         Low       0.89      0.69      0.78      3728

    accuracy                           0.65      4863
   macro avg       0.52      0.56      0.52      4863
weighted avg       0.76      0.65      0.69      4863



Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: microsoft/codebert-base
Key                        | Status     | 
---------------------------+------------+-
pooler.dense.bias          | UNEXPECTED | 
pooler.dense.weight        | UNEXPECTED | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.weight | MISSING    | 
classifier.dense.bias      | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
/tmp/ipykernel_32631/373693269.py:42: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler=torch.cuda.amp.GradScaler(); best,bstate=-1,None


seed 44 ep 1 valMacroF1 0.5079 valAcc 0.6917 (231s)
seed 44 ep 2 valMacroF1 0.5132 valAcc 0.6703 (231s)
seed 44 ep 3 valMacroF1 0.5140 valAcc 0.6294 (231s)
seed 44 ep 4 valMacroF1 0.5180 valAcc 0.6337 (231s)
{'seed': 44, 'valF1': 0.5180245957959214, 'acc': 0.6419905408184249, 'macroF1': 0.5125927082528051}
              precision    recall  f1-score   support

    Critical       0.43      0.60      0.50       571
        High       0.19      0.42      0.27       564
         Low       0.90      0.68      0.78      3728

    accuracy                           0.64      4863
   macro avg       0.51      0.57      0.51      4863
weighted avg       0.76      0.64      0.68      4863

   seed   valF1     acc  macroF1
0    42  0.5181  0.6844   0.5274
1    43  0.5345  0.6482   0.5199
2    44  0.5180  0.6420   0.5126
mean {'acc': 0.6582, 'macroF1': 0.52} sd {'acc': 0.0229, 'macroF1': 0.0074}


In [ ]:
# Step 2a: fair test of distress features on the STRONGEST baseline (TF-IDF + XGBoost / LogReg), Bugzilla clean split.
from google.colab import drive
drive.mount('/content/drive')
import os, numpy as np, pandas as pd, torch, scipy.sparse as sp
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import accuracy_score, f1_score
from xgboost import XGBClassifier

P='/content/drive/MyDrive/Colab Notebooks/'; dev='cuda' if torch.cuda.is_available() else 'cpu'
tr,va,te=[pd.read_csv(P+f'clean_bugzilla_{n}.csv') for n in ('train','val','test')]
for d in (tr,va,te): d['text']=d['text'].fillna('').astype(str)
cid={'Critical':0,'High':1,'Low':2}; ytr,yva,yte=[d['y'].map(cid).values for d in (tr,va,te)]

name='j-hartmann/emotion-english-distilroberta-base'
tok=AutoTokenizer.from_pretrained(name); em=AutoModelForSequenceClassification.from_pretrained(name).to(dev).eval()
if dev=='cuda': em=em.half()
@torch.no_grad()
def emo(texts,bs=128):
    out=[]
    for i in range(0,len(texts),bs):
        b=tok(list(texts[i:i+bs]),padding=True,truncation=True,max_length=256,return_tensors='pt').to(dev)
        out.append(torch.softmax(em(**b).logits.float(),-1).cpu().numpy())
    return np.vstack(out)
def cached(tag,d):
    f=P+f'emo_probs_bz_{tag}.npy'
    if not os.path.exists(f): np.save(f,emo(d.text.values))
    return np.load(f)
E={n:cached(n,d) for n,d in (('train',tr),('val',va),('test',te))}

LEX={'frustration':['frustrating','annoying','broken','failed','crash','error'],'confusion':['confused','unclear','dont understand','unknown','complicated'],
 'anxiety':['worried','anxious','fear','unsafe','security'],'urgency':['urgent','immediately','asap','critical','important'],
 'helplessness':['stuck','unable','cannot','helpless','nothing works'],'anger':['hate','terrible','worst','angry','useless']}
def dds(d): return np.array([[sum(t.lower().count(w) for w in ws) for ws in LEX.values()] for t in d.text])
K={n:dds(d) for n,d in (('train',tr),('val',va),('test',te))}
D={n:np.c_[E[n],K[n],K[n].sum(1)] for n in E}
sc=StandardScaler().fit(D['train']); D={n:sc.transform(D[n]) for n in D}

tf=TfidfVectorizer(max_features=20000,ngram_range=(1,2),min_df=2,sublinear_tf=True).fit(tr.text)
T={n:tf.transform(d.text) for n,d in (('train',tr),('val',va),('test',te))}
def feats(n,use): return sp.hstack([T[n],sp.csr_matrix(D[n])]).tocsr() if use else T[n]

sw=compute_sample_weight('balanced',ytr); rows=[]; preds={}
for use in (False,True):
    tag='+distress' if use else 'base'
    Xtr,Xte=feats('train',use),feats('test',use)
    lr=LogisticRegression(max_iter=2000,class_weight='balanced').fit(Xtr,ytr); p=lr.predict(Xte)
    preds[f'LogReg_{tag}']=p; rows.append(dict(Model='LogReg',Feat=tag,seed=0,Acc=accuracy_score(yte,p),MacroF1=f1_score(yte,p,average='macro')))
    for s in (42,43,44,45,46):
        x=XGBClassifier(n_estimators=300,max_depth=6,learning_rate=0.1,subsample=0.8,colsample_bytree=0.8,tree_method='hist',random_state=s,n_jobs=-1).fit(Xtr,ytr,sample_weight=sw)
        p=x.predict(Xte); preds[f'XGBoost_{tag}_s{s}']=p
        rows.append(dict(Model='XGBoost',Feat=tag,seed=s,Acc=accuracy_score(yte,p),MacroF1=f1_score(yte,p,average='macro')))
res=pd.DataFrame(rows); print(res.groupby(['Model','Feat'])[['Acc','MacroF1']].agg(['mean','std']).round(4))
res.to_csv(P+'clean_bugzilla_distress_on_xgb.csv',index=False); np.savez(P+'clean_bugzilla_distress_on_xgb_preds.npz',y_test=yte,**preds)

# paired bootstrap: XGBoost +distress vs base (seed 42)
rng=np.random.default_rng(0); a,b=preds['XGBoost_+distress_s42'],preds['XGBoost_base_s42']; diffs=[]
for _ in range(2000):
    i=rng.integers(0,len(yte),len(yte)); diffs.append(f1_score(yte[i],a[i],average='macro')-f1_score(yte[i],b[i],average='macro'))
print('XGB +distress minus base, MacroF1: mean %.4f  95%% CI [%.4f, %.4f]  P(diff<=0)=%.3f'%(np.mean(diffs),*np.percentile(diffs,[2.5,97.5]),np.mean(np.array(diffs)<=0)))

Mounted at /content/drive


config.json:   0%|          | 0.00/1.00k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/294 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/798k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

... (3 outputs left out)

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  329MB            

model.safetensors: downloading bytes:           |  0.00B            

                      Acc         MacroF1        
                     mean     std    mean     std
Model   Feat                                     
LogReg  +distress  0.7012     NaN  0.5345     NaN
        base       0.6985     NaN  0.5358     NaN
XGBoost +distress  0.7515  0.0013  0.5557  0.0046
        base       0.7491  0.0019  0.5494  0.0039
XGB +distress minus base, MacroF1: mean 0.0103  95% CI [-0.0009, 0.0219]  P(diff<=0)=0.037


In [ ]:
# Step 2b: final 11-baseline table (mean±SD) + paired bootstrap vs the best model. Run AFTER all baselines finished.
from google.colab import drive
drive.mount('/content/drive')
import os, numpy as np, pandas as pd
from sklearn.metrics import accuracy_score, f1_score
P='/content/drive/MyDrive/Colab Notebooks/'
yte=None; preds={}   # preds[model] = list of per-seed prediction arrays
def add(model,arr):
    preds.setdefault(model,[]).append(np.asarray(arr))
for f in ('clean_bugzilla_classic_preds.npz','clean_bugzilla_dl_preds.npz'):
    if os.path.exists(P+f):
        z=np.load(P+f); yte=z['y_test']
        for k in z.files:
            if k!='y_test': add(k.rsplit('_s',1)[0],z[k])
    else: print('MISSING',f)
names={'roberta':'RoBERTa (fine-tuned)','bert':'BERT (fine-tuned)','codebert':'CodeBERT (fine-tuned)'}
for tag,nm in names.items():
    for s in (42,43,44):
        f=P+f'ft_{tag}_bz_test_logits_s{s}.npy'
        if os.path.exists(f): add(nm,np.load(f).argmax(1))
        else: print('MISSING',f)
rows=[]
for m,L in preds.items():
    a=[accuracy_score(yte,p) for p in L]; f=[f1_score(yte,p,average='macro') for p in L]
    rows.append(dict(Model=m,n_runs=len(L),Acc=np.mean(a),Acc_sd=np.std(a),MacroF1=np.mean(f),MacroF1_sd=np.std(f)))
tab=pd.DataFrame(rows).sort_values('MacroF1',ascending=False); print(tab.round(4).to_string(index=False))
tab.to_csv(P+'clean_bugzilla_final_table.csv',index=False)

best=tab.iloc[0]['Model']; print('\nPaired bootstrap vs best =',best)
rng=np.random.default_rng(0); idx=[rng.integers(0,len(yte),len(yte)) for _ in range(1000)]
pb=preds[best][0]; out=[]
for m,L in preds.items():
    if m==best: continue
    d=np.array([f1_score(yte[i],L[0][i],average='macro')-f1_score(yte[i],pb[i],average='macro') for i in idx])
    out.append(dict(Model=m,diff_vs_best=d.mean(),CI_lo=np.percentile(d,2.5),CI_hi=np.percentile(d,97.5),significant=(np.percentile(d,97.5)<0 or np.percentile(d,2.5)>0)))
sig=pd.DataFrame(out).sort_values('diff_vs_best',ascending=False); print(sig.round(4).to_string(index=False))
sig.to_csv(P+'clean_bugzilla_bootstrap_vs_best.csv',index=False)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
                Model  n_runs    Acc  Acc_sd  MacroF1  MacroF1_sd
              XGBoost       3 0.7452  0.0000   0.5476      0.0000
               LogReg       3 0.6985  0.0000   0.5358      0.0000
                  SVM       3 0.7543  0.0000   0.5330      0.0000
CodeBERT (fine-tuned)       3 0.6582  0.0187   0.5200      0.0060
 RoBERTa (fine-tuned)       3 0.6789  0.0108   0.5195      0.0072
    BERT (fine-tuned)       3 0.6324  0.0087   0.5011      0.0028
           NaiveBayes       3 0.7571  0.0000   0.4859      0.0000
           CNN+BiLSTM       3 0.6856  0.0356   0.4843      0.0137
               BiLSTM       3 0.6312  0.0177   0.4587      0.0167
                 LSTM       3 0.6678  0.0496   0.4585      0.0104
         RandomForest       3 0.7872  0.0009   0.4332      0.0027

Paired bootstrap vs best = XGBoost
                Model  diff_vs_best   CI_lo

In [ ]:
# Step 3: GitHub Issues -> resolution-time classes (Fast/Medium/Slow). NOT severity: an effort/urgency proxy.
# Labels come from Created/Resolved timestamps only; features are text only (no time columns, no keyword labels).
from google.colab import drive
drive.mount('/content/drive')
import numpy as np, pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import accuracy_score, f1_score, classification_report
from xgboost import XGBClassifier
P='/content/drive/MyDrive/Colab Notebooks/'
df=pd.read_csv(P+'GitHub_bugs.csv'); print('raw',df.shape); print(df['Resolution'].value_counts())
df=df[df['Resolution']=='completed'].copy()                      # only truly fixed issues
df['days']=(pd.to_datetime(df['Resolved'],errors='coerce',utc=True)-pd.to_datetime(df['Created'],errors='coerce',utc=True)).dt.total_seconds()/86400
df=df[df['days']>=0].dropna(subset=['days'])
df['text']=(df['Summary'].fillna('')+' '+df['Description'].fillna('')).str.lower().str.replace(r'\s+',' ',regex=True).str.strip()
df=df[df['text'].str.len()>10]; n0=len(df); df=df.drop_duplicates('text').reset_index(drop=True); print('after filters/dedupe',n0,'->',len(df))
print(df['days'].describe())
# tertile thresholds on the TRAIN split only (no leakage), computed after splitting
tr,tmp=train_test_split(df,test_size=0.30,random_state=42); va,te=train_test_split(tmp,test_size=0.50,random_state=42)
q1,q2=tr['days'].quantile([1/3,2/3]); print('thresholds (days): Fast <=%.2f < Medium <=%.2f < Slow'%(q1,q2))
lab=lambda d: np.where(d['days']<=q1,'Fast',np.where(d['days']<=q2,'Medium','Slow'))
for d in (tr,va,te): d['y']=lab(d)
print(tr['y'].value_counts(), te['y'].value_counts())
assert not set(tr.text)&set(te.text) and not set(tr.text)&set(va.text)
for n,d in (('train',tr),('val',va),('test',te)): d.drop(columns=['Resolved','Created'],errors='ignore').to_csv(P+f'clean_github_{n}.csv',index=False)
cid={'Fast':0,'Medium':1,'Slow':2}; ytr,yva,yte=[d['y'].map(cid).values for d in (tr,va,te)]
tf=TfidfVectorizer(max_features=20000,ngram_range=(1,2),min_df=2,sublinear_tf=True).fit(tr.text)
Xtr,Xva,Xte=[tf.transform(d.text) for d in (tr,va,te)]
p=LogisticRegression(max_iter=1000,class_weight='balanced').fit(Xtr,ytr).predict(Xte)
print('LogReg  test acc %.4f macroF1 %.4f'%(accuracy_score(yte,p),f1_score(yte,p,average='macro')))
x=XGBClassifier(n_estimators=300,max_depth=6,learning_rate=0.1,tree_method='hist',random_state=42,n_jobs=-1).fit(Xtr,ytr,sample_weight=compute_sample_weight('balanced',ytr))
p=x.predict(Xte); print('XGBoost test acc %.4f macroF1 %.4f'%(accuracy_score(yte,p),f1_score(yte,p,average='macro')))
print(classification_report(yte,p,target_names=['Fast','Medium','Slow']))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
raw (32829, 6)
Resolution
completed      29387
not_planned      659
reopened         191
Name: count, dtype: int64
after filters/dedupe 29385 -> 29384
count    29384.000000
mean        70.412951
std        176.146210
min          0.000023
25%          0.788006
50%          5.407708
75%         39.190315
max       2358.771748
Name: days, dtype: float64
thresholds (days): Fast <=1.34 < Medium <=20.02 < Slow
y
Slow      6856
Medium    6856
Fast      6856
Name: count, dtype: int64 y
Fast      1499
Medium    1456
Slow      1453
Name: count, dtype: int64
LogReg  test acc 0.4476 macroF1 0.4433
XGBoost test acc 0.4558 macroF1 0.4435
              precision    recall  f1-score   support

        Fast       0.49      0.62      0.54      1499
      Medium       0.37      0.26      0.31      1456
        Slow       0.48      0.48      0.48      1453

    accuracy        

In [ ]:
# GitHub (resolution-time classes): distress features on TF-IDF + XGBoost / LogReg
from google.colab import drive
drive.mount('/content/drive')
import os, numpy as np, pandas as pd, torch, scipy.sparse as sp
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import accuracy_score, f1_score
from xgboost import XGBClassifier

P='/content/drive/MyDrive/Colab Notebooks/'; dev='cuda' if torch.cuda.is_available() else 'cpu'
tr,va,te=[pd.read_csv(P+f'clean_github_{n}.csv') for n in ('train','val','test')]
for d in (tr,va,te): d['text']=d['text'].fillna('').astype(str)
cid={'Fast':0,'Medium':1,'Slow':2}; ytr,yva,yte=[d['y'].map(cid).values for d in (tr,va,te)]

name='j-hartmann/emotion-english-distilroberta-base'
tok=AutoTokenizer.from_pretrained(name); em=AutoModelForSequenceClassification.from_pretrained(name).to(dev).eval()
if dev=='cuda': em=em.half()
@torch.no_grad()
def emo(texts,bs=128):
    out=[]
    for i in range(0,len(texts),bs):
        b=tok(list(texts[i:i+bs]),padding=True,truncation=True,max_length=256,return_tensors='pt').to(dev)
        out.append(torch.softmax(em(**b).logits.float(),-1).cpu().numpy())
    return np.vstack(out)
def cached(tag,d):
    f=P+f'emo_probs_gh_{tag}.npy'
    if not os.path.exists(f): np.save(f,emo(d.text.values))
    return np.load(f)
E={n:cached(n,d) for n,d in (('train',tr),('val',va),('test',te))}

LEX={'frustration':['frustrating','annoying','broken','failed','crash','error'],'confusion':['confused','unclear','dont understand','unknown','complicated'],
 'anxiety':['worried','anxious','fear','unsafe','security'],'urgency':['urgent','immediately','asap','critical','important'],
 'helplessness':['stuck','unable','cannot','helpless','nothing works'],'anger':['hate','terrible','worst','angry','useless']}
def dds(d): return np.array([[sum(t.lower().count(w) for w in ws) for ws in LEX.values()] for t in d.text])
K={n:dds(d) for n,d in (('train',tr),('val',va),('test',te))}
D={n:np.c_[E[n],K[n],K[n].sum(1)] for n in E}
sc=StandardScaler().fit(D['train']); D={n:sc.transform(D[n]) for n in D}

tf=TfidfVectorizer(max_features=20000,ngram_range=(1,2),min_df=2,sublinear_tf=True).fit(tr.text)
T={n:tf.transform(d.text) for n,d in (('train',tr),('val',va),('test',te))}
def feats(n,use): return sp.hstack([T[n],sp.csr_matrix(D[n])]).tocsr() if use else T[n]

sw=compute_sample_weight('balanced',ytr); rows=[]; preds={}
for use in (False,True):
    tag='+distress' if use else 'base'
    Xtr,Xte=feats('train',use),feats('test',use)
    lr=LogisticRegression(max_iter=2000,class_weight='balanced').fit(Xtr,ytr); p=lr.predict(Xte)
    preds[f'LogReg_{tag}']=p; rows.append(dict(Model='LogReg',Feat=tag,seed=0,Acc=accuracy_score(yte,p),MacroF1=f1_score(yte,p,average='macro')))
    for s in (42,43,44,45,46):
        x=XGBClassifier(n_estimators=300,max_depth=6,learning_rate=0.1,subsample=0.8,colsample_bytree=0.8,tree_method='hist',random_state=s,n_jobs=-1).fit(Xtr,ytr,sample_weight=sw)
        p=x.predict(Xte); preds[f'XGBoost_{tag}_s{s}']=p
        rows.append(dict(Model='XGBoost',Feat=tag,seed=s,Acc=accuracy_score(yte,p),MacroF1=f1_score(yte,p,average='macro')))
res=pd.DataFrame(rows); print(res.groupby(['Model','Feat'])[['Acc','MacroF1']].agg(['mean','std']).round(4))
res.to_csv(P+'clean_github_distress_on_xgb.csv',index=False); np.savez(P+'clean_github_distress_on_xgb_preds.npz',y_test=yte,**preds)

# paired bootstrap: XGBoost +distress vs base (seed 42)
rng=np.random.default_rng(0); a,b=preds['XGBoost_+distress_s42'],preds['XGBoost_base_s42']; diffs=[]
for _ in range(2000):
    i=rng.integers(0,len(yte),len(yte)); diffs.append(f1_score(yte[i],a[i],average='macro')-f1_score(yte[i],b[i],average='macro'))
print('XGB +distress minus base, MacroF1: mean %.4f  95%% CI [%.4f, %.4f]  P(diff<=0)=%.3f'%(np.mean(diffs),*np.percentile(diffs,[2.5,97.5]),np.mean(np.array(diffs)<=0)))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

                      Acc         MacroF1        
                     mean     std    mean     std
Model   Feat                                     
LogReg  +distress  0.4440     NaN  0.4397     NaN
        base       0.4476     NaN  0.4433     NaN
XGBoost +distress  0.4590  0.0065  0.4476  0.0068
        base       0.4599  0.0035  0.4493  0.0039
XGB +distress minus base, MacroF1: mean -0.0026  95% CI [-0.0118, 0.0069]  P(diff<=0)=0.705


In [ ]:
# Step A: fine-tune RoBERTa end-to-end on clean Bugzilla split (real Severity, class-weighted loss)
# Colab T4. Model chosen on VAL macro-F1; test scored once per seed. Saves logits for later stacking.
from google.colab import drive
drive.mount('/content/drive')
import numpy as np, pandas as pd, torch, torch.nn as nn, time
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from sklearn.metrics import accuracy_score, f1_score, classification_report
from sklearn.utils.class_weight import compute_class_weight

# Set these two, run once for BERT and once for CodeBERT:
MODEL='roberta-base'
TAG='roberta'
P='/content/drive/MyDrive/Colab Notebooks/'
dev='cuda'
tr,va,te=[pd.read_csv(P+f'clean_github_{n}.csv') for n in ('train','val','test')]
for d in (tr,va,te): d['text']=d['text'].fillna('').astype(str)
classes=['Fast','Medium','Slow']; cid={c:i for i,c in enumerate(classes)}
Y={n:d['y'].map(cid).values for n,d in (('train',tr),('val',va),('test',te))}
tok=AutoTokenizer.from_pretrained(MODEL); MAXLEN=192; BS=32

def enc(d):
    e=tok(list(d.text),truncation=True,max_length=MAXLEN,padding='max_length',return_tensors='pt')
    return e['input_ids'],e['attention_mask']
E={n:enc(d) for n,d in (('train',tr),('val',va),('test',te))}

@torch.no_grad()
def logits_of(model,n):
    model.eval(); ids,am=E[n]; out=[]
    for i in range(0,len(ids),128):
        with torch.autocast('cuda',dtype=torch.float16):
            out.append(model(input_ids=ids[i:i+128].to(dev),attention_mask=am[i:i+128].to(dev)).logits.float().cpu())
    return torch.cat(out).numpy()

def run(seed,epochs=4,lr=2e-5):
    torch.manual_seed(seed); np.random.seed(seed)
    model=AutoModelForSequenceClassification.from_pretrained(MODEL,num_labels=3).to(dev)
    w=torch.tensor(compute_class_weight('balanced',classes=np.arange(3),y=Y['train']),dtype=torch.float32,device=dev)
    lossf=nn.CrossEntropyLoss(weight=w)
    opt=torch.optim.AdamW(model.parameters(),lr=lr,weight_decay=0.01)
    ids,am=E['train']; ytr=torch.tensor(Y['train'])
    steps=epochs*((len(ids)+BS-1)//BS); sch=get_linear_schedule_with_warmup(opt,int(0.06*steps),steps)
    scaler=torch.cuda.amp.GradScaler(); best,bstate=-1,None
    for ep in range(epochs):
        model.train(); perm=torch.randperm(len(ids)); t0=time.time()
        for i in range(0,len(perm),BS):
            ix=perm[i:i+BS]
            with torch.autocast('cuda',dtype=torch.float16):
                lg=model(input_ids=ids[ix].to(dev),attention_mask=am[ix].to(dev)).logits
            loss=lossf(lg.float(),ytr[ix].to(dev))
            opt.zero_grad(); scaler.scale(loss).backward(); scaler.unscale_(opt)
            nn.utils.clip_grad_norm_(model.parameters(),1.0); scaler.step(opt); scaler.update(); sch.step()
        lv=logits_of(model,'val'); f=f1_score(Y['val'],lv.argmax(1),average='macro')
        print(f'seed {seed} ep {ep+1} valMacroF1 {f:.4f} valAcc {accuracy_score(Y["val"],lv.argmax(1)):.4f} ({time.time()-t0:.0f}s)')
        if f>best: best=f; bstate={k:v.detach().clone() for k,v in model.state_dict().items()}
    model.load_state_dict(bstate)
    lv,lt=logits_of(model,'val'),logits_of(model,'test')
    np.save(P+f'ft_{TAG}_gh_val_logits_s{seed}.npy',lv); np.save(P+f'ft_{TAG}_gh_test_logits_s{seed}.npy',lt)
    pt=lt.argmax(1)
    r=dict(seed=seed,valF1=best,acc=accuracy_score(Y['test'],pt),macroF1=f1_score(Y['test'],pt,average='macro'))
    print(r); print(classification_report(Y['test'],pt,target_names=classes))
    del model,opt; torch.cuda.empty_cache(); return r

rows=[run(s) for s in (42,43,44)]
res=pd.DataFrame(rows); print(res.round(4)); print('mean',res[['acc','macroF1']].mean().round(4).to_dict(),'sd',res[['acc','macroF1']].std().round(4).to_dict())
res.to_csv(P+f'clean_github_finetune_{TAG}_results.csv',index=False)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

... (7 outputs left out)

seed 43 ep 1 valMacroF1 0.3846 valAcc 0.4344 (22s)
seed 43 ep 2 valMacroF1 0.4208 valAcc 0.4632 (22s)
seed 43 ep 3 valMacroF1 0.4403 valAcc 0.4691 (22s)
seed 43 ep 4 valMacroF1 0.4535 valAcc 0.4657 (22s)
{'seed': 43, 'valF1': 0.45349717326895506, 'acc': 0.4562159709618875, 'macroF1': 0.44113928466766206}
              precision    recall  f1-score   support

        Fast       0.48      0.65      0.55      1499
      Medium       0.38      0.25      0.30      1456
        Slow       0.48      0.46      0.47      1453

    accuracy                           0.46      4408
   macro avg       0.44      0.45      0.44      4408
weighted avg       0.45      0.46      0.44      4408



Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.bias               | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
classifier.out_proj.weight | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.bias      | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
/tmp/ipykernel_3286/1736831414.py:42: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler=torch.cuda.amp.GradScaler(

seed 44 ep 1 valMacroF1 0.3589 valAcc 0.4480 (22s)
seed 44 ep 2 valMacroF1 0.4456 valAcc 0.4632 (22s)
seed 44 ep 3 valMacroF1 0.4436 valAcc 0.4707 (22s)
seed 44 ep 4 valMacroF1 0.4525 valAcc 0.4644 (22s)
{'seed': 44, 'valF1': 0.45254505531885303, 'acc': 0.4605263157894737, 'macroF1': 0.4483360729845253}
              precision    recall  f1-score   support

        Fast       0.50      0.60      0.54      1499
      Medium       0.37      0.26      0.30      1456
        Slow       0.48      0.51      0.50      1453

    accuracy                           0.46      4408
   macro avg       0.45      0.46      0.45      4408
weighted avg       0.45      0.46      0.45      4408

   seed   valF1     acc  macroF1
0    42  0.4498  0.4608   0.4439
1    43  0.4535  0.4562   0.4411
2    44  0.4525  0.4605   0.4483
mean {'acc': 0.4592, 'macroF1': 0.4445} sd {'acc': 0.0026, 'macroF1': 0.0036}


In [ ]:
# Step 4: Emotion/distress analysis on all three platforms with ONE independent emotion model.
#  (a) Stack Overflow: emotion by API tag (chi-square, Cramer's V)  -- no severity label exists for SO
#  (b) Cross-platform emotion distributions (chi-square)
#  (c) Is distress associated with the label? Bugzilla (real severity) and GitHub (resolution-time class): Kruskal-Wallis
from google.colab import drive
drive.mount('/content/drive')
import os, re, numpy as np, pandas as pd, torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from scipy.stats import chi2_contingency, kruskal

P='/content/drive/MyDrive/Colab Notebooks/'; dev='cuda' if torch.cuda.is_available() else 'cpu'
EMO=['anger','disgust','fear','joy','neutral','sadness','surprise']   # j-hartmann label order
name='j-hartmann/emotion-english-distilroberta-base'
tok=AutoTokenizer.from_pretrained(name); em=AutoModelForSequenceClassification.from_pretrained(name).to(dev).eval()
if dev=='cuda': em=em.half()
assert [em.config.id2label[i] for i in range(7)]==EMO
@torch.no_grad()
def emo(texts,bs=128):
    out=[]
    for i in range(0,len(texts),bs):
        b=tok(list(texts[i:i+bs]),padding=True,truncation=True,max_length=256,return_tensors='pt').to(dev)
        out.append(torch.softmax(em(**b).logits.float(),-1).cpu().numpy())
    return np.vstack(out)

# ---------- Stack Overflow ----------
def clean(t):
    t=str(t).lower(); t=re.sub(r'http\S+','',t); t=re.sub(r'<.*?>','',t); t=re.sub(r'[^a-z\s]',' ',t); return re.sub(r'\s+',' ',t).strip()
so=pd.read_csv(P+'api_related_dataset.csv'); so['text']=(so['Title'].astype(str)+' '+so['Body'].astype(str)).map(clean)
n0=len(so); so=so[so.text.str.len()>10].drop_duplicates('text').reset_index(drop=True); print('SO rows',n0,'-> unique',len(so))
f=P+'emo_probs_so.npy'
if not os.path.exists(f) or len(np.load(f))!=len(so): np.save(f,emo(so.text.values))
so_p=np.load(f); so['emo']=[EMO[i] for i in so_p.argmax(1)]
print('\nSO dominant emotion (%):'); print((so.emo.value_counts(normalize=True)*100).round(1))
tags=so.CollectedTag.value_counts(); keep=tags[tags>=100].index; sub=so[so.CollectedTag.isin(keep)]
ct=pd.crosstab(sub.CollectedTag,sub.emo); chi2,p,dof,_=chi2_contingency(ct); V=np.sqrt(chi2/(ct.values.sum()*(min(ct.shape)-1)))
print('\nSO emotion x API tag: chi2=%.1f dof=%d p=%.3g Cramer V=%.3f'%(chi2,dof,p,V))
print((pd.crosstab(sub.CollectedTag,sub.emo,normalize='index')*100).round(1))
so.to_csv(P+'so_emotion_analysis.csv',index=False)

# ---------- Cross-platform ----------
def load_probs(tag,files):
    arrs=[np.load(P+f'emo_probs_{tag}_{n}.npy') for n in files]; return np.vstack(arrs)
bz_p=load_probs('bz',('train','val','test')); gh_p=load_probs('gh',('train','val','test'))
dist={'StackOverflow':np.bincount(so_p.argmax(1),minlength=7),'GitHub':np.bincount(gh_p.argmax(1),minlength=7),'Bugzilla':np.bincount(bz_p.argmax(1),minlength=7)}
tab=pd.DataFrame(dist,index=EMO).T; print('\nDominant emotion % by platform:'); print((tab.div(tab.sum(1),axis=0)*100).round(1))
chi2,p,dof,_=chi2_contingency(tab.values); print('cross-platform chi2=%.1f dof=%d p=%.3g Cramer V=%.3f'%(chi2,dof,p,np.sqrt(chi2/(tab.values.sum()*2))))
tab.to_csv(P+'emotion_by_platform.csv')

# ---------- Distress vs label ----------
def assoc(tag,files,labelfile_fmt,order,title):
    probs=load_probs(tag,files); y=np.concatenate([pd.read_csv(P+labelfile_fmt.format(n))['y'].values for n in files])
    assert len(y)==len(probs)
    print(f'\n{title}: mean emotion prob by class  (Kruskal-Wallis p, epsilon^2)')
    rows=[]
    for j,e in enumerate(EMO):
        groups=[probs[y==c,j] for c in order]; H,p=kruskal(*groups); eps=H/(len(y)-1)
        rows.append(dict(emotion=e,**{c:groups[i].mean() for i,c in enumerate(order)},p=p,eps2=eps))
    r=pd.DataFrame(rows); print(r.round(4).to_string(index=False)); r.to_csv(P+f'distress_vs_label_{tag}.csv',index=False)
assoc('bz',('train','val','test'),'clean_bugzilla_{}.csv',['Critical','High','Low'],'Bugzilla (real severity)')
assoc('gh',('train','val','test'),'clean_github_{}.csv',['Fast','Medium','Slow'],'GitHub (resolution-time class)')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

SO rows 13927 -> unique 13927

SO dominant emotion (%):
emo
neutral     60.7
sadness     23.9
surprise    11.9
fear         1.6
anger        1.0
joy          0.9
disgust      0.0
Name: proportion, dtype: float64

SO emotion x API tag: chi2=213.8 dof=36 p=3.82e-27 Cramer V=0.051
emo                 anger  disgust  fear  joy  neutral  sadness  surprise
CollectedTag                                                             
android-api           1.0      0.0   1.4  1.0     55.7     29.8      11.1
facebook-graph-api    0.8      0.1   1.1  1.0     55.9     27.2      13.9
github-api            0.7      0.0   1.1  0.6     67.8     17.8      12.1
google-api            1.2      0.0   1.8  0.8     60.7     23.3      12.2
java-api              1.0      0.0   2.1  1.2     63.2     22.1      10.4
rest-api              1.3      0.2   2.8  1.0     68.7     16.8       9.2
twitter-api           1.2      0.0   1.7  1.0     57.1     25.7      13.3

Dominant emotion % by platform:
               anger  

In [ ]:
# Step 5: Explainability + error analysis for the Bugzilla TF-IDF+XGBoost(+distress) model (clean split).
# Uses XGBoost's native SHAP (pred_contribs) -> fast and exact for trees. LIME on a few misclassified examples.
from google.colab import drive
drive.mount('/content/drive')
!pip -q install lime
import numpy as np, pandas as pd, scipy.sparse as sp, xgboost as xgb
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import confusion_matrix, f1_score
P='/content/drive/MyDrive/Colab Notebooks/'
tr,va,te=[pd.read_csv(P+f'clean_bugzilla_{n}.csv') for n in ('train','val','test')]
for d in (tr,va,te): d['text']=d['text'].fillna('').astype(str)
classes=['Critical','High','Low']; cid={c:i for i,c in enumerate(classes)}
ytr,yte=tr['y'].map(cid).values,te['y'].map(cid).values
E={n:np.load(P+f'emo_probs_bz_{n}.npy') for n in ('train','val','test')}
LEX={'frustration':['frustrating','annoying','broken','failed','crash','error'],'confusion':['confused','unclear','dont understand','unknown','complicated'],
 'anxiety':['worried','anxious','fear','unsafe','security'],'urgency':['urgent','immediately','asap','critical','important'],
 'helplessness':['stuck','unable','cannot','helpless','nothing works'],'anger':['hate','terrible','worst','angry','useless']}
K={n:np.array([[sum(t.lower().count(w) for w in ws) for ws in LEX.values()] for t in d.text]) for n,d in (('train',tr),('val',va),('test',te))}
D={n:np.c_[E[n],K[n],K[n].sum(1)] for n in E}; sc=StandardScaler().fit(D['train']); D={n:sc.transform(D[n]) for n in D}
dnames=['emo_'+e for e in ['anger','disgust','fear','joy','neutral','sadness','surprise']]+['kw_'+k for k in LEX]+['kw_total']
tf=TfidfVectorizer(max_features=20000,ngram_range=(1,2),min_df=2,sublinear_tf=True).fit(tr.text)
names=list(tf.get_feature_names_out())+dnames
X={n:sp.hstack([tf.transform(d.text),sp.csr_matrix(D[n])]).tocsr() for n,d in (('train',tr),('test',te))}
m=xgb.XGBClassifier(n_estimators=300,max_depth=6,learning_rate=0.1,subsample=0.8,colsample_bytree=0.8,tree_method='hist',device='cuda',random_state=42)
m.fit(X['train'],ytr,sample_weight=compute_sample_weight('balanced',ytr)); pred=m.predict(X['test'])
print('test MacroF1 %.4f'%f1_score(yte,pred,average='macro'))
print('confusion matrix (rows=true, cols=pred):\n',pd.DataFrame(confusion_matrix(yte,pred),index=classes,columns=classes))

# ---- SHAP (native) on a stratified test sample ----
rng=np.random.default_rng(0); idx=rng.choice(len(yte),2000,replace=False)
contrib=m.get_booster().predict(xgb.DMatrix(X['test'][idx]),pred_contribs=True)   # (n, classes, feats+1)
contrib=contrib[:,:,:-1]; imp=np.abs(contrib).mean(0)                                # (classes, feats)
tot=imp.sum(1); dist_share=imp[:,-len(dnames):].sum(1)/tot
print('\nShare of total |SHAP| carried by the 15 distress features, per class:',dict(zip(classes,dist_share.round(4))))
rows=[]
for c,cn in enumerate(classes):
    top=np.argsort(-imp[c])[:15]
    print(f'\nTop 15 features for {cn}:'); print([ (names[i],round(float(imp[c,i]),4)) for i in top])
    for i in top: rows.append(dict(cls=cn,feature=names[i],mean_abs_shap=imp[c,i],is_distress=i>=len(names)-len(dnames)))
dimp=pd.DataFrame({'feature':dnames,**{cn:imp[c,-len(dnames):] for c,cn in enumerate(classes)}}).sort_values('Critical',ascending=False)
print('\nDistress feature importance (mean |SHAP|):'); print(dimp.round(4).to_string(index=False))
pd.DataFrame(rows).to_csv(P+'bugzilla_shap_top_features.csv',index=False); dimp.to_csv(P+'bugzilla_shap_distress.csv',index=False)

# ---- Error analysis: 12 misclassified test examples (4 per error type) ----
te['pred']=[classes[i] for i in pred]; err=te[te.y!=te.pred]
pick=pd.concat([err[(err.y==a)&(err.pred==b)].sample(min(4,((err.y==a)&(err.pred==b)).sum()),random_state=1) for a,b in (('Critical','Low'),('Low','Critical'),('High','Low'))])
pick['excerpt']=pick.text.str.slice(0,220); print(pick[['y','pred','excerpt']].to_string(index=False)); pick[['y','pred','excerpt']].to_csv(P+'bugzilla_error_examples.csv',index=False)

# ---- LIME on 3 misclassified examples (text-only TF-IDF+XGBoost) ----
from lime.lime_text import LimeTextExplainer
def predict_proba(texts):
    t=tf.transform(texts); z=np.zeros((t.shape[0],len(dnames)))   # distress cols neutral (mean) for LIME text perturbation
    return m.predict_proba(sp.hstack([t,sp.csr_matrix(z)]).tocsr())
ex=LimeTextExplainer(class_names=classes,random_state=0)
for _,r in pick.head(3).iterrows():
    e=ex.explain_instance(r.text,predict_proba,num_features=8,num_samples=500,labels=[cid[r.pred]])
    print(f'\nLIME  true={r.y} pred={r.pred}:',e.as_list(label=cid[r.pred]))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


/usr/local/lib/python3.13/dist-packages/xgboost/core.py:569: UserWarning: [05:13:35] WARNING: /__w/xgboost/xgboost/src/common/error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


test MacroF1 0.5590
confusion matrix (rows=true, cols=pred):
           Critical  High   Low
Critical       355    70   146
High            98   148   318
Low            271   310  3147

Share of total |SHAP| carried by the 15 distress features, per class: {'Critical': np.float32(0.087), 'High': np.float32(0.0222), 'Low': np.float32(0.0995)}

Top 15 features for Critical:
[('crash', 0.1956), ('blocker', 0.1076), ('kw_frustration', 0.0843), ('emo_fear', 0.0761), ('crashes', 0.0734), ('critical', 0.056), ('should', 0.053), ('kw_total', 0.0473), ('the', 0.0447), ('emo_sadness', 0.044), ('stack', 0.0428), ('first', 0.0331), ('page', 0.0322), ('behavior', 0.0288), ('what', 0.0265)]

Top 15 features for High:
[('major', 0.0552), ('select', 0.0349), ('crash', 0.0317), ('this bug', 0.0299), ('html', 0.0274), ('page', 0.0232), ('2000', 0.0215), ('apprunner', 0.0186), ('bug has', 0.017), ('blocker', 0.0161), ('of reproduce', 0.0151), ('bug', 0.0136), ('and', 0.0121), ('now', 0.0117), ('results',

In [ ]:
# Sensitivity check: remove severity-name tokens (blocker/critical/major/...) from text, then rerun TF-IDF + XGBoost ± distress.
from google.colab import drive
drive.mount('/content/drive')
import re, numpy as np, pandas as pd, scipy.sparse as sp
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import accuracy_score, f1_score
from xgboost import XGBClassifier
P='/content/drive/MyDrive/Colab Notebooks/'
tr,va,te=[pd.read_csv(P+f'clean_bugzilla_{n}.csv') for n in ('train','val','test')]
for d in (tr,va,te): d['text']=d['text'].fillna('').astype(str)
cid={'Critical':0,'High':1,'Low':2}; ytr,yte=tr['y'].map(cid).values,te['y'].map(cid).values
SEV=r'\b(blockers?|critical|criticals|major|majors|minor|minors|trivial|normal|severity|sev|enhancement|regression)\b'
mask=lambda s: re.sub(r'\s+',' ',re.sub(SEV,' ',s))
frac=np.mean([bool(re.search(SEV,t)) for t in tr.text]); print('train texts containing a severity-name token: %.1f%%'%(100*frac))
for lab in ('Critical','High','Low'):
    print(lab,'contains token: %.1f%%'%(100*np.mean([bool(re.search(SEV,t)) for t in tr.text[tr.y==lab]])))
E={n:np.load(P+f'emo_probs_bz_{n}.npy') for n in ('train','val','test')}   # emotion probs from ORIGINAL text (fixed)
LEX={'frustration':['frustrating','annoying','broken','failed','crash','error'],'confusion':['confused','unclear','dont understand','unknown','complicated'],
 'anxiety':['worried','anxious','fear','unsafe','security'],'urgency':['urgent','immediately','asap','critical','important'],
 'helplessness':['stuck','unable','cannot','helpless','nothing works'],'anger':['hate','terrible','worst','angry','useless']}
def dds(texts): return np.array([[sum(t.lower().count(w) for w in ws) for ws in LEX.values()] for t in texts])
def run(masked):
    T={n:(d.text.map(mask) if masked else d.text) for n,d in (('train',tr),('test',te))}
    K={n:dds(T[n]) for n in T}                                             # keyword counts recomputed on (masked) text
    D={n:np.c_[E[n],K[n],K[n].sum(1)] for n in T}; sc=StandardScaler().fit(D['train']); D={n:sc.transform(D[n]) for n in D}
    tf=TfidfVectorizer(max_features=20000,ngram_range=(1,2),min_df=2,sublinear_tf=True).fit(T['train'])
    out={}
    for use in (False,True):
        X={n:(sp.hstack([tf.transform(T[n]),sp.csr_matrix(D[n])]).tocsr() if use else tf.transform(T[n])) for n in T}
        f=[]
        for s in (42,43,44):
            m=XGBClassifier(n_estimators=300,max_depth=6,learning_rate=0.1,subsample=0.8,colsample_bytree=0.8,tree_method='hist',device='cuda',random_state=s)
            m.fit(X['train'],ytr,sample_weight=compute_sample_weight('balanced',ytr)); p=m.predict(X['test']); f.append(f1_score(yte,p,average='macro'))
        out['+distress' if use else 'base']=(np.mean(f),np.std(f))
    return out
for masked in (False,True):
    r=run(masked); print('MASKED' if masked else 'ORIGINAL',{k:'%.4f ± %.4f'%v for k,v in r.items()})

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
train texts containing a severity-name token: 18.9%
Critical contains token: 28.0%
High contains token: 21.9%
Low contains token: 17.0%
ORIGINAL {'base': '0.5504 ± 0.0029', '+distress': '0.5580 ± 0.0013'}
MASKED {'base': '0.5326 ± 0.0019', '+distress': '0.5444 ± 0.0056'}


In [ ]:
# Baselines on the SAME clean Bugzilla split: LR, SVM, RF, NB, XGBoost, LSTM, BiLSTM, CNN+BiLSTM
# Class-balanced where supported; model selection on VAL only; test scored once. Saves test predictions.
from google.colab import drive
drive.mount('/content/drive')
import numpy as np, pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import ComplementNB
from sklearn.utils.class_weight import compute_class_weight, compute_sample_weight
from sklearn.metrics import accuracy_score, f1_score
from xgboost import XGBClassifier

P='/content/drive/MyDrive/Colab Notebooks/'
tr,va,te=[pd.read_csv(P+f'clean_github_{n}.csv') for n in ('train','val','test')]
for d in (tr,va,te): d['text']=d['text'].fillna('').astype(str)
cid={'Fast':0,'Medium':1,'Slow':2}
ytr,yva,yte=[d['y'].map(cid).values for d in (tr,va,te)]
tf_=TfidfVectorizer(max_features=20000,ngram_range=(1,2),min_df=2,sublinear_tf=True).fit(tr.text)
Xtr,Xva,Xte=[tf_.transform(d.text) for d in (tr,va,te)]
rows=[]; preds={}
def rec(name,seed,p):
    preds[f'{name}_s{seed}']=p
    rows.append(dict(Model=name,seed=seed,Acc=accuracy_score(yte,p),MacroF1=f1_score(yte,p,average='macro')))
    print(rows[-1])

for s in (42,43,44):
    rec('LogReg',s,LogisticRegression(max_iter=1000,class_weight='balanced',random_state=s).fit(Xtr,ytr).predict(Xte))
    rec('SVM',s,LinearSVC(class_weight='balanced',random_state=s).fit(Xtr,ytr).predict(Xte))
    rec('RandomForest',s,RandomForestClassifier(300,class_weight='balanced_subsample',n_jobs=-1,random_state=s).fit(Xtr,ytr).predict(Xte))
    rec('NaiveBayes',s,ComplementNB().fit(Xtr,ytr).predict(Xte))
    sw=compute_sample_weight('balanced',ytr)
    rec('XGBoost',s,XGBClassifier(n_estimators=300,max_depth=6,learning_rate=0.1,tree_method='hist',random_state=s,n_jobs=-1).fit(Xtr,ytr,sample_weight=sw).predict(Xte))


res=pd.DataFrame(rows); res.to_csv(P+'clean_github_classic_seedwise.csv',index=False)
np.savez(P+'clean_github_classic_preds.npz',y_test=yte,**preds)
print(res.groupby('Model')[['Acc','MacroF1']].agg(['mean','std']).round(4))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
{'Model': 'LogReg', 'seed': 42, 'Acc': 0.44759528130671505, 'MacroF1': 0.4433082107568138}
{'Model': 'SVM', 'seed': 42, 'Acc': 0.41084392014519056, 'MacroF1': 0.4095873353179753}
{'Model': 'RandomForest', 'seed': 42, 'Acc': 0.448502722323049, 'MacroF1': 0.4211559327832906}
{'Model': 'NaiveBayes', 'seed': 42, 'Acc': 0.4471415607985481, 'MacroF1': 0.39362945844135266}
{'Model': 'XGBoost', 'seed': 42, 'Acc': 0.4557622504537205, 'MacroF1': 0.4434582352207022}
{'Model': 'LogReg', 'seed': 43, 'Acc': 0.44759528130671505, 'MacroF1': 0.4433082107568138}
{'Model': 'SVM', 'seed': 43, 'Acc': 0.41084392014519056, 'MacroF1': 0.4095873353179753}
{'Model': 'RandomForest', 'seed': 43, 'Acc': 0.448956442831216, 'MacroF1': 0.41944415310352595}
{'Model': 'NaiveBayes', 'seed': 43, 'Acc': 0.4471415607985481, 'MacroF1': 0.39362945844135266}
{'Model': 'XGBoost', 'seed': 43, 'Acc': 0

In [ ]:
# Baselines on the SAME clean Bugzilla split: LR, SVM, RF, NB, XGBoost, LSTM, BiLSTM, CNN+BiLSTM
# Class-balanced where supported; model selection on VAL only; test scored once. Saves test predictions.
from google.colab import drive
drive.mount('/content/drive')
import numpy as np, pandas as pd, tensorflow as tf
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import ComplementNB
from sklearn.utils.class_weight import compute_class_weight, compute_sample_weight
from sklearn.metrics import accuracy_score, f1_score


P='/content/drive/MyDrive/Colab Notebooks/'
tr,va,te=[pd.read_csv(P+f'clean_github_{n}.csv') for n in ('train','val','test')]
for d in (tr,va,te): d['text']=d['text'].fillna('').astype(str)
cid={'Fast':0,'Medium':1,'Slow':2}
ytr,yva,yte=[d['y'].map(cid).values for d in (tr,va,te)]
rows=[]; preds={}
def rec(name,seed,p):
    preds[f'{name}_s{seed}']=p
    rows.append(dict(Model=name,seed=seed,Acc=accuracy_score(yte,p),MacroF1=f1_score(yte,p,average='macro')))
    print(rows[-1])
# --- Keras DL ---
MAXLEN,VOC=200,30000
tk=tf.keras.preprocessing.text.Tokenizer(num_words=VOC,oov_token='<unk>'); tk.fit_on_texts(list(tr.text))
seq=lambda d: tf.keras.preprocessing.sequence.pad_sequences(tk.texts_to_sequences(list(d.text)),maxlen=MAXLEN)
Str,Sva,Ste=seq(tr),seq(va),seq(te)
cw=dict(enumerate(compute_class_weight('balanced',classes=np.arange(3),y=ytr)))
L=tf.keras.layers
def build(kind):
    m=tf.keras.Sequential([L.Embedding(VOC,128)])
    if kind=='LSTM': m.add(L.LSTM(64))
    elif kind=='BiLSTM': m.add(L.Bidirectional(L.LSTM(64)))
    else:
        m.add(L.Conv1D(64,5,activation='relu')); m.add(L.MaxPooling1D(2)); m.add(L.Bidirectional(L.LSTM(64)))
    m.add(L.Dropout(0.4)); m.add(L.Dense(3,activation='softmax'))
    m.compile('adam','sparse_categorical_crossentropy'); return m
class ValF1(tf.keras.callbacks.Callback):
    def on_train_begin(self,logs=None): self.best=-1; self.w=None
    def on_epoch_end(self,ep,logs=None):
        f=f1_score(yva,self.model.predict(Sva,verbose=0).argmax(1),average='macro')
        if f>self.best: self.best,self.w=f,self.model.get_weights()
for kind in ('LSTM','BiLSTM','CNN+BiLSTM'):
    for s in (42,43,44):
        tf.keras.utils.set_random_seed(s); m=build(kind); cb=ValF1()
        m.fit(Str,ytr,validation_data=(Sva,yva),epochs=8,batch_size=64,class_weight=cw,callbacks=[cb],verbose=0)
        m.set_weights(cb.w); rec(kind,s,m.predict(Ste,verbose=0).argmax(1))

res=pd.DataFrame(rows); res.to_csv(P+'clean_github_dl_seedwise.csv',index=False)
np.savez(P+'clean_github_dl_preds.npz',y_test=yte,**preds)
print(res.groupby('Model')[['Acc','MacroF1']].agg(['mean','std']).round(4))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
{'Model': 'LSTM', 'seed': 42, 'Acc': 0.43738656987295826, 'MacroF1': 0.4291225783667241}
{'Model': 'LSTM', 'seed': 43, 'Acc': 0.44646098003629764, 'MacroF1': 0.41829200212997764}
{'Model': 'LSTM', 'seed': 44, 'Acc': 0.45803085299455537, 'MacroF1': 0.43512720740124006}
{'Model': 'BiLSTM', 'seed': 42, 'Acc': 0.45394736842105265, 'MacroF1': 0.43610672421339663}
{'Model': 'BiLSTM', 'seed': 43, 'Acc': 0.45167876588021777, 'MacroF1': 0.41927090857802546}
{'Model': 'BiLSTM', 'seed': 44, 'Acc': 0.43035390199637025, 'MacroF1': 0.42133529186481805}
{'Model': 'CNN+BiLSTM', 'seed': 42, 'Acc': 0.45508166969147007, 'MacroF1': 0.4272993973780435}
{'Model': 'CNN+BiLSTM', 'seed': 43, 'Acc': 0.411524500907441, 'MacroF1': 0.4067118046426616}
{'Model': 'CNN+BiLSTM', 'seed': 44, 'Acc': 0.4423774954627949, 'MacroF1': 0.4217792560267321}
               Acc         MacroF1        
 

In [ ]:
# GitHub fine-tune (real Severity, class-weighted loss)
# Colab T4. Model chosen on VAL macro-F1; test scored once per seed. Saves logits for later stacking.
from google.colab import drive
drive.mount('/content/drive')
import numpy as np, pandas as pd, torch, torch.nn as nn, time
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from sklearn.metrics import accuracy_score, f1_score, classification_report
from sklearn.utils.class_weight import compute_class_weight

# Set these two, run once for BERT and once for CodeBERT:
MODEL='bert-base-uncased'   # or 'microsoft/codebert-base'
TAG='bert'                  # or 'codebert'
P='/content/drive/MyDrive/Colab Notebooks/'
dev='cuda'
tr,va,te=[pd.read_csv(P+f'clean_github_{n}.csv') for n in ('train','val','test')]
for d in (tr,va,te): d['text']=d['text'].fillna('').astype(str)
classes=['Fast','Medium','Slow']; cid={c:i for i,c in enumerate(classes)}
Y={n:d['y'].map(cid).values for n,d in (('train',tr),('val',va),('test',te))}
tok=AutoTokenizer.from_pretrained(MODEL); MAXLEN=192; BS=32

def enc(d):
    e=tok(list(d.text),truncation=True,max_length=MAXLEN,padding='max_length',return_tensors='pt')
    return e['input_ids'],e['attention_mask']
E={n:enc(d) for n,d in (('train',tr),('val',va),('test',te))}

@torch.no_grad()
def logits_of(model,n):
    model.eval(); ids,am=E[n]; out=[]
    for i in range(0,len(ids),128):
        with torch.autocast('cuda',dtype=torch.float16):
            out.append(model(input_ids=ids[i:i+128].to(dev),attention_mask=am[i:i+128].to(dev)).logits.float().cpu())
    return torch.cat(out).numpy()

def run(seed,epochs=4,lr=2e-5):
    torch.manual_seed(seed); np.random.seed(seed)
    model=AutoModelForSequenceClassification.from_pretrained(MODEL,num_labels=3).to(dev)
    w=torch.tensor(compute_class_weight('balanced',classes=np.arange(3),y=Y['train']),dtype=torch.float32,device=dev)
    lossf=nn.CrossEntropyLoss(weight=w)
    opt=torch.optim.AdamW(model.parameters(),lr=lr,weight_decay=0.01)
    ids,am=E['train']; ytr=torch.tensor(Y['train'])
    steps=epochs*((len(ids)+BS-1)//BS); sch=get_linear_schedule_with_warmup(opt,int(0.06*steps),steps)
    scaler=torch.cuda.amp.GradScaler(); best,bstate=-1,None
    for ep in range(epochs):
        model.train(); perm=torch.randperm(len(ids)); t0=time.time()
        for i in range(0,len(perm),BS):
            ix=perm[i:i+BS]
            with torch.autocast('cuda',dtype=torch.float16):
                lg=model(input_ids=ids[ix].to(dev),attention_mask=am[ix].to(dev)).logits
            loss=lossf(lg.float(),ytr[ix].to(dev))
            opt.zero_grad(); scaler.scale(loss).backward(); scaler.unscale_(opt)
            nn.utils.clip_grad_norm_(model.parameters(),1.0); scaler.step(opt); scaler.update(); sch.step()
        lv=logits_of(model,'val'); f=f1_score(Y['val'],lv.argmax(1),average='macro')
        print(f'seed {seed} ep {ep+1} valMacroF1 {f:.4f} valAcc {accuracy_score(Y["val"],lv.argmax(1)):.4f} ({time.time()-t0:.0f}s)')
        if f>best: best=f; bstate={k:v.detach().clone() for k,v in model.state_dict().items()}
    model.load_state_dict(bstate)
    lv,lt=logits_of(model,'val'),logits_of(model,'test')
    np.save(P+f'ft_{TAG}_gh_val_logits_s{seed}.npy',lv); np.save(P+f'ft_{TAG}_gh_test_logits_s{seed}.npy',lt)
    pt=lt.argmax(1)
    r=dict(seed=seed,valF1=best,acc=accuracy_score(Y['test'],pt),macroF1=f1_score(Y['test'],pt,average='macro'))
    print(r); print(classification_report(Y['test'],pt,target_names=classes))
    del model,opt; torch.cuda.empty_cache(); return r

rows=[run(s) for s in (42,43,44)]
res=pd.DataFrame(rows); print(res.round(4)); print('mean',res[['acc','macroF1']].mean().round(4).to_dict(),'sd',res[['acc','macroF1']].std().round(4).to_dict())
res.to_csv(P+f'clean_github_finetune_{TAG}_results.csv',index=False)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

... (6 outputs left out)

seed 43 ep 1 valMacroF1 0.3803 valAcc 0.4564 (22s)
seed 43 ep 2 valMacroF1 0.4256 valAcc 0.4564 (22s)
seed 43 ep 3 valMacroF1 0.4391 valAcc 0.4567 (21s)
seed 43 ep 4 valMacroF1 0.4415 valAcc 0.4476 (22s)
{'seed': 43, 'valF1': 0.44152326293805694, 'acc': 0.44101633393829404, 'macroF1': 0.4362055172282106}
              precision    recall  f1-score   support

        Fast       0.50      0.49      0.50      1499
      Medium       0.36      0.30      0.33      1456
        Slow       0.45      0.53      0.48      1453

    accuracy                           0.44      4408
   macro avg       0.44      0.44      0.44      4408
weighted avg       0.44      0.44      0.44      4408



Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
/tmp/ipykernel_3286/37

seed 44 ep 1 valMacroF1 0.4147 valAcc 0.4469 (21s)
seed 44 ep 2 valMacroF1 0.4419 valAcc 0.4646 (21s)
seed 44 ep 3 valMacroF1 0.4539 valAcc 0.4648 (22s)
seed 44 ep 4 valMacroF1 0.4506 valAcc 0.4551 (22s)
{'seed': 44, 'valF1': 0.4539346120987691, 'acc': 0.4557622504537205, 'macroF1': 0.44563666233885035}
              precision    recall  f1-score   support

        Fast       0.50      0.58      0.54      1499
      Medium       0.38      0.27      0.32      1456
        Slow       0.46      0.51      0.49      1453

    accuracy                           0.46      4408
   macro avg       0.45      0.45      0.45      4408
weighted avg       0.45      0.46      0.45      4408

   seed   valF1     acc  macroF1
0    42  0.4625  0.4587   0.4558
1    43  0.4415  0.4410   0.4362
2    44  0.4539  0.4558   0.4456
mean {'acc': 0.4518, 'macroF1': 0.4459} sd {'acc': 0.0095, 'macroF1': 0.0098}


In [ ]:
# Step 2b: final 11-baseline table (mean±SD) + paired bootstrap vs the best model. Run AFTER all baselines finished.
from google.colab import drive
drive.mount('/content/drive')
import os, numpy as np, pandas as pd
from sklearn.metrics import accuracy_score, f1_score
P='/content/drive/MyDrive/Colab Notebooks/'
yte=None; preds={}   # preds[model] = list of per-seed prediction arrays
def add(model,arr):
    preds.setdefault(model,[]).append(np.asarray(arr))
for f in ('clean_github_classic_preds.npz','clean_github_dl_preds.npz'):
    if os.path.exists(P+f):
        z=np.load(P+f); yte=z['y_test']
        for k in z.files:
            if k!='y_test': add(k.rsplit('_s',1)[0],z[k])
    else: print('MISSING',f)
names={'roberta':'RoBERTa (fine-tuned)','bert':'BERT (fine-tuned)','codebert':'CodeBERT (fine-tuned)'}
for tag,nm in names.items():
    for s in (42,43,44):
        f=P+f'ft_{tag}_gh_test_logits_s{s}.npy'
        if os.path.exists(f): add(nm,np.load(f).argmax(1))
        else: print('MISSING',f)
rows=[]
for m,L in preds.items():
    a=[accuracy_score(yte,p) for p in L]; f=[f1_score(yte,p,average='macro') for p in L]
    rows.append(dict(Model=m,n_runs=len(L),Acc=np.mean(a),Acc_sd=np.std(a),MacroF1=np.mean(f),MacroF1_sd=np.std(f)))
tab=pd.DataFrame(rows).sort_values('MacroF1',ascending=False); print(tab.round(4).to_string(index=False))
tab.to_csv(P+'clean_github_final_table.csv',index=False)

best=tab.iloc[0]['Model']; print('\nPaired bootstrap vs best =',best)
rng=np.random.default_rng(0); idx=[rng.integers(0,len(yte),len(yte)) for _ in range(1000)]
pb=preds[best][0]; out=[]
for m,L in preds.items():
    if m==best: continue
    d=np.array([f1_score(yte[i],L[0][i],average='macro')-f1_score(yte[i],pb[i],average='macro') for i in idx])
    out.append(dict(Model=m,diff_vs_best=d.mean(),CI_lo=np.percentile(d,2.5),CI_hi=np.percentile(d,97.5),significant=(np.percentile(d,97.5)<0 or np.percentile(d,2.5)>0)))
sig=pd.DataFrame(out).sort_values('diff_vs_best',ascending=False); print(sig.round(4).to_string(index=False))
sig.to_csv(P+'clean_github_bootstrap_vs_best.csv',index=False)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
MISSING /content/drive/MyDrive/Colab Notebooks/ft_codebert_gh_test_logits_s42.npy
MISSING /content/drive/MyDrive/Colab Notebooks/ft_codebert_gh_test_logits_s43.npy
MISSING /content/drive/MyDrive/Colab Notebooks/ft_codebert_gh_test_logits_s44.npy
               Model  n_runs    Acc  Acc_sd  MacroF1  MacroF1_sd
   BERT (fine-tuned)       3 0.4518  0.0077   0.4459      0.0080
RoBERTa (fine-tuned)       3 0.4592  0.0021   0.4445      0.0030
             XGBoost       3 0.4558  0.0000   0.4435      0.0000
              LogReg       3 0.4476  0.0000   0.4433      0.0000
                LSTM       3 0.4473  0.0084   0.4275      0.0070
              BiLSTM       3 0.4453  0.0106   0.4256      0.0075
        RandomForest       3 0.4488  0.0002   0.4204      0.0007
          CNN+BiLSTM       3 0.4363  0.0183   0.4186      0.0087
                 SVM       3 0.4108  0.0

In [ ]:
# Step R2: clean ablation on Bugzilla (severity words masked) and GitHub (resolution class). TF-IDF + XGBoost, 5 seeds.
# Configs: A1 TF-IDF | A2 +emotion probs | A3 +keyword DDS | A4 +both (distress) | A5 distress only
from google.colab import drive
drive.mount('/content/drive')
import re, numpy as np, pandas as pd, scipy.sparse as sp
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import accuracy_score, f1_score
from xgboost import XGBClassifier
P='/content/drive/MyDrive/Colab Notebooks/'
SEV=r'\b(blockers?|critical|criticals|major|majors|minor|minors|trivial|normal|severity|sev|enhancement|regression)\b'
LEX={'frustration':['frustrating','annoying','broken','failed','crash','error'],'confusion':['confused','unclear','dont understand','unknown','complicated'],
 'anxiety':['worried','anxious','fear','unsafe','security'],'urgency':['urgent','immediately','asap','critical','important'],
 'helplessness':['stuck','unable','cannot','helpless','nothing works'],'anger':['hate','terrible','worst','angry','useless']}
def dds(texts): return np.array([[sum(t.lower().count(w) for w in ws) for ws in LEX.values()] for t in texts])
PLAT={'Bugzilla':dict(pre='bugzilla',tag='bz',cid={'Critical':0,'High':1,'Low':2},mask=True),
      'GitHub':dict(pre='github',tag='gh',cid={'Fast':0,'Medium':1,'Slow':2},mask=False)}
allrows=[]
for pname,c in PLAT.items():
    d={n:pd.read_csv(P+f"clean_{c['pre']}_{n}.csv") for n in ('train','val','test')}
    for n in d: d[n]['text']=d[n]['text'].fillna('').astype(str)
    if c['mask']:
        for n in d: d[n]['text']=d[n]['text'].map(lambda s: re.sub(r'\s+',' ',re.sub(SEV,' ',s)))
    y={n:d[n]['y'].map(c['cid']).values for n in d}
    E={n:np.load(P+f"emo_probs_{c['tag']}_{n}.npy") for n in d}
    K={n:dds(d[n].text) for n in d}
    blocks={'emo':{n:E[n] for n in d},'kw':{n:np.c_[K[n],K[n].sum(1)] for n in d}}
    def dense(keys,n): return np.hstack([blocks[k][n] for k in keys]) if keys else None
    tf=TfidfVectorizer(max_features=20000,ngram_range=(1,2),min_df=2,sublinear_tf=True).fit(d['train'].text)
    T={n:tf.transform(d[n].text) for n in d}
    cfgs={'A1 TF-IDF':(True,[]),'A2 +emotion probs':(True,['emo']),'A3 +keyword DDS':(True,['kw']),'A4 +both (distress)':(True,['emo','kw']),'A5 distress only':(False,['emo','kw'])}
    sw=compute_sample_weight('balanced',y['train']); preds={}
    for cname,(use_t,keys) in cfgs.items():
        sc=StandardScaler().fit(dense(keys,'train')) if keys else None
        def X(n):
            parts=([T[n]] if use_t else [])+([sp.csr_matrix(sc.transform(dense(keys,n)))] if keys else [])
            return sp.hstack(parts).tocsr()
        f=[];a=[]
        for s in (42,43,44,45,46):
            m=XGBClassifier(n_estimators=300,max_depth=6,learning_rate=0.1,subsample=0.8,colsample_bytree=0.8,tree_method='hist',device='cuda',random_state=s)
            m.fit(X('train'),y['train'],sample_weight=sw); p=m.predict(X('test')); f.append(f1_score(y['test'],p,average='macro')); a.append(accuracy_score(y['test'],p))
            if s==42: preds[cname]=p
        allrows.append(dict(Platform=pname,Config=cname,Acc=np.mean(a),MacroF1=np.mean(f),MacroF1_sd=np.std(f)))
    # bootstrap each config vs A1 (seed 42)
    rng=np.random.default_rng(0); idx=[rng.integers(0,len(y['test']),len(y['test'])) for _ in range(1000)]; base=preds['A1 TF-IDF']
    for cname in cfgs:
        if cname=='A1 TF-IDF': continue
        dd=np.array([f1_score(y['test'][i],preds[cname][i],average='macro')-f1_score(y['test'][i],base[i],average='macro') for i in idx])
        for r in allrows:
            if r['Platform']==pname and r['Config']==cname: r.update(diff_vs_A1=dd.mean(),CI_lo=np.percentile(dd,2.5),CI_hi=np.percentile(dd,97.5))
res=pd.DataFrame(allrows); print(res.round(4).to_string(index=False)); res.to_csv(P+'clean_ablation_results.csv',index=False)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Platform              Config    Acc  MacroF1  MacroF1_sd  diff_vs_A1   CI_lo   CI_hi
Bugzilla           A1 TF-IDF 0.7396   0.5344      0.0027         NaN     NaN     NaN
Bugzilla   A2 +emotion probs 0.7418   0.5410      0.0043      0.0019 -0.0096  0.0135
Bugzilla     A3 +keyword DDS 0.7434   0.5390      0.0042      0.0080 -0.0034  0.0200
Bugzilla A4 +both (distress) 0.7425   0.5413      0.0058      0.0161  0.0030  0.0286
Bugzilla    A5 distress only 0.6543   0.4595      0.0043     -0.0685 -0.0881 -0.0503
  GitHub           A1 TF-IDF 0.4558   0.4446      0.0030         NaN     NaN     NaN
  GitHub   A2 +emotion probs 0.4574   0.4464      0.0020      0.0005 -0.0092  0.0106
  GitHub     A3 +keyword DDS 0.4546   0.4437      0.0029      0.0014 -0.0068  0.0095
  GitHub A4 +both (distress) 0.4574   0.4465      0.0051      0.0026 -0.0061  0.0125
  GitHub    A5 distre

In [ ]:
# Step R3: ROC-AUC, per-class P/R/F1, confusion-matrix and ROC figures, McNemar tests. Bugzilla (masked) and GitHub.
from google.colab import drive
drive.mount('/content/drive')
import re, numpy as np, pandas as pd, scipy.sparse as sp, matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scipy.special import softmax
from scipy.stats import chi2
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler, label_binarize
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import roc_auc_score, roc_curve, classification_report, confusion_matrix, f1_score
from xgboost import XGBClassifier
P='/content/drive/MyDrive/Colab Notebooks/'
SEV=r'\b(blockers?|critical|criticals|major|majors|minor|minors|trivial|normal|severity|sev|enhancement|regression)\b'
LEX={'frustration':['frustrating','annoying','broken','failed','crash','error'],'confusion':['confused','unclear','dont understand','unknown','complicated'],
 'anxiety':['worried','anxious','fear','unsafe','security'],'urgency':['urgent','immediately','asap','critical','important'],
 'helplessness':['stuck','unable','cannot','helpless','nothing works'],'anger':['hate','terrible','worst','angry','useless']}
dds=lambda texts: np.array([[sum(t.lower().count(w) for w in ws) for ws in LEX.values()] for t in texts])
PLAT={'Bugzilla':dict(pre='bugzilla',tag='bz',classes=['Critical','High','Low'],mask=True),
      'GitHub':dict(pre='github',tag='gh',classes=['Fast','Medium','Slow'],mask=False)}
def mcnemar(y,a,b):
    ca,cb=(a==y),(b==y); n01=int((~ca&cb).sum()); n10=int((ca&~cb).sum()); s=n01+n10
    stat=(abs(n01-n10)-1)**2/s if s else 0.0; return n01,n10,1-chi2.cdf(stat,1)
summary=[]
for pname,c in PLAT.items():
    cls=c['classes']; cid={k:i for i,k in enumerate(cls)}
    d={n:pd.read_csv(P+f"clean_{c['pre']}_{n}.csv") for n in ('train','val','test')}
    for n in d: d[n]['text']=d[n]['text'].fillna('').astype(str)
    if c['mask']:
        for n in d: d[n]['text']=d[n]['text'].map(lambda s: re.sub(r'\s+',' ',re.sub(SEV,' ',s)))
    y={n:d[n]['y'].map(cid).values for n in d}; Y=label_binarize(y['test'],classes=[0,1,2])
    E={n:np.load(P+f"emo_probs_{c['tag']}_{n}.npy") for n in d}; K={n:dds(d[n].text) for n in d}
    D={n:np.c_[E[n],K[n],K[n].sum(1)] for n in d}; sc=StandardScaler().fit(D['train']); D={n:sc.transform(D[n]) for n in d}
    tf=TfidfVectorizer(max_features=20000,ngram_range=(1,2),min_df=2,sublinear_tf=True).fit(d['train'].text); T={n:tf.transform(d[n].text) for n in d}
    sw=compute_sample_weight('balanced',y['train']); probs={}; aucs={}
    for name,use in (('XGB TF-IDF',False),('XGB TF-IDF+distress',True)):
        X=lambda n: sp.hstack([T[n],sp.csr_matrix(D[n])]).tocsr() if use else T[n]; a=[]
        for s in (42,43,44):
            m=XGBClassifier(n_estimators=300,max_depth=6,learning_rate=0.1,subsample=0.8,colsample_bytree=0.8,tree_method='hist',device='cuda',random_state=s).fit(X('train'),y['train'],sample_weight=sw)
            pr=m.predict_proba(X('test')); a.append(roc_auc_score(Y,pr,average='macro'))
            if s==42: probs[name]=pr
        aucs[name]=(np.mean(a),np.std(a))
    a=[];
    for s in (42,43,44):
        pr=softmax(np.load(P+f"ft_roberta_{c['tag']}_test_logits_s{s}.npy"),axis=1); a.append(roc_auc_score(Y,pr,average='macro'))
        if s==42: probs['RoBERTa (fine-tuned)']=pr
    aucs['RoBERTa (fine-tuned)']=(np.mean(a),np.std(a))
    print(f'\n===== {pname} =====')
    for k,v in aucs.items(): print('macro OvR ROC-AUC %-24s %.4f ± %.4f'%(k,*v)); summary.append(dict(Platform=pname,Model=k,ROC_AUC=v[0],AUC_sd=v[1]))
    best='XGB TF-IDF+distress'; pb=probs[best].argmax(1)
    print(f'\nPer-class report ({best}, seed 42):'); print(classification_report(y['test'],pb,target_names=cls,digits=3))
    for other in ('XGB TF-IDF','RoBERTa (fine-tuned)'):
        n01,n10,p=mcnemar(y['test'],pb,probs[other].argmax(1)); print(f'McNemar {best} vs {other}: only-{best}-right={n10}, only-{other}-right={n01}, p={p:.4f}')
    # confusion matrices (row-normalised)
    fig,ax=plt.subplots(1,3,figsize=(13,4))
    for a_,(k,pr) in zip(ax,probs.items()):
        cm=confusion_matrix(y['test'],pr.argmax(1),normalize='true'); a_.imshow(cm,cmap='Blues',vmin=0,vmax=1)
        for i in range(3):
            for j in range(3): a_.text(j,i,f'{cm[i,j]:.2f}',ha='center',va='center',color='white' if cm[i,j]>0.5 else 'black')
        a_.set_xticks(range(3),cls); a_.set_yticks(range(3),cls); a_.set_title(k,fontsize=10); a_.set_xlabel('Predicted'); a_.set_ylabel('True')
    plt.tight_layout(); plt.savefig(P+f'fig_confusion_{pname.lower()}.png',dpi=300); plt.close()
    # ROC curves (one-vs-rest, per class, best model) + macro comparison
    fig,ax=plt.subplots(1,2,figsize=(10,4))
    for j,cn in enumerate(cls):
        fpr,tpr,_=roc_curve(Y[:,j],probs[best][:,j]); ax[0].plot(fpr,tpr,label=f'{cn} (AUC {roc_auc_score(Y[:,j],probs[best][:,j]):.2f})')
    ax[0].plot([0,1],[0,1],'k--',lw=0.8); ax[0].set_title(f'{pname}: per-class ROC ({best})',fontsize=10); ax[0].set_xlabel('False positive rate'); ax[0].set_ylabel('True positive rate'); ax[0].legend()
    for k,pr in probs.items():
        mf=np.mean([roc_curve(Y[:,j],pr[:,j])[1] for j in range(3)],axis=0) if False else None
    names=list(aucs); vals=[aucs[k][0] for k in names]; sds=[aucs[k][1] for k in names]
    ax[1].bar(range(len(names)),vals,yerr=sds,color=['#7f8fa6','#2f6db5','#c0803a']); ax[1].set_xticks(range(len(names)),[n.replace(' ','\n',1) for n in names],fontsize=8)
    ax[1].set_ylim(0.5,max(vals)+0.08); ax[1].set_title('Macro ROC-AUC (mean ± SD, 3 seeds)',fontsize=10)
    plt.tight_layout(); plt.savefig(P+f'fig_roc_{pname.lower()}.png',dpi=300); plt.close()
pd.DataFrame(summary).to_csv(P+'clean_roc_auc_summary.csv',index=False); print('\nsaved figures + csv to Drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

===== Bugzilla =====
macro OvR ROC-AUC XGB TF-IDF               0.7542 ± 0.0029
macro OvR ROC-AUC XGB TF-IDF+distress      0.7618 ± 0.0017
macro OvR ROC-AUC RoBERTa (fine-tuned)     0.7602 ± 0.0062

Per-class report (XGB TF-IDF+distress, seed 42):
              precision    recall  f1-score   support

    Critical      0.474     0.615     0.535       571
        High      0.283     0.241     0.261       564
         Low      0.870     0.850     0.859      3728

    accuracy                          0.751      4863
   macro avg      0.542     0.568     0.552      4863
weighted avg      0.755     0.751     0.752      4863

McNemar XGB TF-IDF+distress vs XGB TF-IDF: only-XGB TF-IDF+distress-right=206, only-XGB TF-IDF-right=154, p=0.0072
McNemar XGB TF-IDF+distress vs RoBERTa (fine-tuned): only-XGB TF-IDF+distress-right=791, only-RoBERTa (fine-tuned)-right=365, 

In [ ]:
from google.colab import drive, files
import glob, zipfile, os
drive.mount('/content/drive')
P='/content/drive/MyDrive/Colab Notebooks/'

patterns=['fig_*.png','clean_*results*.csv','clean_*seedwise.csv','clean_*final_table.csv','clean_*bootstrap*.csv',
          'clean_ablation_results.csv','clean_roc_auc_summary.csv','clean_*distress*.csv','distress_vs_label_*.csv',
          'emotion_by_platform.csv','bugzilla_shap_*.csv','bugzilla_error_examples.csv','so_emotion_analysis.csv']
found=sorted({f for p in patterns for f in glob.glob(P+p)})
print(len(found),'result files:'); [print(' ',os.path.basename(f),'%.1f KB'%(os.path.getsize(f)/1024)) for f in found]
with zipfile.ZipFile('/content/all_results.zip','w',zipfile.ZIP_DEFLATED) as z:
    for f in found: z.write(f,os.path.basename(f))
files.download('/content/all_results.zip')

# OPTIONAL: clean train/val/test split files (reviewers/Prof-ku share panna)
splits=sorted(glob.glob(P+'clean_bugzilla_train.csv')+glob.glob(P+'clean_bugzilla_val.csv')+glob.glob(P+'clean_bugzilla_test.csv')
              +glob.glob(P+'clean_github_train.csv')+glob.glob(P+'clean_github_val.csv')+glob.glob(P+'clean_github_test.csv'))
with zipfile.ZipFile('/content/clean_splits.zip','w',zipfile.ZIP_DEFLATED) as z:
    for f in splits: z.write(f,os.path.basename(f))
print('splits zip: %.1f MB'%(os.path.getsize('/content/clean_splits.zip')/1e6))
files.download('/content/clean_splits.zip')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
30 result files:
  bugzilla_error_examples.csv 2.7 KB
  bugzilla_shap_distress.csv 0.6 KB
  bugzilla_shap_top_features.csv 1.4 KB
  clean_ablation_results.csv 1.4 KB
  clean_bugzilla_bootstrap_vs_best.csv 0.8 KB
  clean_bugzilla_classic_seedwise.csv 0.8 KB
  clean_bugzilla_distress_on_xgb.csv 0.7 KB
  clean_bugzilla_distress_results.csv 0.4 KB
  clean_bugzilla_dl_seedwise.csv 0.5 KB
  clean_bugzilla_final_table.csv 0.9 KB
  clean_bugzilla_finetune_bert_results.csv 0.2 KB
  clean_bugzilla_finetune_codebert_results.csv 0.2 KB
  clean_bugzilla_finetune_results.csv 0.2 KB
  clean_bugzilla_results.csv 0.5 KB
  clean_github_bootstrap_vs_best.csv 0.7 KB
  clean_github_classic_seedwise.csv 0.8 KB
  clean_github_distress_on_xgb.csv 0.7 KB
  clean_github_dl_seedwise.csv 0.5 KB
  clean_github_final_table.csv 0.8 KB
  clean_github_finetune_bert_results.csv 0.2 KB
  clean

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

splits zip: 48.0 MB


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# GitHub fine-tune (real Severity, class-weighted loss)
# Colab T4. Model chosen on VAL macro-F1; test scored once per seed. Saves logits for later stacking.
from google.colab import drive
drive.mount('/content/drive')
import numpy as np, pandas as pd, torch, torch.nn as nn, time
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from sklearn.metrics import accuracy_score, f1_score, classification_report
from sklearn.utils.class_weight import compute_class_weight

# Set these two, run once for BERT and once for CodeBERT:
MODEL='microsoft/codebert-base'
TAG='codebert'
P='/content/drive/MyDrive/Colab Notebooks/'
dev='cuda'
tr,va,te=[pd.read_csv(P+f'clean_github_{n}.csv') for n in ('train','val','test')]
for d in (tr,va,te): d['text']=d['text'].fillna('').astype(str)
classes=['Fast','Medium','Slow']; cid={c:i for i,c in enumerate(classes)}
Y={n:d['y'].map(cid).values for n,d in (('train',tr),('val',va),('test',te))}
tok=AutoTokenizer.from_pretrained(MODEL); MAXLEN=192; BS=32

def enc(d):
    e=tok(list(d.text),truncation=True,max_length=MAXLEN,padding='max_length',return_tensors='pt')
    return e['input_ids'],e['attention_mask']
E={n:enc(d) for n,d in (('train',tr),('val',va),('test',te))}

@torch.no_grad()
def logits_of(model,n):
    model.eval(); ids,am=E[n]; out=[]
    for i in range(0,len(ids),128):
        with torch.autocast('cuda',dtype=torch.float16):
            out.append(model(input_ids=ids[i:i+128].to(dev),attention_mask=am[i:i+128].to(dev)).logits.float().cpu())
    return torch.cat(out).numpy()

def run(seed,epochs=4,lr=2e-5):
    torch.manual_seed(seed); np.random.seed(seed)
    model=AutoModelForSequenceClassification.from_pretrained(MODEL,num_labels=3).to(dev)
    w=torch.tensor(compute_class_weight('balanced',classes=np.arange(3),y=Y['train']),dtype=torch.float32,device=dev)
    lossf=nn.CrossEntropyLoss(weight=w)
    opt=torch.optim.AdamW(model.parameters(),lr=lr,weight_decay=0.01)
    ids,am=E['train']; ytr=torch.tensor(Y['train'])
    steps=epochs*((len(ids)+BS-1)//BS); sch=get_linear_schedule_with_warmup(opt,int(0.06*steps),steps)
    scaler=torch.cuda.amp.GradScaler(); best,bstate=-1,None
    for ep in range(epochs):
        model.train(); perm=torch.randperm(len(ids)); t0=time.time()
        for i in range(0,len(perm),BS):
            ix=perm[i:i+BS]
            with torch.autocast('cuda',dtype=torch.float16):
                lg=model(input_ids=ids[ix].to(dev),attention_mask=am[ix].to(dev)).logits
            loss=lossf(lg.float(),ytr[ix].to(dev))
            opt.zero_grad(); scaler.scale(loss).backward(); scaler.unscale_(opt)
            nn.utils.clip_grad_norm_(model.parameters(),1.0); scaler.step(opt); scaler.update(); sch.step()
        lv=logits_of(model,'val'); f=f1_score(Y['val'],lv.argmax(1),average='macro')
        print(f'seed {seed} ep {ep+1} valMacroF1 {f:.4f} valAcc {accuracy_score(Y["val"],lv.argmax(1)):.4f} ({time.time()-t0:.0f}s)')
        if f>best: best=f; bstate={k:v.detach().clone() for k,v in model.state_dict().items()}
    model.load_state_dict(bstate)
    lv,lt=logits_of(model,'val'),logits_of(model,'test')
    np.save(P+f'ft_{TAG}_gh_val_logits_s{seed}.npy',lv); np.save(P+f'ft_{TAG}_gh_test_logits_s{seed}.npy',lt)
    pt=lt.argmax(1)
    r=dict(seed=seed,valF1=best,acc=accuracy_score(Y['test'],pt),macroF1=f1_score(Y['test'],pt,average='macro'))
    print(r); print(classification_report(Y['test'],pt,target_names=classes))
    del model,opt; torch.cuda.empty_cache(); return r

rows=[run(s) for s in (42,43,44)]
res=pd.DataFrame(rows); print(res.round(4)); print('mean',res[['acc','macroF1']].mean().round(4).to_dict(),'sd',res[['acc','macroF1']].std().round(4).to_dict())
res.to_csv(P+f'clean_github_finetune_{TAG}_results.csv',index=False)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


config.json:   0%|          | 0.00/498 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/150 [00:00<?, ?B/s]

... (9 outputs left out)

seed 43 ep 1 valMacroF1 0.4109 valAcc 0.4480 (22s)
seed 43 ep 2 valMacroF1 0.4532 valAcc 0.4669 (22s)
seed 43 ep 3 valMacroF1 0.4743 valAcc 0.4825 (22s)
seed 43 ep 4 valMacroF1 0.4576 valAcc 0.4646 (22s)
{'seed': 43, 'valF1': 0.47433512038202935, 'acc': 0.4643829401088929, 'macroF1': 0.4547706752935981}
              precision    recall  f1-score   support

        Fast       0.51      0.59      0.54      1499
      Medium       0.38      0.28      0.32      1456
        Slow       0.48      0.52      0.50      1453

    accuracy                           0.46      4408
   macro avg       0.45      0.46      0.45      4408
weighted avg       0.45      0.46      0.46      4408



Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: microsoft/codebert-base
Key                        | Status     | 
---------------------------+------------+-
pooler.dense.bias          | UNEXPECTED | 
pooler.dense.weight        | UNEXPECTED | 
classifier.out_proj.weight | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.bias      | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
/tmp/ipykernel_3286/2716675055.py:42: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler=torch.cuda.amp.GradScaler(); best,bstate=-1,None


seed 44 ep 1 valMacroF1 0.3594 valAcc 0.4478 (22s)
seed 44 ep 2 valMacroF1 0.4460 valAcc 0.4673 (22s)
seed 44 ep 3 valMacroF1 0.4386 valAcc 0.4682 (23s)
seed 44 ep 4 valMacroF1 0.4568 valAcc 0.4680 (22s)
{'seed': 44, 'valF1': 0.4568260612041284, 'acc': 0.45462794918330307, 'macroF1': 0.4436523688952409}
              precision    recall  f1-score   support

        Fast       0.50      0.57      0.53      1499
      Medium       0.37      0.26      0.31      1456
        Slow       0.46      0.54      0.49      1453

    accuracy                           0.45      4408
   macro avg       0.44      0.45      0.44      4408
weighted avg       0.44      0.45      0.44      4408

   seed   valF1     acc  macroF1
0    42  0.4489  0.4583   0.4403
1    43  0.4743  0.4644   0.4548
2    44  0.4568  0.4546   0.4437
mean {'acc': 0.4591, 'macroF1': 0.4462} sd {'acc': 0.0049, 'macroF1': 0.0076}


In [ ]:
# Step 2b: final 11-baseline table (mean±SD) + paired bootstrap vs the best model. Run AFTER all baselines finished.
from google.colab import drive
drive.mount('/content/drive')
import os, numpy as np, pandas as pd
from sklearn.metrics import accuracy_score, f1_score
P='/content/drive/MyDrive/Colab Notebooks/'
yte=None; preds={}   # preds[model] = list of per-seed prediction arrays
def add(model,arr):
    preds.setdefault(model,[]).append(np.asarray(arr))
for f in ('clean_github_classic_preds.npz','clean_github_dl_preds.npz'):
    if os.path.exists(P+f):
        z=np.load(P+f); yte=z['y_test']
        for k in z.files:
            if k!='y_test': add(k.rsplit('_s',1)[0],z[k])
    else: print('MISSING',f)
names={'roberta':'RoBERTa (fine-tuned)','bert':'BERT (fine-tuned)','codebert':'CodeBERT (fine-tuned)'}
for tag,nm in names.items():
    for s in (42,43,44):
        f=P+f'ft_{tag}_gh_test_logits_s{s}.npy'
        if os.path.exists(f): add(nm,np.load(f).argmax(1))
        else: print('MISSING',f)
rows=[]
for m,L in preds.items():
    a=[accuracy_score(yte,p) for p in L]; f=[f1_score(yte,p,average='macro') for p in L]
    rows.append(dict(Model=m,n_runs=len(L),Acc=np.mean(a),Acc_sd=np.std(a),MacroF1=np.mean(f),MacroF1_sd=np.std(f)))
tab=pd.DataFrame(rows).sort_values('MacroF1',ascending=False); print(tab.round(4).to_string(index=False))
tab.to_csv(P+'clean_github_final_table.csv',index=False)

best=tab.iloc[0]['Model']; print('\nPaired bootstrap vs best =',best)
rng=np.random.default_rng(0); idx=[rng.integers(0,len(yte),len(yte)) for _ in range(1000)]
pb=preds[best][0]; out=[]
for m,L in preds.items():
    if m==best: continue
    d=np.array([f1_score(yte[i],L[0][i],average='macro')-f1_score(yte[i],pb[i],average='macro') for i in idx])
    out.append(dict(Model=m,diff_vs_best=d.mean(),CI_lo=np.percentile(d,2.5),CI_hi=np.percentile(d,97.5),significant=(np.percentile(d,97.5)<0 or np.percentile(d,2.5)>0)))
sig=pd.DataFrame(out).sort_values('diff_vs_best',ascending=False); print(sig.round(4).to_string(index=False))
sig.to_csv(P+'clean_github_bootstrap_vs_best.csv',index=False)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
                Model  n_runs    Acc  Acc_sd  MacroF1  MacroF1_sd
CodeBERT (fine-tuned)       3 0.4591  0.0040   0.4462      0.0062
    BERT (fine-tuned)       3 0.4518  0.0077   0.4459      0.0080
 RoBERTa (fine-tuned)       3 0.4592  0.0021   0.4445      0.0030
              XGBoost       3 0.4558  0.0000   0.4435      0.0000
               LogReg       3 0.4476  0.0000   0.4433      0.0000
                 LSTM       3 0.4473  0.0084   0.4275      0.0070
               BiLSTM       3 0.4453  0.0106   0.4256      0.0075
         RandomForest       3 0.4488  0.0002   0.4204      0.0007
           CNN+BiLSTM       3 0.4363  0.0183   0.4186      0.0087
                  SVM       3 0.4108  0.0000   0.4096      0.0000
           NaiveBayes       3 0.4471  0.0000   0.3936      0.0000

Paired bootstrap vs best = CodeBERT (fine-tuned)
               Model  diff_vs

In [ ]:
# Step 2b: final 11-baseline table (mean±SD) + paired bootstrap vs the best model. Run AFTER all baselines finished.
from google.colab import drive
drive.mount('/content/drive')
import os, numpy as np, pandas as pd
from sklearn.metrics import accuracy_score, f1_score
P='/content/drive/MyDrive/Colab Notebooks/'
yte=None; preds={}   # preds[model] = list of per-seed prediction arrays
def add(model,arr):
    preds.setdefault(model,[]).append(np.asarray(arr))
for f in ('clean_github_classic_preds.npz','clean_github_dl_preds.npz'):
    if os.path.exists(P+f):
        z=np.load(P+f); yte=z['y_test']
        for k in z.files:
            if k!='y_test': add(k.rsplit('_s',1)[0],z[k])
    else: print('MISSING',f)
names={'roberta':'RoBERTa (fine-tuned)','bert':'BERT (fine-tuned)','codebert':'CodeBERT (fine-tuned)'}
for tag,nm in names.items():
    for s in (42,43,44):
        f=P+f'ft_{tag}_gh_test_logits_s{s}.npy'
        if os.path.exists(f): add(nm,np.load(f).argmax(1))
        else: print('MISSING',f)
rows=[]
for m,L in preds.items():
    a=[accuracy_score(yte,p) for p in L]; f=[f1_score(yte,p,average='macro') for p in L]
    rows.append(dict(Model=m,n_runs=len(L),Acc=np.mean(a),Acc_sd=np.std(a),MacroF1=np.mean(f),MacroF1_sd=np.std(f)))
tab=pd.DataFrame(rows).sort_values('MacroF1',ascending=False); print(tab.round(4).to_string(index=False))
tab.to_csv(P+'clean_github_final_table.csv',index=False)

best=tab.iloc[0]['Model']; print('\nPaired bootstrap vs best =',best,'(differences averaged over matched seeds)')
rng=np.random.default_rng(0); idx=[rng.integers(0,len(yte),len(yte)) for _ in range(1000)]
pb=preds[best]; out=[]
for m,L in preds.items():
    if m==best: continue
    k=min(len(L),len(pb))
    d=np.mean([[f1_score(yte[i],L[s][i],average='macro')-f1_score(yte[i],pb[s][i],average='macro') for i in idx] for s in range(k)],axis=0)
    out.append(dict(Model=m,diff_vs_best=d.mean(),CI_lo=np.percentile(d,2.5),CI_hi=np.percentile(d,97.5),significant=(np.percentile(d,97.5)<0 or np.percentile(d,2.5)>0)))
sig=pd.DataFrame(out).sort_values('diff_vs_best',ascending=False); print(sig.round(4).to_string(index=False))
sig.to_csv(P+'clean_github_bootstrap_vs_best.csv',index=False)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
                Model  n_runs    Acc  Acc_sd  MacroF1  MacroF1_sd
CodeBERT (fine-tuned)       3 0.4591  0.0040   0.4462      0.0062
    BERT (fine-tuned)       3 0.4518  0.0077   0.4459      0.0080
 RoBERTa (fine-tuned)       3 0.4592  0.0021   0.4445      0.0030
              XGBoost       3 0.4558  0.0000   0.4435      0.0000
               LogReg       3 0.4476  0.0000   0.4433      0.0000
                 LSTM       3 0.4473  0.0084   0.4275      0.0070
               BiLSTM       3 0.4453  0.0106   0.4256      0.0075
         RandomForest       3 0.4488  0.0002   0.4204      0.0007
           CNN+BiLSTM       3 0.4363  0.0183   0.4186      0.0087
                  SVM       3 0.4108  0.0000   0.4096      0.0000
           NaiveBayes       3 0.4471  0.0000   0.3936      0.0000

Paired bootstrap vs best = CodeBERT (fine-tuned) (differences averaged over ma

In [ ]:
# Step 2b: final 11-baseline table (mean±SD) + paired bootstrap vs the best model. Run AFTER all baselines finished.
from google.colab import drive
drive.mount('/content/drive')
import os, numpy as np, pandas as pd
from sklearn.metrics import accuracy_score, f1_score
P='/content/drive/MyDrive/Colab Notebooks/'
yte=None; preds={}   # preds[model] = list of per-seed prediction arrays
def add(model,arr):
    preds.setdefault(model,[]).append(np.asarray(arr))
for f in ('clean_bugzilla_classic_preds.npz','clean_bugzilla_dl_preds.npz'):
    if os.path.exists(P+f):
        z=np.load(P+f); yte=z['y_test']
        for k in z.files:
            if k!='y_test': add(k.rsplit('_s',1)[0],z[k])
    else: print('MISSING',f)
names={'roberta':'RoBERTa (fine-tuned)','bert':'BERT (fine-tuned)','codebert':'CodeBERT (fine-tuned)'}
for tag,nm in names.items():
    for s in (42,43,44):
        f=P+f'ft_{tag}_bz_test_logits_s{s}.npy'
        if os.path.exists(f): add(nm,np.load(f).argmax(1))
        else: print('MISSING',f)
rows=[]
for m,L in preds.items():
    a=[accuracy_score(yte,p) for p in L]; f=[f1_score(yte,p,average='macro') for p in L]
    rows.append(dict(Model=m,n_runs=len(L),Acc=np.mean(a),Acc_sd=np.std(a),MacroF1=np.mean(f),MacroF1_sd=np.std(f)))
tab=pd.DataFrame(rows).sort_values('MacroF1',ascending=False); print(tab.round(4).to_string(index=False))
tab.to_csv(P+'clean_bugzilla_final_table.csv',index=False)

best=tab.iloc[0]['Model']; print('\nPaired bootstrap vs best =',best,'(differences averaged over matched seeds)')
rng=np.random.default_rng(0); idx=[rng.integers(0,len(yte),len(yte)) for _ in range(1000)]
pb=preds[best]; out=[]
for m,L in preds.items():
    if m==best: continue
    k=min(len(L),len(pb))
    d=np.mean([[f1_score(yte[i],L[s][i],average='macro')-f1_score(yte[i],pb[s][i],average='macro') for i in idx] for s in range(k)],axis=0)
    out.append(dict(Model=m,diff_vs_best=d.mean(),CI_lo=np.percentile(d,2.5),CI_hi=np.percentile(d,97.5),significant=(np.percentile(d,97.5)<0 or np.percentile(d,2.5)>0)))
sig=pd.DataFrame(out).sort_values('diff_vs_best',ascending=False); print(sig.round(4).to_string(index=False))
sig.to_csv(P+'clean_bugzilla_bootstrap_vs_best.csv',index=False)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
                Model  n_runs    Acc  Acc_sd  MacroF1  MacroF1_sd
              XGBoost       3 0.7452  0.0000   0.5476      0.0000
               LogReg       3 0.6985  0.0000   0.5358      0.0000
                  SVM       3 0.7543  0.0000   0.5330      0.0000
CodeBERT (fine-tuned)       3 0.6582  0.0187   0.5200      0.0060
 RoBERTa (fine-tuned)       3 0.6789  0.0108   0.5195      0.0072
    BERT (fine-tuned)       3 0.6324  0.0087   0.5011      0.0028
           NaiveBayes       3 0.7571  0.0000   0.4859      0.0000
           CNN+BiLSTM       3 0.6856  0.0356   0.4843      0.0137
               BiLSTM       3 0.6312  0.0177   0.4587      0.0167
                 LSTM       3 0.6678  0.0496   0.4585      0.0104
         RandomForest       3 0.7872  0.0009   0.4332      0.0027

Paired bootstrap vs best = XGBoost (differences averaged over matched seeds)
 

In [ ]:
# Pending experiment 2: (a) 10 repeated stratified 70:15:15 splits -> Wilcoxon signed-rank (XGB+distress vs XGB vs LogReg)
#                       (b) chi-square top TF-IDF terms per class (Bugzilla masked, GitHub)
from google.colab import drive
drive.mount('/content/drive')
import re, numpy as np, pandas as pd, scipy.sparse as sp
from scipy.stats import wilcoxon
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_selection import chi2
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import f1_score
from xgboost import XGBClassifier
P='/content/drive/MyDrive/Colab Notebooks/'
SEV=r'\b(blockers?|critical|criticals|major|majors|minor|minors|trivial|normal|severity|sev|enhancement|regression)\b'
LEX={'frustration':['frustrating','annoying','broken','failed','crash','error'],'confusion':['confused','unclear','dont understand','unknown','complicated'],
 'anxiety':['worried','anxious','fear','unsafe','security'],'urgency':['urgent','immediately','asap','critical','important'],
 'helplessness':['stuck','unable','cannot','helpless','nothing works'],'anger':['hate','terrible','worst','angry','useless']}
dds=lambda texts: np.array([[sum(t.lower().count(w) for w in ws) for ws in LEX.values()] for t in texts])
PLAT={'Bugzilla':dict(pre='bugzilla',tag='bz',cls=['Critical','High','Low'],mask=True),'GitHub':dict(pre='github',tag='gh',cls=['Fast','Medium','Slow'],mask=False)}
out=[]
for pname,c in PLAT.items():
    parts=[pd.read_csv(P+f"clean_{c['pre']}_{n}.csv") for n in ('train','val','test')]
    df=pd.concat(parts,ignore_index=True); df['text']=df['text'].fillna('').astype(str)
    E=np.vstack([np.load(P+f"emo_probs_{c['tag']}_{n}.npy") for n in ('train','val','test')])
    if c['mask']: df['text']=df['text'].map(lambda s: re.sub(r'\s+',' ',re.sub(SEV,' ',s)))
    K=dds(df.text); Dall=np.c_[E,K,K.sum(1)]; y=df['y'].map({k:i for i,k in enumerate(c['cls'])}).values
    print(f'\n{pname}: pooled n={len(df)}')
    rows=[]
    for rep in range(10):
        tr,tmp=train_test_split(np.arange(len(df)),test_size=0.30,stratify=y,random_state=rep)
        _,te=train_test_split(tmp,test_size=0.50,stratify=y[tmp],random_state=rep)
        tf=TfidfVectorizer(max_features=20000,ngram_range=(1,2),min_df=2,sublinear_tf=True).fit(df.text.iloc[tr])
        Ttr,Tte=tf.transform(df.text.iloc[tr]),tf.transform(df.text.iloc[te])
        sc=StandardScaler().fit(Dall[tr]); Dtr,Dte=sc.transform(Dall[tr]),sc.transform(Dall[te]); sw=compute_sample_weight('balanced',y[tr])
        r=dict(rep=rep)
        r['LogReg']=f1_score(y[te],LogisticRegression(max_iter=1000,class_weight='balanced').fit(Ttr,y[tr]).predict(Tte),average='macro')
        mk=lambda: XGBClassifier(n_estimators=300,max_depth=6,learning_rate=0.1,subsample=0.8,colsample_bytree=0.8,tree_method='hist',device='cuda',random_state=rep)
        r['XGB']=f1_score(y[te],mk().fit(Ttr,y[tr],sample_weight=sw).predict(Tte),average='macro')
        r['XGB+distress']=f1_score(y[te],mk().fit(sp.hstack([Ttr,sp.csr_matrix(Dtr)]).tocsr(),y[tr],sample_weight=sw).predict(sp.hstack([Tte,sp.csr_matrix(Dte)]).tocsr()),average='macro')
        rows.append(r); print(rep,{k:round(v,4) for k,v in r.items() if k!='rep'})
    R=pd.DataFrame(rows); print(R[['LogReg','XGB','XGB+distress']].agg(['mean','std']).round(4))
    for a,b in (('XGB+distress','XGB'),('XGB+distress','LogReg'),('XGB','LogReg')):
        d=R[a]-R[b]; w=wilcoxon(R[a],R[b]); print(f'Wilcoxon {a} vs {b}: mean diff {d.mean():+.4f}, W={w.statistic:.1f}, p={w.pvalue:.4f}, wins={int((d>0).sum())}/10')
        out.append(dict(Platform=pname,A=a,B=b,mean_diff=d.mean(),W=w.statistic,p=w.pvalue,wins=int((d>0).sum())))
    R.to_csv(P+f'repeated_splits_{pname.lower()}.csv',index=False)
    # chi-square top terms per class on pooled TF-IDF (unigrams)
    tfu=TfidfVectorizer(max_features=20000,min_df=3,sublinear_tf=True,stop_words='english'); Xu=tfu.fit_transform(df.text); names=np.array(tfu.get_feature_names_out())
    print(f'\n{pname}: top chi-square terms per class (one-vs-rest)')
    for j,cn in enumerate(c['cls']):
        sc2,p=chi2(Xu,(y==j).astype(int)); top=np.argsort(-sc2)[:15]; print(' ',cn,':',', '.join(names[top]))
        out.append(dict(Platform=pname,A='chi2_top_terms',B=cn,mean_diff=np.nan,W=np.nan,p=np.nan,wins=np.nan,terms=', '.join(names[top])))
pd.DataFrame(out).to_csv(P+'wilcoxon_chi2_summary.csv',index=False)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

Bugzilla: pooled n=32415
0 {'LogReg': 0.5333, 'XGB': 0.5391, 'XGB+distress': 0.5461}
1 {'LogReg': 0.5221, 'XGB': 0.5281, 'XGB+distress': 0.5281}
2 {'LogReg': 0.5369, 'XGB': 0.5445, 'XGB+distress': 0.5412}
3 {'LogReg': 0.5317, 'XGB': 0.5389, 'XGB+distress': 0.5383}
4 {'LogReg': 0.534, 'XGB': 0.5428, 'XGB+distress': 0.544}
5 {'LogReg': 0.5163, 'XGB': 0.5421, 'XGB+distress': 0.5293}
6 {'LogReg': 0.5315, 'XGB': 0.5329, 'XGB+distress': 0.5287}
7 {'LogReg': 0.5347, 'XGB': 0.5321, 'XGB+distress': 0.5386}
8 {'LogReg': 0.5241, 'XGB': 0.5336, 'XGB+distress': 0.538}
9 {'LogReg': 0.5283, 'XGB': 0.5296, 'XGB+distress': 0.5271}
      LogReg     XGB  XGB+distress
mean  0.5293  0.5364        0.5360
std   0.0065  0.0058        0.0071
Wilcoxon XGB+distress vs XGB: mean diff -0.0004, W=27.0, p=1.0000, wins=5/10
Wilcoxon XGB+distress vs LogReg: mean diff +0.0067, W=3.0, p=0.009

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import re, json, hashlib, numpy as np, pandas as pd, scipy.sparse as sp, matplotlib
matplotlib.use('Agg'); import matplotlib.pyplot as plt
from scipy.stats import chi2, kruskal, wilcoxon
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler, label_binarize
from sklearn.linear_model import LogisticRegression
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import f1_score, accuracy_score, roc_auc_score, classification_report, confusion_matrix
from xgboost import XGBClassifier
import torch
from transformers import pipeline

P = '/content/drive/MyDrive/Colab Notebooks/'
SEEDS = [42, 43, 44, 45, 46]
CLS = ['Critical', 'High', 'Low']; CID = {k: i for i, k in enumerate(CLS)}
SEV = r'\b(blockers?|critical|criticals|major|majors|minor|minors|trivial|normal|severity|sev|enhancement|regression)\b'
mask = lambda s: re.sub(r'\s+', ' ', re.sub(SEV, ' ', s)).strip()
LEX = {'frustration': ['frustrating', 'annoying', 'broken', 'failed', 'crash', 'error'],
       'confusion': ['confused', 'unclear', 'dont understand', 'unknown', 'complicated'],
       'anxiety': ['worried', 'anxious', 'fear', 'unsafe', 'security'],
       'urgency': ['urgent', 'immediately', 'asap', 'critical', 'important'],
       'helplessness': ['stuck', 'unable', 'cannot', 'helpless', 'nothing works'],
       'anger': ['hate', 'terrible', 'worst', 'angry', 'useless']}
EMO = ['anger', 'disgust', 'fear', 'joy', 'neutral', 'sadness', 'surprise']
dds = lambda texts: np.array([[sum(t.lower().count(w) for w in ws) for ws in LEX.values()] for t in texts])
xgb = lambda s: XGBClassifier(n_estimators=300, max_depth=6, learning_rate=0.1, subsample=0.8, colsample_bytree=0.8,
                              tree_method='hist', device='cuda', random_state=s)
md5 = lambda f: hashlib.md5(open(f, 'rb').read()).hexdigest()[:10]

# ---------- 0. data + dataset version ----------
d = {n: pd.read_csv(P + f'clean_bugzilla_{n}.csv') for n in ('train', 'val', 'test')}
for n in d: d[n]['text'] = d[n]['text'].fillna('').astype(str); d[n]['mtext'] = d[n]['text'].map(mask)
y = {n: d[n]['y'].map(CID).values for n in d}
VERSION = {n: md5(P + f'clean_bugzilla_{n}.csv') for n in d}
print('DATASET VERSION (md5 of clean_bugzilla_*.csv):', VERSION)
print('Bugzilla class counts [Critical, High, Low]:', {n: d[n]['y'].value_counts().reindex(CLS).tolist() for n in d})
g = {n: pd.read_csv(P + f'clean_github_{n}.csv') for n in ('train', 'val', 'test')}
print('GitHub class counts [Fast, Medium, Slow]:', {n: g[n]['y'].value_counts().reindex(['Fast', 'Medium', 'Slow']).tolist() for n in g})
print('GitHub version (md5):', {n: md5(P + f'clean_github_{n}.csv') for n in g})
print('GitHub columns:', list(g['train'].columns))
raw_bz = pd.read_csv(P + 'Bugzilla_sev.csv'); print('Bugzilla raw columns:', list(raw_bz.columns))
print('Bugzilla raw severity counts:', raw_bz['Severity'].value_counts().to_dict())
for c in raw_bz.columns:
    if any(k in c.lower() for k in ('product', 'component', 'date', 'creat', 'open', 'time')):
        s = raw_bz[c]; print(f'  {c}: e.g. {s.dropna().astype(str).head(3).tolist()}  n_unique={s.nunique()}')
raw_gh = pd.read_csv(P + 'GitHub_bugs.csv'); print('GitHub raw columns:', list(raw_gh.columns))
_t = pd.to_datetime(raw_gh['Created'], errors='coerce', utc=True); print('GitHub created range:', _t.min(), '->', _t.max())
for c in raw_gh.columns:
    if any(k in c.lower() for k in ('repo', 'project', 'url', 'owner')):
        print(f'  {c}: n_unique={raw_gh[c].nunique()}  top={raw_gh[c].value_counts().head(5).to_dict()}')

# ---------- 1. emotion probabilities on MASKED text ----------
clf = pipeline('text-classification', model='j-hartmann/emotion-english-distilroberta-base', top_k=None,
               device=0 if torch.cuda.is_available() else -1, truncation=True, max_length=256)
def emo(texts, bs=64):
    out = clf(list(texts), batch_size=bs)
    return np.array([[{r['label']: r['score'] for r in o}[e] for e in EMO] for o in out])
_chk = emo(d['train']['text'].iloc[:200]); _old = np.load(P + 'emo_probs_bz_train.npy')[:200]
print('column-order check vs stored original probs (should be ~0): %.4f' % np.abs(_chk - _old).mean())
Eo, Em = {}, {}
for n in d:
    Eo[n] = np.load(P + f'emo_probs_bz_{n}.npy')
    f = P + f'emo_probs_bzmask_{n}.npy'
    try: Em[n] = np.load(f); print('loaded', f)
    except FileNotFoundError: Em[n] = emo(d[n]['mtext']); np.save(f, Em[n]); print('saved', f)
ch = np.abs(Em['train'] - Eo['train']).sum(1)
print('emotion vectors changed by masking (train): %.1f%% of texts; mean L1 change %.4f; top emotion changed %.1f%%'
      % (100 * (ch > 1e-6).mean(), ch.mean(), 100 * (Em['train'].argmax(1) != Eo['train'].argmax(1)).mean()))

def feats(text_col, E):
    K = {n: dds(d[n][text_col]) for n in d}
    return {'emo': E, 'kw': {n: np.c_[K[n], K[n].sum(1)] for n in d}}

def build(text_col, blocks, keys, use_text=True):
    tf = TfidfVectorizer(max_features=20000, ngram_range=(1, 2), min_df=2, sublinear_tf=True).fit(d['train'][text_col])
    dense = lambda n: np.hstack([blocks[k][n] for k in keys]) if keys else None
    sc = StandardScaler().fit(dense('train')) if keys else None
    X = {}
    for n in d:
        parts = ([tf.transform(d[n][text_col])] if use_text else []) + ([sp.csr_matrix(sc.transform(dense(n)))] if keys else [])
        X[n] = sp.hstack(parts).tocsr()
    return X

def run(X):
    sw = compute_sample_weight('balanced', y['train']); P_, F, A, AUC = {}, [], [], []
    for s in SEEDS:
        m = xgb(s).fit(X['train'], y['train'], sample_weight=sw); pr = m.predict_proba(X['test']); p = pr.argmax(1)
        P_[s] = (p, pr); F.append(f1_score(y['test'], p, average='macro')); A.append(accuracy_score(y['test'], p))
        AUC.append(roc_auc_score(label_binarize(y['test'], classes=[0, 1, 2]), pr, average='macro'))
    return dict(preds=P_, f1=np.array(F), acc=np.array(A), auc=np.array(AUC))

rng = np.random.default_rng(0); IDX = [rng.integers(0, len(y['test']), len(y['test'])) for _ in range(1000)]
def boot(ra, rb):  # matched-seed bootstrap: mean over seeds of (b - a) in each resample
    dd = np.array([np.mean([f1_score(y['test'][i], rb['preds'][s][0][i], average='macro') -
                            f1_score(y['test'][i], ra['preds'][s][0][i], average='macro') for s in SEEDS]) for i in IDX])
    return dd.mean(), np.percentile(dd, 2.5), np.percentile(dd, 97.5)

RECORD = []
def rec(item, text, feats_, seeds, note=''):
    RECORD.append(dict(item=item, dataset='bugzilla ' + json.dumps(VERSION), split='stratified 70/15/15, seed 42',
                       model_seeds=seeds, masking=text, features=feats_, note=note))

# ---------- 2. masking sensitivity (original vs masked, all features consistent) ----------
print('\n=== MASKING SENSITIVITY (XGBoost, 5 seeds) ===')
R = {}
for tag, col, E in (('original', 'text', Eo), ('masked', 'mtext', Em)):
    B = feats(col, E)
    R[tag, 'base'] = run(build(col, B, []))
    R[tag, 'dist'] = run(build(col, B, ['emo', 'kw']))
    dlt = boot(R[tag, 'base'], R[tag, 'dist'])
    print(f"{tag:9s} TF-IDF {R[tag,'base']['f1'].mean():.4f}±{R[tag,'base']['f1'].std():.4f}  "
          f"+distress {R[tag,'dist']['f1'].mean():.4f}±{R[tag,'dist']['f1'].std():.4f}  "
          f"diff {dlt[0]:+.4f} [{dlt[1]:+.4f}, {dlt[2]:+.4f}]")
    rec(f'Masking paragraph ({tag})', tag, 'TF-IDF; TF-IDF+emotion+keywords', '42-46')

# ---------- 3. ablation A1-A5, masked text, masked emotion ----------
print('\n=== ABLATION (masked text, masked emotion, 5 seeds) ===')
B = feats('mtext', Em); ABL = {}
cfg = {'A1 TF-IDF': (True, []), 'A2 +emotion': (True, ['emo']), 'A3 +keywords': (True, ['kw']),
       'A4 +both': (True, ['emo', 'kw']), 'A5 distress only': (False, ['emo', 'kw'])}
for c, (ut, k) in cfg.items():
    ABL[c] = run(build('mtext', B, k, ut))
for c in cfg:
    r = ABL[c]; line = f"{c:17s} acc {r['acc'].mean():.4f}  F1 {r['f1'].mean():.4f}±{r['f1'].std():.4f}"
    if c != 'A1 TF-IDF':
        dlt = boot(ABL['A1 TF-IDF'], r); line += f"  diff {dlt[0]:+.4f} [{dlt[1]:+.4f}, {dlt[2]:+.4f}]"
    print(line)
rec('Table XI (ablation, Bugzilla)', 'masked (TF-IDF, emotion, keywords)', 'A1-A5', '42-46')

# ---------- 4. ROC-AUC, per-class, confusion, McNemar (masked) ----------
print('\n=== ROC-AUC / PER-CLASS / CONFUSION / McNEMAR (masked) ===')
base, dist = ABL['A1 TF-IDF'], ABL['A4 +both']
print('ROC-AUC XGB %.4f±%.4f   XGB+distress %.4f±%.4f' % (base['auc'].mean(), base['auc'].std(), dist['auc'].mean(), dist['auc'].std()))
p0, p1 = base['preds'][42][0], dist['preds'][42][0]
print(classification_report(y['test'], p1, target_names=CLS, digits=3))
cm = confusion_matrix(y['test'], p1); print('confusion (counts, rows=true):\n', cm)
ca, cb = p0 == y['test'], p1 == y['test']; n01, n10 = int((~ca & cb).sum()), int((ca & ~cb).sum())
st = (abs(n01 - n10) - 1) ** 2 / (n01 + n10); print('McNemar +distress right/base wrong %d vs %d, p=%.4f' % (n01, n10, 1 - chi2.cdf(st, 1)))
fig, ax = plt.subplots(1, 2, figsize=(10, 4.2))
for a, (nm, p) in zip(ax, (('XGBoost, TF-IDF', p0), ('XGBoost, TF-IDF + distress', p1))):
    c = confusion_matrix(y['test'], p); cn = c / c.sum(1, keepdims=True)
    a.imshow(cn, cmap='Blues', vmin=0, vmax=1)
    for i in range(3):
        for j in range(3): a.text(j, i, f'{cn[i,j]:.2f}\n({c[i,j]})', ha='center', va='center', color='white' if cn[i, j] > .5 else 'black', fontsize=9)
    a.set_xticks(range(3)); a.set_yticks(range(3)); a.set_xticklabels(CLS); a.set_yticklabels(CLS)
    a.set_xlabel('Predicted'); a.set_ylabel('True'); a.set_title(nm, fontsize=10)
plt.tight_layout(); plt.savefig(P + 'fig_confusion_bugzilla_masked.png', dpi=300); print('saved fig_confusion_bugzilla_masked.png')
rec('Table IX, per-class table, Fig. 7 (Bugzilla), McNemar', 'masked (all features)', 'XGB vs XGB+distress', '42-46 (AUC); 42 (per-class, confusion, McNemar)')

# ---------- 5. emotion-severity association on masked emotion ----------
print('\n=== ASSOCIATION (Kruskal-Wallis, all Bugzilla reports) ===')
Eall_o = np.vstack([Eo[n] for n in d]); Eall_m = np.vstack([Em[n] for n in d]); yall = np.concatenate([y[n] for n in d])
for nm, E in (('original', Eall_o), ('masked', Eall_m)):
    out = []
    for j, e in enumerate(EMO):
        H, p = kruskal(*[E[yall == k, j] for k in range(3)]); out.append(f'{e} eps2={H/(len(yall)-1):.3f}')
    print(nm, '; '.join(out))
rec('Table X / Fig. 4 (association, Bugzilla)', 'original and masked', 'emotion probabilities', 'n/a')

# ---------- 6. ten repeated splits, masked emotion ----------
print('\n=== REPEATED SPLITS (10 stratified 70/15/15 splits of pooled masked Bugzilla) ===')
df = pd.concat([d[n] for n in d], ignore_index=True); Ea = Eall_m; K = dds(df.mtext); Da = np.c_[Ea, K, K.sum(1)]; ya = yall
rows = []
for rep in range(10):
    tr, tmp = train_test_split(np.arange(len(df)), test_size=.30, stratify=ya, random_state=rep)
    _, te = train_test_split(tmp, test_size=.50, stratify=ya[tmp], random_state=rep)
    tf = TfidfVectorizer(max_features=20000, ngram_range=(1, 2), min_df=2, sublinear_tf=True).fit(df.mtext.iloc[tr])
    Ttr, Tte = tf.transform(df.mtext.iloc[tr]), tf.transform(df.mtext.iloc[te])
    sc = StandardScaler().fit(Da[tr]); Dtr, Dte = sc.transform(Da[tr]), sc.transform(Da[te]); sw = compute_sample_weight('balanced', ya[tr])
    r = dict(rep=rep)
    r['LogReg'] = f1_score(ya[te], LogisticRegression(max_iter=1000, class_weight='balanced').fit(Ttr, ya[tr]).predict(Tte), average='macro')
    r['XGB'] = f1_score(ya[te], xgb(rep).fit(Ttr, ya[tr], sample_weight=sw).predict(Tte), average='macro')
    r['XGB+distress'] = f1_score(ya[te], xgb(rep).fit(sp.hstack([Ttr, sp.csr_matrix(Dtr)]).tocsr(), ya[tr], sample_weight=sw)
                                 .predict(sp.hstack([Tte, sp.csr_matrix(Dte)]).tocsr()), average='macro')
    rows.append(r); print(rep, {k: round(v, 4) for k, v in r.items() if k != 'rep'})
rs = pd.DataFrame(rows)
for k in ('LogReg', 'XGB', 'XGB+distress'): print(f'{k}: {rs[k].mean():.4f}±{rs[k].std():.4f}')
for a, b in (('XGB', 'XGB+distress'), ('LogReg', 'XGB')):
    dd = rs[b] - rs[a]; print(f'{b} - {a}: mean {dd.mean():+.4f}, Wilcoxon p={wilcoxon(rs[b], rs[a]).pvalue:.4f}, wins {(dd>0).sum()}/10')
rec('Table XII / Fig. 5 (repeated splits, Bugzilla)', 'masked (all features)', 'LogReg, XGB, XGB+distress', 'seed = split index 0-9',
    'splits of one corpus, not independent datasets')

pd.DataFrame(RECORD).to_csv(P + 'run_record_bugzilla_masked.csv', index=False)
print('\nsaved run_record_bugzilla_masked.csv')


In [ ]:
# Rerun: emotion probabilities recomputed on MASKED Bugzilla text, then every Bugzilla
# comparison that uses distress features is rerun with one seed set (42-46).

from google.colab import drive
drive.mount('/content/drive')
import re, json, hashlib, numpy as np, pandas as pd, scipy.sparse as sp, matplotlib
matplotlib.use('Agg'); import matplotlib.pyplot as plt
from scipy.stats import chi2, kruskal, wilcoxon
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler, label_binarize
from sklearn.linear_model import LogisticRegression
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import f1_score, accuracy_score, roc_auc_score, classification_report, confusion_matrix
from xgboost import XGBClassifier
import torch
from transformers import pipeline

P = '/content/drive/MyDrive/Colab Notebooks/'
SEEDS = [42, 43, 44, 45, 46]
CLS = ['Critical', 'High', 'Low']; CID = {k: i for i, k in enumerate(CLS)}
SEV = r'\b(blockers?|critical|criticals|major|majors|minor|minors|trivial|normal|severity|sev|enhancement|regression)\b'
mask = lambda s: re.sub(r'\s+', ' ', re.sub(SEV, ' ', s)).strip()
LEX = {'frustration': ['frustrating', 'annoying', 'broken', 'failed', 'crash', 'error'],
       'confusion': ['confused', 'unclear', 'dont understand', 'unknown', 'complicated'],
       'anxiety': ['worried', 'anxious', 'fear', 'unsafe', 'security'],
       'urgency': ['urgent', 'immediately', 'asap', 'critical', 'important'],
       'helplessness': ['stuck', 'unable', 'cannot', 'helpless', 'nothing works'],
       'anger': ['hate', 'terrible', 'worst', 'angry', 'useless']}
EMO = ['anger', 'disgust', 'fear', 'joy', 'neutral', 'sadness', 'surprise']
dds = lambda texts: np.array([[sum(t.lower().count(w) for w in ws) for ws in LEX.values()] for t in texts])
xgb = lambda s: XGBClassifier(n_estimators=300, max_depth=6, learning_rate=0.1, subsample=0.8, colsample_bytree=0.8,
                              tree_method='hist', device='cuda' if torch.cuda.is_available() else 'cpu', random_state=s)
md5 = lambda f: hashlib.md5(open(f, 'rb').read()).hexdigest()[:10]

# ---------- 0. data + dataset version ----------
d = {n: pd.read_csv(P + f'clean_bugzilla_{n}.csv') for n in ('train', 'val', 'test')}
for n in d: d[n]['text'] = d[n]['text'].fillna('').astype(str); d[n]['mtext'] = d[n]['text'].map(mask)
y = {n: d[n]['y'].map(CID).values for n in d}
VERSION = {n: md5(P + f'clean_bugzilla_{n}.csv') for n in d}
print('DATASET VERSION (md5 of clean_bugzilla_*.csv):', VERSION)
print('Bugzilla class counts [Critical, High, Low]:', {n: d[n]['y'].value_counts().reindex(CLS).tolist() for n in d})
g = {n: pd.read_csv(P + f'clean_github_{n}.csv') for n in ('train', 'val', 'test')}
print('GitHub class counts [Fast, Medium, Slow]:', {n: g[n]['y'].value_counts().reindex(['Fast', 'Medium', 'Slow']).tolist() for n in g})
print('GitHub version (md5):', {n: md5(P + f'clean_github_{n}.csv') for n in g})
print('GitHub columns:', list(g['train'].columns))
raw_bz = pd.read_csv(P + 'Bugzilla_sev.csv'); print('Bugzilla raw columns:', list(raw_bz.columns))
print('Bugzilla raw severity counts:', raw_bz['Severity'].value_counts().to_dict())
for c in raw_bz.columns:
    if any(k in c.lower() for k in ('product', 'component', 'date', 'creat', 'open', 'time')):
        s = raw_bz[c]; print(f'  {c}: e.g. {s.dropna().astype(str).head(3).tolist()}  n_unique={s.nunique()}')
raw_gh = pd.read_csv(P + 'GitHub_bugs.csv'); print('GitHub raw columns:', list(raw_gh.columns))
_t = pd.to_datetime(raw_gh['Created'], errors='coerce', utc=True); print('GitHub created range:', _t.min(), '->', _t.max())
for c in raw_gh.columns:
    if any(k in c.lower() for k in ('repo', 'project', 'url', 'owner')):
        print(f'  {c}: n_unique={raw_gh[c].nunique()}  top={raw_gh[c].value_counts().head(5).to_dict()}')

# ---------- 1. emotion probabilities on MASKED text ----------
clf = pipeline('text-classification', model='j-hartmann/emotion-english-distilroberta-base', top_k=None,
               device=0 if torch.cuda.is_available() else -1, truncation=True, max_length=256)
def emo(texts, bs=64):
    out = clf(list(texts), batch_size=bs)
    return np.array([[{r['label']: r['score'] for r in o}[e] for e in EMO] for o in out])
_chk = emo(d['train']['text'].iloc[:200]); _old = np.load(P + 'emo_probs_bz_train.npy')[:200]
print('column-order check vs stored original probs (should be ~0): %.4f' % np.abs(_chk - _old).mean())
Eo, Em = {}, {}
for n in d:
    Eo[n] = np.load(P + f'emo_probs_bz_{n}.npy')
    f = P + f'emo_probs_bzmask_{n}.npy'
    try: Em[n] = np.load(f); print('loaded', f)
    except FileNotFoundError: Em[n] = emo(d[n]['mtext']); np.save(f, Em[n]); print('saved', f)
ch = np.abs(Em['train'] - Eo['train']).sum(1)
print('emotion vectors changed by masking (train): %.1f%% of texts; mean L1 change %.4f; top emotion changed %.1f%%'
      % (100 * (ch > 1e-6).mean(), ch.mean(), 100 * (Em['train'].argmax(1) != Eo['train'].argmax(1)).mean()))

def feats(text_col, E):
    K = {n: dds(d[n][text_col]) for n in d}
    return {'emo': E, 'kw': {n: np.c_[K[n], K[n].sum(1)] for n in d}}

def build(text_col, blocks, keys, use_text=True):
    tf = TfidfVectorizer(max_features=20000, ngram_range=(1, 2), min_df=2, sublinear_tf=True).fit(d['train'][text_col])
    dense = lambda n: np.hstack([blocks[k][n] for k in keys]) if keys else None
    sc = StandardScaler().fit(dense('train')) if keys else None
    X = {}
    for n in d:
        parts = ([tf.transform(d[n][text_col])] if use_text else []) + ([sp.csr_matrix(sc.transform(dense(n)))] if keys else [])
        X[n] = sp.hstack(parts).tocsr()
    return X

def run(X):
    sw = compute_sample_weight('balanced', y['train']); P_, F, A, AUC = {}, [], [], []
    for s in SEEDS:
        m = xgb(s).fit(X['train'], y['train'], sample_weight=sw); pr = m.predict_proba(X['test']); p = pr.argmax(1)
        P_[s] = (p, pr); F.append(f1_score(y['test'], p, average='macro')); A.append(accuracy_score(y['test'], p))
        AUC.append(roc_auc_score(label_binarize(y['test'], classes=[0, 1, 2]), pr, average='macro'))
    return dict(preds=P_, f1=np.array(F), acc=np.array(A), auc=np.array(AUC))

rng = np.random.default_rng(0); IDX = [rng.integers(0, len(y['test']), len(y['test'])) for _ in range(1000)]
def boot(ra, rb):  # matched-seed bootstrap: mean over seeds of (b - a) in each resample
    dd = np.array([np.mean([f1_score(y['test'][i], rb['preds'][s][0][i], average='macro') -
                            f1_score(y['test'][i], ra['preds'][s][0][i], average='macro') for s in SEEDS]) for i in IDX])
    return dd.mean(), np.percentile(dd, 2.5), np.percentile(dd, 97.5)

RECORD = []
def rec(item, text, feats_, seeds, note=''):
    RECORD.append(dict(item=item, dataset='bugzilla ' + json.dumps(VERSION), split='stratified 70/15/15, seed 42',
                       model_seeds=seeds, masking=text, features=feats_, note=note))

# ---------- 2. masking sensitivity ----------
print('\n=== MASKING SENSITIVITY (XGBoost, 5 seeds) ===')
R = {}
for tag, col, E in (('original', 'text', Eo), ('masked', 'mtext', Em)):
    B = feats(col, E)
    R[tag, 'base'] = run(build(col, B, []))
    R[tag, 'dist'] = run(build(col, B, ['emo', 'kw']))
    dlt = boot(R[tag, 'base'], R[tag, 'dist'])
    print(f"{tag:9s} TF-IDF {R[tag,'base']['f1'].mean():.4f}±{R[tag,'base']['f1'].std():.4f}  "
          f"+distress {R[tag,'dist']['f1'].mean():.4f}±{R[tag,'dist']['f1'].std():.4f}  "
          f"diff {dlt[0]:+.4f} [{dlt[1]:+.4f}, {dlt[2]:+.4f}]")
    rec(f'Masking paragraph ({tag})', tag, 'TF-IDF; TF-IDF+emotion+keywords', '42-46')

# ---------- 3. ablation A1-A5 ----------
print('\n=== ABLATION (masked text, masked emotion, 5 seeds) ===')
B = feats('mtext', Em); ABL = {}
cfg = {'A1 TF-IDF': (True, []), 'A2 +emotion': (True, ['emo']), 'A3 +keywords': (True, ['kw']),
       'A4 +both': (True, ['emo', 'kw']), 'A5 distress only': (False, ['emo', 'kw'])}
for c, (ut, k) in cfg.items():
    ABL[c] = run(build('mtext', B, k, ut))
for c in cfg:
    r = ABL[c]; line = f"{c:17s} acc {r['acc'].mean():.4f}  F1 {r['f1'].mean():.4f}±{r['f1'].std():.4f}"
    if c != 'A1 TF-IDF':
        dlt = boot(ABL['A1 TF-IDF'], r); line += f"  diff {dlt[0]:+.4f} [{dlt[1]:+.4f}, {dlt[2]:+.4f}]"
    print(line)
rec('Table XI (ablation, Bugzilla)', 'masked (TF-IDF, emotion, keywords)', 'A1-A5', '42-46')

# ---------- 4. ROC-AUC, per-class, confusion, McNemar ----------
print('\n=== ROC-AUC / PER-CLASS / CONFUSION / McNEMAR (masked) ===')
base, dist = ABL['A1 TF-IDF'], ABL['A4 +both']
print('ROC-AUC XGB %.4f±%.4f   XGB+distress %.4f±%.4f' % (base['auc'].mean(), base['auc'].std(), dist['auc'].mean(), dist['auc'].std()))
p0, p1 = base['preds'][42][0], dist['preds'][42][0]
print(classification_report(y['test'], p1, target_names=CLS, digits=3))
cm = confusion_matrix(y['test'], p1); print('confusion (counts, rows=true):\n', cm)
ca, cb = p0 == y['test'], p1 == y['test']; n01, n10 = int((~ca & cb).sum()), int((ca & ~cb).sum())
st = (abs(n01 - n10) - 1) ** 2 / (n01 + n10); print('McNemar +distress right/base wrong %d vs %d, p=%.4f' % (n01, n10, 1 - chi2.cdf(st, 1)))
fig, ax = plt.subplots(1, 2, figsize=(10, 4.2))
for a, (nm, p) in zip(ax, (('XGBoost, TF-IDF', p0), ('XGBoost, TF-IDF + distress', p1))):
    c = confusion_matrix(y['test'], p); cn = c / c.sum(1, keepdims=True)
    a.imshow(cn, cmap='Blues', vmin=0, vmax=1)
    for i in range(3):
        for j in range(3): a.text(j, i, f'{cn[i,j]:.2f}\n({c[i,j]})', ha='center', va='center', color='white' if cn[i, j] > .5 else 'black', fontsize=9)
    a.set_xticks(range(3)); a.set_yticks(range(3)); a.set_xticklabels(CLS); a.set_yticklabels(CLS)
    a.set_xlabel('Predicted'); a.set_ylabel('True'); a.set_title(nm, fontsize=10)
plt.tight_layout(); plt.savefig(P + 'fig_confusion_bugzilla_masked.png', dpi=300); print('saved fig_confusion_bugzilla_masked.png')
rec('Table IX, per-class table, Fig. 7 (Bugzilla), McNemar', 'masked (all features)', 'XGB vs XGB+distress', '42-46 (AUC); 42 (per-class, confusion, McNemar)')

# ---------- 5. emotion-severity association ----------
print('\n=== ASSOCIATION (Kruskal-Wallis, all Bugzilla reports) ===')
Eall_o = np.vstack([Eo[n] for n in d]); Eall_m = np.vstack([Em[n] for n in d]); yall = np.concatenate([y[n] for n in d])
for nm, E in (('original', Eall_o), ('masked', Eall_m)):
    out = []
    for j, e in enumerate(EMO):
        H, p = kruskal(*[E[yall == k, j] for k in range(3)]); out.append(f'{e} eps2={H/(len(yall)-1):.3f}')
    print(nm, '; '.join(out))
rec('Table X / Fig. 4 (association, Bugzilla)', 'original and masked', 'emotion probabilities', 'n/a')

# ---------- 6. ten repeated splits ----------
print('\n=== REPEATED SPLITS (10 stratified 70/15/15 splits of pooled masked Bugzilla) ===')
df = pd.concat([d[n] for n in d], ignore_index=True); Ea = Eall_m; K = dds(df.mtext); Da = np.c_[Ea, K, K.sum(1)]; ya = yall
rows = []
for rep in range(10):
    tr, tmp = train_test_split(np.arange(len(df)), test_size=.30, stratify=ya, random_state=rep)
    _, te = train_test_split(tmp, test_size=.50, stratify=ya[tmp], random_state=rep)
    tf = TfidfVectorizer(max_features=20000, ngram_range=(1, 2), min_df=2, sublinear_tf=True).fit(df.mtext.iloc[tr])
    Ttr, Tte = tf.transform(df.mtext.iloc[tr]), tf.transform(df.mtext.iloc[te])
    sc = StandardScaler().fit(Da[tr]); Dtr, Dte = sc.transform(Da[tr]), sc.transform(Da[te]); sw = compute_sample_weight('balanced', ya[tr])
    r = dict(rep=rep)
    r['LogReg'] = f1_score(ya[te], LogisticRegression(max_iter=1000, class_weight='balanced').fit(Ttr, ya[tr]).predict(Tte), average='macro')
    r['XGB'] = f1_score(ya[te], xgb(rep).fit(Ttr, ya[tr], sample_weight=sw).predict(Tte), average='macro')
    r['XGB+distress'] = f1_score(ya[te], xgb(rep).fit(sp.hstack([Ttr, sp.csr_matrix(Dtr)]).tocsr(), ya[tr], sample_weight=sw)
                                 .predict(sp.hstack([Tte, sp.csr_matrix(Dte)]).tocsr()), average='macro')
    rows.append(r); print(rep, {k: round(v, 4) for k, v in r.items() if k != 'rep'})
rs = pd.DataFrame(rows)
for k in ('LogReg', 'XGB', 'XGB+distress'): print(f'{k}: {rs[k].mean():.4f}±{rs[k].std():.4f}')
for a, b in (('XGB', 'XGB+distress'), ('LogReg', 'XGB')):
    dd = rs[b] - rs[a]; print(f'{b} - {a}: mean {dd.mean():+.4f}, Wilcoxon p={wilcoxon(rs[b], rs[a]).pvalue:.4f}, wins {(dd>0).sum()}/10')
rec('Table XII / Fig. 5 (repeated splits, Bugzilla)', 'masked (all features)', 'LogReg, XGB, XGB+distress', 'seed = split index 0-9',
    'splits of one corpus, not independent datasets')

pd.DataFrame(RECORD).to_csv(P + 'run_record_bugzilla_masked.csv', index=False)
print('\nsaved run_record_bugzilla_masked.csv')

Mounted at /content/drive
DATASET VERSION (md5 of clean_bugzilla_*.csv): {'train': '52b63da134', 'val': 'cd246f452d', 'test': '043d879265'}
Bugzilla class counts [Critical, High, Low]: {'train': [2665, 2630, 17395], 'val': [571, 564, 3727], 'test': [571, 564, 3728]}
GitHub class counts [Fast, Medium, Slow]: {'train': [6856, 6856, 6856], 'val': [1481, 1464, 1463], 'test': [1499, 1456, 1453]}
GitHub version (md5): {'train': '01861445e1', 'val': 'a6ee84872f', 'test': 'f8aa878fe3'}
GitHub columns: ['Issue id', 'Summary', 'Description', 'Resolution', 'days', 'text', 'y']
Bugzilla raw columns: ['Unnamed: 0.1', 'Unnamed: 0', 'Description', 'Severity', 'Label']
Bugzilla raw severity counts: {'normal': 22828, 'major': 3760, 'critical': 2726, 'minor': 1462, 'blocker': 1082, 'trivial': 570}
GitHub raw columns: ['Issue id', 'Summary', 'Created', 'Resolved', 'Description', 'Resolution']
GitHub created range: 2015-11-13 16:34:23+00:00 -> 2024-07-25 14:34:59+00:00


config.json:   0%|          | 0.00/1.00k [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  329MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/294 [00:00<?, ?B/s]

... (5 outputs left out)

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

column-order check vs stored original probs (should be ~0): 0.0001
saved /content/drive/MyDrive/Colab Notebooks/emo_probs_bzmask_train.npy
saved /content/drive/MyDrive/Colab Notebooks/emo_probs_bzmask_val.npy
saved /content/drive/MyDrive/Colab Notebooks/emo_probs_bzmask_test.npy
emotion vectors changed by masking (train): 100.0% of texts; mean L1 change 0.0091; top emotion changed 0.6%

=== MASKING SENSITIVITY (XGBoost, 5 seeds) ===


/usr/local/lib/python3.13/dist-packages/xgboost/core.py:569: UserWarning: [04:47:20] WARNING: /__w/xgboost/xgboost/src/common/error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


original  TF-IDF 0.5513±0.0040  +distress 0.5596±0.0024  diff +0.0082 [+0.0019, +0.0142]
masked    TF-IDF 0.5344±0.0027  +distress 0.5426±0.0051  diff +0.0082 [+0.0028, +0.0142]

=== ABLATION (masked text, masked emotion, 5 seeds) ===
A1 TF-IDF         acc 0.7396  F1 0.5344±0.0027
A2 +emotion       acc 0.7436  F1 0.5400±0.0028  diff +0.0055 [+0.0004, +0.0108]
A3 +keywords      acc 0.7434  F1 0.5390±0.0042  diff +0.0046 [-0.0011, +0.0102]
A4 +both          acc 0.7446  F1 0.5426±0.0051  diff +0.0082 [+0.0028, +0.0142]
A5 distress only  acc 0.6531  F1 0.4569±0.0014  diff -0.0774 [-0.0942, -0.0611]

=== ROC-AUC / PER-CLASS / CONFUSION / McNEMAR (masked) ===
ROC-AUC XGB 0.7551±0.0026   XGB+distress 0.7605±0.0010
              precision    recall  f1-score   support

    Critical      0.469     0.601     0.527       571
        High      0.291     0.248     0.268       564
         Low      0.869     0.851     0.860      3728

    accuracy                          0.752      4863
   macro av

In [ ]:
# Supervisor round 3:
#  (A) audit records for the conference pipeline (balanced files, 80:20 split overlap)
#  (B) GitHub provenance check
#  (C) GitHub ten resplits with tertile thresholds recomputed inside EACH training partition
#  (D) per-class precision/recall/F1 on the main split (Bugzilla masked, GitHub) + saved predictions
# Everything is written to  MyDrive/Colab Notebooks/repro/  for the review repository.

from google.colab import drive
drive.mount('/content/drive')
import os, re, json, numpy as np, pandas as pd, scipy.sparse as sp, torch
from scipy.stats import wilcoxon
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import f1_score, accuracy_score, precision_recall_fscore_support
from xgboost import XGBClassifier

P = '/content/drive/MyDrive/Colab Notebooks/'
R = P + 'repro/'; os.makedirs(R, exist_ok=True)
DEV = 'cuda' if torch.cuda.is_available() else 'cpu'
LEX = {'frustration': ['frustrating', 'annoying', 'broken', 'failed', 'crash', 'error'],
       'confusion': ['confused', 'unclear', 'dont understand', 'unknown', 'complicated'],
       'anxiety': ['worried', 'anxious', 'fear', 'unsafe', 'security'],
       'urgency': ['urgent', 'immediately', 'asap', 'critical', 'important'],
       'helplessness': ['stuck', 'unable', 'cannot', 'helpless', 'nothing works'],
       'anger': ['hate', 'terrible', 'worst', 'angry', 'useless']}
dds = lambda texts: np.array([[sum(t.lower().count(w) for w in ws) for ws in LEX.values()] for t in texts])
xgb = lambda s: XGBClassifier(n_estimators=300, max_depth=6, learning_rate=0.1, subsample=0.8, colsample_bytree=0.8,
                              tree_method='hist', device=DEV, random_state=s)
tfidf = lambda: TfidfVectorizer(max_features=20000, ngram_range=(1, 2), min_df=2, sublinear_tf=True)

# ---------------- (A) audit of the conference-era files ----------------
print('=== (A) AUDIT OF CONFERENCE-ERA BALANCED FILES ===')
for f in ('so_severity_balanced.csv', 'github_severity_balanced.csv', 'bugzilla_severity_balanced.csv'):
    try: b = pd.read_csv(P + f)
    except FileNotFoundError: print(f, 'not found'); continue
    tc = next(c for c in ('text', 'clean_text', 'Text', 'Body', 'Description') if c in b.columns)
    lc = next((c for c in ('severity', 'Severity', 'label', 'severity_label') if c in b.columns), None)
    t = b[tc].fillna('').astype(str)
    line = f'{f}: rows={len(b)} unique_texts={t.nunique()} repeated_rows={len(b)-t.nunique()}'
    if lc:
        line += f' class_counts={b[lc].value_counts().to_dict()}'
        tr, te = train_test_split(b, test_size=0.2, random_state=42, stratify=b[lc])   # the conference 80:20 split
        line += f' | 80:20 seed-42 split: train={len(tr)} test={len(te)} identical_texts_in_both={len(set(tr[tc].astype(str)) & set(te[tc].astype(str)))}'
    print(line)

# ---------------- (B) GitHub provenance check ----------------
print('\n=== (B) GITHUB PROVENANCE CHECK ===')
gr = pd.read_csv(P + 'GitHub_bugs.csv')
print('Issue id range:', gr['Issue id'].min(), '->', gr['Issue id'].max(), ' n =', len(gr), ' unique ids =', gr['Issue id'].nunique())
txt = (gr['Summary'].fillna('') + ' ' + gr['Description'].fillna('')).str.lower()
for k in ('vscode', 'vs code', 'visual studio code', 'copilot', 'extension host', 'workbench', 'monaco'):
    print(f'  texts mentioning "{k}": {txt.str.contains(k, regex=False).mean()*100:.1f}%')
print('  sample titles:', gr['Summary'].dropna().head(8).tolist())

# ---------------- (C) GitHub resplits, thresholds per training partition ----------------
print('\n=== (C) GITHUB TEN RESPLITS, TERTILE THRESHOLDS FROM EACH TRAINING PARTITION ===')
parts = [pd.read_csv(P + f'clean_github_{n}.csv') for n in ('train', 'val', 'test')]
df = pd.concat(parts, ignore_index=True); df['text'] = df['text'].fillna('').astype(str)
E = np.vstack([np.load(P + f'emo_probs_gh_{n}.npy') for n in ('train', 'val', 'test')])
assert len(E) == len(df) and 'days' in df.columns
K = dds(df.text); D = np.c_[E, K, K.sum(1)]; days = df['days'].values
rows, split_rec = [], []
for rep in range(10):
    idx = np.arange(len(df))
    tr, tmp = train_test_split(idx, test_size=0.30, random_state=rep)          # split first (labels do not exist yet)
    va, te = train_test_split(tmp, test_size=0.50, random_state=rep)
    q1, q2 = np.quantile(days[tr], [1/3, 2/3])                                  # thresholds from THIS training partition only
    y = np.where(days <= q1, 0, np.where(days <= q2, 1, 2))                      # applied unchanged to val and test
    tf = tfidf().fit(df.text.iloc[tr]); Ttr, Tte = tf.transform(df.text.iloc[tr]), tf.transform(df.text.iloc[te])
    sc = StandardScaler().fit(D[tr]); Dtr, Dte = sc.transform(D[tr]), sc.transform(D[te])
    sw = compute_sample_weight('balanced', y[tr])
    pl = LogisticRegression(max_iter=1000, class_weight='balanced').fit(Ttr, y[tr]).predict(Tte)
    px = xgb(rep).fit(Ttr, y[tr], sample_weight=sw).predict(Tte)
    pd_ = xgb(rep).fit(sp.hstack([Ttr, sp.csr_matrix(Dtr)]).tocsr(), y[tr], sample_weight=sw).predict(sp.hstack([Tte, sp.csr_matrix(Dte)]).tocsr())
    r = dict(rep=rep, q1_days=round(q1, 3), q2_days=round(q2, 3),
             test_counts=np.bincount(y[te], minlength=3).tolist(),
             LogReg=f1_score(y[te], pl, average='macro'), XGB=f1_score(y[te], px, average='macro'), **{'XGB+distress': f1_score(y[te], pd_, average='macro')})
    rows.append(r); print(rep, {k: (round(v, 4) if isinstance(v, float) else v) for k, v in r.items() if k != 'rep'})
    pd.DataFrame({'row': te, 'y_true': y[te], 'LogReg': pl, 'XGB': px, 'XGB+distress': pd_}).to_csv(R + f'gh_resplit{rep}_test_predictions.csv', index=False)
    split_rec.append(dict(rep=rep, train=tr.tolist(), val=va.tolist(), test=te.tolist(), q1=float(q1), q2=float(q2)))
json.dump(split_rec, open(R + 'gh_resplits_indices_thresholds.json', 'w'))
rs = pd.DataFrame(rows); rs.to_csv(R + 'gh_resplits_results.csv', index=False)
for k in ('LogReg', 'XGB', 'XGB+distress'): print(f'{k}: {rs[k].mean():.4f}±{rs[k].std():.4f}')
for a, b in (('XGB', 'XGB+distress'), ('LogReg', 'XGB')):
    dd = rs[b] - rs[a]; print(f'{b} - {a}: mean {dd.mean():+.4f}, Wilcoxon p={wilcoxon(rs[b], rs[a]).pvalue:.4f}, wins {(dd>0).sum()}/10')
print('threshold range over resplits: q1 %.2f-%.2f days, q2 %.2f-%.2f days' % (rs.q1_days.min(), rs.q1_days.max(), rs.q2_days.min(), rs.q2_days.max()))

# ---------------- (D) per-class P/R/F1 on the main split + saved predictions ----------------
print('\n=== (D) PER-CLASS PRECISION / RECALL / F1, MAIN SPLIT (seed 42) ===')
SEV = r'\b(blockers?|critical|criticals|major|majors|minor|minors|trivial|normal|severity|sev|enhancement|regression)\b'
def main_split(pre, classes, emo_tag, mask):
    d = {n: pd.read_csv(P + f'clean_{pre}_{n}.csv') for n in ('train', 'test')}
    for n in d:
        d[n]['text'] = d[n]['text'].fillna('').astype(str)
        if mask: d[n]['text'] = d[n]['text'].map(lambda s: re.sub(r'\s+', ' ', re.sub(SEV, ' ', s)).strip())
    y = {n: d[n]['y'].map({c: i for i, c in enumerate(classes)}).values for n in d}
    Em = {n: np.load(P + f'emo_probs_{emo_tag}_{n}.npy') for n in d}
    Kd = {n: dds(d[n].text) for n in d}; Dd = {n: np.c_[Em[n], Kd[n], Kd[n].sum(1)] for n in d}
    sc = StandardScaler().fit(Dd['train']); tf = tfidf().fit(d['train'].text)
    T = {n: tf.transform(d[n].text) for n in d}; sw = compute_sample_weight('balanced', y['train'])
    out = {'y_true': y['test']}
    out['XGB'] = xgb(42).fit(T['train'], y['train'], sample_weight=sw).predict(T['test'])
    Xd = {n: sp.hstack([T[n], sp.csr_matrix(sc.transform(Dd[n]))]).tocsr() for n in d}
    out['XGB+distress'] = xgb(42).fit(Xd['train'], y['train'], sample_weight=sw).predict(Xd['test'])
    pd.DataFrame(out).to_csv(R + f'{pre}_main_split_test_predictions.csv', index=False)
    for m in ('XGB', 'XGB+distress'):
        p, r_, f, s = precision_recall_fscore_support(y['test'], out[m], labels=[0, 1, 2], zero_division=0)
        print(f'{pre} {m}: ' + '; '.join(f'{c} P={p[i]:.3f} R={r_[i]:.3f} F1={f[i]:.3f} n={s[i]}' for i, c in enumerate(classes))
              + f' | macro-F1={f1_score(y["test"], out[m], average="macro"):.4f} acc={accuracy_score(y["test"], out[m]):.4f}')
main_split('bugzilla', ['Critical', 'High', 'Low'], 'bzmask', True)
main_split('github', ['Fast', 'Medium', 'Slow'], 'gh', False)
print('\nSaved all records in', R, ':', sorted(os.listdir(R)))

Mounted at /content/drive
=== (A) AUDIT OF CONFERENCE-ERA BALANCED FILES ===
so_severity_balanced.csv: rows=13926 unique_texts=7017 repeated_rows=6909 class_counts={'Low': 4642, 'Critical': 4642, 'High': 4642} | 80:20 seed-42 split: train=11140 test=2786 identical_texts_in_both=1042
github_severity_balanced.csv: rows=32829 unique_texts=20530 repeated_rows=12299 class_counts={'Low': 10943, 'High': 10943, 'Critical': 10943} | 80:20 seed-42 split: train=26263 test=6566 identical_texts_in_both=3302
bugzilla_severity_balanced.csv: rows=32427 unique_texts=22731 repeated_rows=9696 class_counts={'Low': 10809, 'High': 10809, 'Critical': 10809} | 80:20 seed-42 split: train=25941 test=6486 identical_texts_in_both=2490

=== (B) GITHUB PROVENANCE CHECK ===
Issue id range: 3 -> 223706  n = 32829  unique ids = 32828
  texts mentioning "vscode": 59.2%
  texts mentioning "vs code": 32.0%
  texts mentioning "visual studio code": 3.1%
  texts mentioning "copilot": 1.5%
  texts mentioning "extension host"

/usr/local/lib/python3.13/dist-packages/xgboost/core.py:569: UserWarning: [14:42:33] WARNING: /__w/xgboost/xgboost/src/common/error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


0 {'q1_days': np.float64(1.323), 'q2_days': np.float64(20.211), 'test_counts': [1473, 1540, 1395], 'LogReg': 0.4471, 'XGB': 0.4412, 'XGB+distress': 0.4393}
1 {'q1_days': np.float64(1.322), 'q2_days': np.float64(20.034), 'test_counts': [1441, 1527, 1440], 'LogReg': 0.4518, 'XGB': 0.4418, 'XGB+distress': 0.4427}
2 {'q1_days': np.float64(1.325), 'q2_days': np.float64(20.008), 'test_counts': [1502, 1456, 1450], 'LogReg': 0.4498, 'XGB': 0.4383, 'XGB+distress': 0.4458}
3 {'q1_days': np.float64(1.306), 'q2_days': np.float64(19.826), 'test_counts': [1417, 1502, 1489], 'LogReg': 0.4511, 'XGB': 0.4498, 'XGB+distress': 0.4421}
4 {'q1_days': np.float64(1.295), 'q2_days': np.float64(19.826), 'test_counts': [1431, 1526, 1451], 'LogReg': 0.444, 'XGB': 0.448, 'XGB+distress': 0.4467}
5 {'q1_days': np.float64(1.349), 'q2_days': np.float64(20.148), 'test_counts': [1495, 1486, 1427], 'LogReg': 0.4465, 'XGB': 0.4495, 'XGB+distress': 0.4497}
6 {'q1_days': np.float64(1.328), 'q2_days': np.float64(19.829), 't